# Fly Chess V5: resumable training and measured comparisons

V4 and V5 share the same fly, graft, supervised labels, curriculum and initialization. V4 plays its raw policy. V5 adds fixed-budget PUCT and gated search-guided self-play.

Default **smoke** checks a synthetic graph. Set MODE_SETTING to **full** for the real connectome
on a T4. Change RUN_HOURS to allocate a fresh training allowance each invocation. Resume by
re-running; Drive stores datasets, prepared graph, teacher labels, optimizer/RNG state, complete
self-play replay, and checkpoints. Remove a run's STOP file to resume a deliberate pause.

Training rotates balanced shards through foundation, general play and consolidation; pinned
Stockfish MultiPV labels progressively improve the cached dataset and supply WDL targets.
Supervised source CP labels remain identified as proxies until relabelled. V5 self-play starts
only after search improves a training-only engine probe; unresolved capped games never become draws.
Validation and test position families never enter self-play replay. Plateau returns control after
10 non-improving rounds in consolidation; this is an experimental route towards 2000, not a promise.

The evaluation uses paired openings/colors at **120+1**, pinned Stockfish UCI_Elo references
(minimum supported rating, 1600, 2000), immutable checkpoint fingerprints and uncertainty bounds.
This conditional benchmark is not a FIDE/Chess.com/Lichess rating. All-loss results report a bound.
Search simulations and any assistance are saved with every result. A single T4 alternates the
paired notebook's training branches; separate notebooks can run concurrently on separate GPUs.

In [ ]:
import os
# Edit the quoted fallback or assign MODE_SETTING = "full" explicitly, then Run all.
MODE_SETTING = os.environ.get("FLY_CHESS_MODE", "smoke")
RUN_HOURS = float(os.environ.get("FLY_CHESS_TRAIN_HOURS", "2"))
EVALUATE = os.environ.get("FLY_CHESS_EVALUATE", "1") == "1"
USE_DRIVE_SETTING = os.environ.get("FLY_CHESS_USE_DRIVE", "1")
os.environ["FLY_CHESS_MODE"] = MODE_SETTING
os.environ["FLY_CHESS_USE_DRIVE"] = USE_DRIVE_SETTING
os.environ.setdefault("FLY_CHESS_TRAIN_POOL", "32768")
os.environ.setdefault("FLY_CHESS_BATCH", "128")
os.environ.setdefault("FLY_CHESS_TEACHER_NODES", "50000")
os.environ.setdefault("FLY_CHESS_RELABEL_PER_ROUND", "256")
os.environ.setdefault("FLY_CHESS_SIMULATIONS", "64")
os.environ.setdefault("FLY_CHESS_MATCH_PAIRS", "20")
# Optional: FLY_CHESS_SOURCE_DATASET points to your existing pinned V2 .npz + .json.
# Otherwise source MultiPV is streamed and saved once. Set FLY_CHESS_NEW_RUN=1 once
# for a fresh run, then remove it before the next invocation. Defaults resume active run.

In [ ]:
import subprocess
import sys

_pip = subprocess.run(
    [sys.executable, "-m", "pip", "install", "chess>=1.11.2", "zstandard", "scikit-learn", "pyarrow"],
    capture_output=True, text=True,
)
print(_pip.stdout[-2000:])
if _pip.returncode != 0:
    print(_pip.stderr[-4000:])
    # !pip (shell magic) never raises on failure, so a silent/transient install error would
    # otherwise surface several cells later as a confusing "ModuleNotFoundError: chess" instead
    # of here, where the actual cause is visible.
    raise RuntimeError("pip install failed \u2014 see the output above (often a transient network "
                        "issue in the Colab VM); re-run this cell, or Runtime -> Restart session "
                        "and Run all if it keeps failing")
import chess
import chess.engine
import chess.pgn
print(f"python-chess {chess.__version__} installed OK, with engine + pgn submodules")

In [ ]:
import chess
import chess.engine
import chess.pgn
import pickle
import random
import contextlib
import fcntl
import io
import uuid
import hashlib
import json
import math
import os
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import zstandard
from scipy.sparse import load_npz

# Optional: mount Google Drive so downloads and checkpoints survive a runtime reset/disconnect.
# Set True, re-run this cell, and approve the access prompt.
USE_DRIVE = os.environ.get("FLY_CHESS_USE_DRIVE", "1") == "1"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PERSISTENT_ROOT = Path("/content/drive/MyDrive/fly-chess-v4-v5")
    ROOT = Path("/content/fly-chess-v4-v5")
else:
    ROOT = Path(os.environ.get("FLY_CHESS_ROOT", "/content/fly-chess-v4-v5"))
    PERSISTENT_ROOT = ROOT

DATA = Path(os.environ.get("FLY_CHESS_DATA_ROOT", str(PERSISTENT_ROOT / "data" / 'v5')))
RAW = DATA / "malecns-v1.0"
CHESS_DATA = DATA / "fly-chess"
RUNS = PERSISTENT_ROOT / "runs" / "v5"
for _d in (RAW, CHESS_DATA, RUNS):
    _d.mkdir(parents=True, exist_ok=True)

# "smoke" runs every section in minutes on a small synthetic connectome, to check the notebook
# end to end without a big download. "full" uses the real MaleCNS graph and the multi-hour
# budget. Colab has no separate process to launch this into, so just edit this line (and
# FLY_CHESS_RUN_ID below, for a fresh run) and re-run rather than setting environment variables.
MODE = os.environ.get("FLY_CHESS_MODE", "smoke")
assert MODE in ("smoke", "full")

DEVICE = torch.device(os.environ.get("FLY_CHESS_DEVICE", "cuda" if torch.cuda.is_available() else "cpu"))
if MODE == "full" and DEVICE.type != "cuda":
    raise RuntimeError("Full training requires a GPU. Select a GPU runtime before Run all.")
if DEVICE.type != "cuda":
    print("WARNING: no GPU. Runtime -> Change runtime type -> GPU, then Runtime -> Restart session.")
SEED = int(os.environ.get("FLY_CHESS_SEED", "0"))
random.seed(SEED)
torch.manual_seed(SEED)
np.random.seed(SEED)


def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        while block := handle.read(8 * 1024 * 1024):
            digest.update(block)
    return digest.hexdigest()


def sh(*args, cwd=ROOT):
    args = [str(a) for a in args]
    print("$", " ".join(args), flush=True)
    with subprocess.Popen(args, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
    process.wait()
    if process.returncode:
        raise subprocess.CalledProcessError(process.returncode, args)


INK, INK_2, MUTED, GRID, AXIS, SURFACE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7", "#fcfcfb"
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "font.size": 10, "text.color": INK, "axes.labelcolor": INK_2, "axes.titlesize": 11,
    "axes.titleweight": "bold", "axes.titlelocation": "left", "axes.edgecolor": AXIS,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "axes.grid.axis": "y",
    "grid.color": GRID, "grid.linewidth": 0.8, "xtick.color": MUTED, "ytick.color": MUTED,
    "xtick.labelcolor": INK_2, "ytick.labelcolor": INK_2, "legend.frameon": False,
    "lines.linewidth": 2, "figure.dpi": 110,
})
print(f"mode={MODE}  device={DEVICE}  torch={torch.__version__}  python-chess={chess.__version__}")

In [ ]:
EXPERIMENT_VERSION = 5
active_pointer = RUNS / f"active-{MODE}.json"
if os.environ.get("FLY_CHESS_RUN_ID"):
    RUN_ID = os.environ["FLY_CHESS_RUN_ID"]
elif active_pointer.exists() and os.environ.get("FLY_CHESS_NEW_RUN") != "1":
    RUN_ID = json.loads(active_pointer.read_text())["run_id"]
else:
    RUN_ID = f"{MODE}-{time.strftime('%Y%m%d-%H%M%S')}-{uuid.uuid4().hex[:8]}"
assert Path(RUN_ID).name == RUN_ID and RUN_ID not in (".", "..")
RUN_DIR = RUNS / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)
RUN_START = time.monotonic()
TOTAL_BUDGET = 240 * 60 if MODE == "full" else 15 * 60

class RunPaused(Exception):
    pass


def atomic_json(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w") as handle:
        json.dump(value, handle, indent=2, allow_nan=False)
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(temporary, path)


STOP_ROOT = RUN_DIR

def check_stop():
    if (STOP_ROOT / "STOP").exists() or (RUN_DIR / "STOP").exists():
        raise RunPaused("Stop requested; remove STOP before resuming")


@contextlib.contextmanager
def writer_lock():
    if globals().get("RUN_LOCK_HANDLE") is not None:
        yield
        return
    with (RUN_DIR / "writer.lock").open("a") as handle:
        try:
            fcntl.flock(handle, fcntl.LOCK_EX | fcntl.LOCK_NB)
        except BlockingIOError:
            raise RuntimeError("Another writer owns this run")
        try:
            yield
        finally:
            fcntl.flock(handle, fcntl.LOCK_UN)


def rng_state(sampler):
    return {"python": random.getstate(), "numpy": np.random.get_state(),
            "sampler": sampler.get_state(), "torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all() if DEVICE.type == "cuda" else None}


def restore_rng(state, sampler):
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    sampler.set_state(state["sampler"])
    torch.set_rng_state(state["torch"].cpu())
    if state["cuda"] is not None and DEVICE.type == "cuda":
        torch.cuda.set_rng_state_all([s.cpu() for s in state["cuda"]])


def save_checkpoint(path, state):
    path = Path(path)
    temporary = path.with_suffix(".tmp")
    with temporary.open("wb") as handle:
        torch.save(state, handle)
        handle.flush()
        os.fsync(handle.fileno())
    if path.exists():
        os.replace(path, path.with_suffix(".previous.pt"))
    os.replace(temporary, path)


def load_checkpoint(path, identity):
    path = Path(path)
    candidates = [path, path.with_suffix(".previous.pt")]
    for candidate in candidates:
        if not candidate.exists():
            continue
        try:
            # Only local checkpoints created by this notebook are accepted.
            state = torch.load(candidate, map_location="cpu", weights_only=False)
        except (EOFError, RuntimeError, OSError, pickle.UnpicklingError):
            continue
        if state["identity"] != identity:
            raise ValueError("Checkpoint manifest mismatch; create a new run")
        return state
    if any(p.exists() for p in candidates):
        raise RuntimeError("No valid checkpoint remains")
    return None


def event(record):
    with (RUN_DIR / "events.jsonl").open("a") as handle:
        handle.write(json.dumps(record, allow_nan=False) + "\n")
        handle.flush()

atomic_json(active_pointer, {"run_id": RUN_ID})
print("run", RUN_ID, "checkpoints", RUN_DIR)

# Hold exclusive ownership across setup, calibration, training and evaluation.
RUN_LOCK_HANDLE = (RUN_DIR / "writer.lock").open("a")
try:
    fcntl.flock(RUN_LOCK_HANDLE, fcntl.LOCK_EX | fcntl.LOCK_NB)
except BlockingIOError:
    RUN_LOCK_HANDLE.close()
    RUN_LOCK_HANDLE = None
    raise RuntimeError("Another kernel owns this run")

In [ ]:
PIECE_INDEX = {chess.PAWN: 0, chess.KNIGHT: 1, chess.BISHOP: 2, chess.ROOK: 3, chess.QUEEN: 4, chess.KING: 5}
N_FEATURES = 788
PROMOTION_PAIRS = [(frm, to) for frm in range(48, 56) for to in range(56, 64)
                   if abs(chess.square_file(frm) - chess.square_file(to)) <= 1]
PROMOTION_KEYS = [(frm, to, piece) for frm, to in PROMOTION_PAIRS
                  for piece in (chess.QUEEN, chess.ROOK, chess.BISHOP, chess.KNIGHT)]
PROMOTION_INDEX = {key: 4096 + i for i, key in enumerate(PROMOTION_KEYS)}
N_MOVES = 4096 + len(PROMOTION_KEYS)
assert N_MOVES == 4184


def perspective_square(square, us):
    return square if us == chess.WHITE else chess.square_mirror(square)


def encode_base_board(board):
    x = np.zeros(780, dtype=np.float32)
    us, them = board.turn, not board.turn
    for square, piece in board.piece_map().items():
        sq = perspective_square(square, us)
        offset = 0 if piece.color == us else 6
        x[(offset + PIECE_INDEX[piece.piece_type]) * 64 + sq] = 1.0
    base = 768
    x[base + 0] = board.has_kingside_castling_rights(us)
    x[base + 1] = board.has_queenside_castling_rights(us)
    x[base + 2] = board.has_kingside_castling_rights(them)
    x[base + 3] = board.has_queenside_castling_rights(them)
    if board.ep_square is not None:
        ep = perspective_square(board.ep_square, us)
        x[base + 4 + chess.square_file(ep)] = 1.0
    return x


def encode_history_board(board):
    x = np.zeros(788, np.float32)
    x[:780] = encode_base_board(board)
    known = bool(board.move_stack)
    last = board.peek() if known else None
    x[780:] = [min(board.halfmove_clock / 150, 1), board.is_repetition(2),
               board.is_repetition(3), known, min(board.fullmove_number / 200, 1),
               perspective_square(last.from_square,board.turn)/63 if known else 0,
               perspective_square(last.to_square,board.turn)/63 if known else 0,
               bool(last.promotion) if known else 0]
    return x

encode_board = encode_history_board


def move_index(move, us):
    frm = perspective_square(move.from_square, us)
    to = perspective_square(move.to_square, us)
    return PROMOTION_INDEX[(frm, to, move.promotion)] if move.promotion else frm * 64 + to


def legal_move_table(board):
    """{move_index: [legal chess.Move, ...]} for every legal move, mirrored to the mover's frame."""
    us = board.turn
    table = {}
    for move in board.legal_moves:
        table.setdefault(move_index(move, us), []).append(move)
    return table


def index_to_move(idx, board):
    candidates = legal_move_table(board)[int(idx)]
    assert len(candidates) == 1, "Move vocabulary collision"
    return candidates[0]


def terminal_value(board):
    outcome = board.outcome(claim_draw=True)
    if outcome is None:
        return None
    return 0.5 if outcome.winner is None else float(outcome.winner == board.turn)


def source_value(board, cp=None, mate=None):
    terminal = terminal_value(board)
    if terminal is not None:
        return terminal
    sign = 1 if board.turn == chess.WHITE else -1
    if mate is not None:
        if mate == 0:
            raise ValueError("Nonterminal zero-distance mate score")
        return float(sign * mate > 0)
    return float(torch.sigmoid(torch.tensor(0.00368208 * sign * cp, dtype=torch.float64)))


def position_key(board):
    return hashlib.sha256(encode_base_board(board).astype(np.uint8).tobytes()).hexdigest()


def partition(board):
    bucket = int(position_key(board)[:16], 16) % 100
    return "train" if bucket < 90 else "validation" if bucket < 95 else "test"


def mask_logits(logits, boards):
    mask = torch.full_like(logits, -torch.inf)
    for i, board in enumerate(boards):
        indices = list(legal_move_table(board))
        mask[i, indices] = 0
    return logits + mask

In [ ]:
"""Shared V4/V5 data, curriculum, checkpoint and self-play implementation.

The notebook builder embeds this file verbatim. The namespace argument supplies
the notebook's frozen fly simulator, model, encoding, and tested MCTS functions.
"""
import copy
import gzip
import hashlib
import json
import math
import os
import pickle
from pathlib import Path
import shutil
import sqlite3
import time

import chess
import chess.engine
import numpy as np
import torch
from torch.nn import functional as F


def policy_distribution(scores, temperature=80.0):
    scores = np.asarray(scores, dtype=np.float64)
    if len(scores) == 0 or not np.isfinite(scores).all() or temperature <= 0:
        raise ValueError("Finite, nonempty move scores and positive temperature required")
    weights = np.exp((scores - scores.max()) / temperature)
    return (weights / weights.sum()).astype(np.float32)


def ranking_score(cp=None, mate=None):
    if mate is not None:
        return (10000 - min(abs(mate), 90)*100) * (1 if mate > 0 else -1)
    return float(np.clip(cp, -2000, 2000))


def soft_policy_loss(logits, move_indices, probabilities):
    indices = torch.as_tensor(move_indices, device=logits.device, dtype=torch.long)
    weights = torch.as_tensor(probabilities, device=logits.device, dtype=torch.float32)
    if indices.ndim != 2 or weights.shape != indices.shape or indices.shape[0] != len(logits):
        raise ValueError("Policy targets must have shape (batch, candidates)")
    if torch.any(weights < 0) or not torch.allclose(weights.sum(1), torch.ones(len(logits), device=logits.device, dtype=torch.float32), atol=.002):
        raise ValueError("Target probabilities must sum to one")
    gathered = F.log_softmax(logits.float(), dim=1).gather(1, indices)
    if not torch.isfinite(gathered[weights > 0]).all():
        raise ValueError("A target move is illegal or its probability is nonfinite")
    return -(torch.where(weights > 0, gathered, torch.zeros_like(gathered)) * weights).sum(1).mean()


def curriculum_weights(phase):
    # Foundation, tactics, endgame, ordinary. Earlier skills remain in later phases.
    return {"foundation": (.55, .20, .20, .05),
            "general": (.15, .25, .25, .35),
            "consolidation": (.10, .25, .25, .40)}[phase]


def stratified_pool(groups, count, rng, phase="general"):
    chosen = []
    weights = curriculum_weights(phase)
    available = [np.asarray(g, dtype=np.int64) for g in groups]
    if not any(len(g) for g in available):
        raise ValueError("Empty training split")
    for group, weight in zip(available, weights):
        if len(group):
            chosen.extend(rng.choice(group, min(len(group), int(count*weight)), replace=False).tolist())
    chosen = np.unique(np.asarray(chosen, dtype=np.int64))
    total = np.unique(np.concatenate(available))
    if len(chosen) < min(count, len(total)):
        rest = np.setdiff1d(total, chosen)
        chosen = np.concatenate([chosen, rng.choice(rest, min(count-len(chosen), len(rest)), replace=False)])
    rng.shuffle(chosen)
    return chosen


def training_groups(data, train_indices):
    groups = [[], [], [], []]
    fens, moves = data["fen"], data["move"]
    for offset, i in enumerate(train_indices):
        board = chess.Board(str(fens[i]))
        move = chess.Move.from_uci(str(moves[i]))
        pieces = len(board.piece_map())
        short_mate = bool(data["is_mate"][i]) and abs(int(data["raw_mate"][i])) <= 2
        category = 0 if short_mate else 1 if board.is_check() or board.is_capture(move) or move.promotion else 2 if pieces <= 10 else 3
        groups[category].append(int(i))
        if offset and offset % 100000 == 0:
            print("curriculum indexing", offset, "/", len(train_indices), flush=True)
    return [np.asarray(g, dtype=np.int64) for g in groups]


def ingest_advanced(lines, limit, min_depth, ns):
    rows, seen, rejected = [], set(), 0
    for raw in lines:
        ns["check_stop"]()
        try:
            source = json.loads(raw)
            board = chess.Board(source["fen"])
            if not board.is_valid() or ns["terminal_value"](board) is not None:
                raise ValueError("invalid_or_terminal")
            key = ns["position_key"](board)
            if key in seen:
                continue
            best = max(source["evals"], key=lambda e: (e["depth"], e.get("knodes", 0)))
            if best["depth"] < min_depth:
                continue
            candidates, scores = [], []
            sign = 1 if board.turn else -1
            for pv in best["pvs"]:
                move = board.parse_uci(pv["line"].split()[0])
                if move.uci() in candidates:
                    continue
                candidates.append(move.uci())
                scores.append(ranking_score(sign*pv["cp"] if "cp" in pv else None,
                                           sign*pv["mate"] if "mate" in pv else None))
                if len(candidates) == 3:
                    break
            probabilities = policy_distribution(scores)
            pv = best["pvs"][0]
            rows.append((board.fen(en_passant="fen"), candidates[0],
                         ns["source_value"](board, pv.get("cp"), pv.get("mate")),
                         best["depth"], pv.get("cp", 0), pv.get("mate", 0), "mate" in pv,
                         key, ns["partition"](board),
                         candidates + [candidates[0]]*(3-len(candidates)),
                         probabilities.tolist() + [0.]*(3-len(candidates))))
            seen.add(key)
        except (ValueError, KeyError, IndexError, TypeError):
            rejected += 1
        if len(rows) >= limit:
            break
    return rows, rejected


def prepare_advanced_data(ns):
    count = int(os.environ.get("FLY_CHESS_POSITIONS", "2000" if ns["MODE"] == "smoke" else "1000000"))
    path = ns["CHESS_DATA"] / f"positions-v4-{ns['MODE']}-{count}.npz"
    pin_path = path.with_suffix(".json")
    if not path.exists():
        legacy = Path(os.environ.get("FLY_CHESS_SOURCE_DATASET", str(ns["CHESS_DATA"] / f"positions-v2-{ns['MODE']}-{count}.npz")))
        if legacy.exists():
            legacy_pin = json.loads(legacy.with_suffix(".json").read_text())
            if ns["sha256"](legacy) != legacy_pin["sha256"]:
                raise ValueError("Source dataset integrity mismatch")
            with np.load(legacy, allow_pickle=False) as old:
                arrays = {k:old[k] for k in old.files}
            if "policy_moves" not in arrays:
                arrays["policy_moves"] = np.repeat(arrays["move"][:, None], 3, axis=1)
                arrays["policy_probs"] = np.tile(np.array([1, 0, 0], np.float32), (len(arrays["move"]), 1))
            # V4/V5 use the same canonical split as V3. Cached source labels are
            # explicitly identified as single-move/proxy labels until relabelled.
            origin = {"source": str(legacy), "source_sha256": legacy_pin["sha256"], "policy": "copied_source_multipv" if legacy_pin.get("schema") == 4 else "single_move_source"}
        else:
            import contextlib
            with contextlib.closing(ns["stream_zst_lines"]("https://database.lichess.org/lichess_db_eval.jsonl.zst")) as lines:
                rows, rejected = ingest_advanced(lines, count, 10 if ns["MODE"] == "smoke" else 14, ns)
            if not rows:
                raise ValueError("No accepted positions")
            names = ("fen", "move", "value", "depth", "raw_cp", "raw_mate", "is_mate", "key", "split", "policy_moves", "policy_probs")
            arrays = {name: np.array(col) for name, col in zip(names, zip(*rows))}
            origin = {"source": "https://database.lichess.org/lichess_db_eval.jsonl.zst", "rejected": rejected, "policy": "available_source_multipv"}
        np.savez_compressed(path, **arrays)
        ns["atomic_json"](pin_path, {"schema": 4, "sha256": ns["sha256"](path), "origin": origin,
                                      "source_value": "side_to_move_centipawn_proxy; pinned teacher supplies WDL"})
    pin = json.loads(pin_path.read_text())
    if pin["schema"] != 4 or ns["sha256"](path) != pin["sha256"]:
        raise ValueError("V4 dataset integrity mismatch")
    with np.load(path, allow_pickle=False) as archive:
        data = {k:archive[k] for k in archive.files}
    ns.update(positions_path=path, positions=data, fens=data["fen"], labels_uci=data["move"], values=data["value"],
              train_idx=np.flatnonzero(data["split"] == "train"),
              validation_idx=np.flatnonzero(data["split"] == "validation"), test_idx=np.flatnonzero(data["split"] == "test"))
    if not all(len(ns[k]) for k in ("train_idx", "validation_idx", "test_idx")):
        raise ValueError("All three dataset splits must be populated")
    print("dataset", path, "positions", len(data["fen"]), "split sizes", *[len(ns[k]) for k in ("train_idx", "validation_idx", "test_idx")], flush=True)
    return data


class TeacherStore:
    """Local SQLite with atomic persistent snapshots; one writer per run."""
    def __init__(self, local, persistent, config):
        self.local, self.persistent = Path(local), Path(persistent)
        self.local.parent.mkdir(parents=True, exist_ok=True)
        self.persistent.parent.mkdir(parents=True, exist_ok=True)
        if not self.local.exists() and self.persistent.exists():
            shutil.copyfile(self.persistent, self.local)
        self.db = sqlite3.connect(self.local)
        self.db.execute("CREATE TABLE IF NOT EXISTS metadata (key TEXT PRIMARY KEY, value TEXT)")
        self.db.execute("CREATE TABLE IF NOT EXISTS labels (key TEXT PRIMARY KEY, value TEXT)")
        encoded = json.dumps(config, sort_keys=True)
        old = self.db.execute("SELECT value FROM metadata WHERE key='config'").fetchone()
        if old and old[0] != encoded:
            raise ValueError("Teacher configuration changed; start a new run")
        self.db.execute("INSERT OR IGNORE INTO metadata VALUES ('config', ?)", (encoded,))
        self.db.commit()
    def get(self, key):
        row = self.db.execute("SELECT value FROM labels WHERE key=?", (key,)).fetchone()
        return json.loads(row[0]) if row else None
    def put(self, key, value):
        self.db.execute("INSERT OR IGNORE INTO labels VALUES (?, ?)", (key, json.dumps(value, allow_nan=False)))
        self.db.commit()
    def count(self):
        return self.db.execute("SELECT COUNT(*) FROM labels").fetchone()[0]
    def stage_done(self, key):
        return self.db.execute("SELECT 1 FROM metadata WHERE key=?", ("stage:"+key,)).fetchone() is not None
    def mark_stage(self, key):
        self.db.execute("INSERT OR IGNORE INTO metadata VALUES (?, 'done')", ("stage:"+key,))
        self.db.commit()
    def snapshot(self):
        temporary = self.persistent.with_suffix(".tmp.sqlite")
        with sqlite3.connect(temporary) as destination:
            self.db.backup(destination)
        temporary.replace(self.persistent)
    def close(self):
        self.db.close()


def teacher_label(engine, board, nodes=50000, candidates=3):
    information = engine.analyse(board, chess.engine.Limit(nodes=nodes), multipv=min(candidates, board.legal_moves.count()), game=object())
    if isinstance(information, dict):
        information = [information]
    moves, ranks = [], []
    for info in information:
        move = info["pv"][0]
        score = info["score"].pov(board.turn)
        moves.append(move.uci())
        ranks.append(ranking_score(score.score(), score.mate()))
    best = information[0]
    if "wdl" not in best:
        raise ValueError("Pinned teacher did not produce WDL; enable UCI_ShowWDL")
    wdl = best["wdl"].pov(board.turn)
    return {"moves": moves, "probs": policy_distribution(ranks).tolist(), "value": wdl.expectation(),
            "wdl": [wdl.wins, wdl.draws, wdl.losses], "depth": best.get("depth"),
            "nodes": best.get("nodes"), "value_source": "pinned_stockfish_wdl"}


def replay_board(record):
    board = chess.Board(record["start_fen"])
    history = record["history"].split() if isinstance(record["history"], str) else record["history"]
    for uci in history:
        board.push_uci(uci)
    return board


def record_moves(record):
    return record["moves"].split() if isinstance(record["moves"],str) else record["moves"]


def clone_checkpoint_model(ns, state, path):
    rng = ns["rng_state"](state["sampler"])
    try:
        saved = ns["load_checkpoint"](path,state["identity"])
        model = ns["make_advanced_model"]().to(ns["DEVICE"])
        model.load_state_dict(saved["model"])
        model.eval()
        return model
    finally:
        ns["restore_rng"](rng,state["sampler"])


def outcome_targets(samples, winner):
    return [dict(sample, value=.5 if winner is None else float(sample["turn"] == winner),
                 value_source="completed_selfplay_outcome") for sample in samples]


def search_target(ns, model, board, simulations, rng, exploration=False):
    priors, _ = ns["evaluate_leaves"](model, [board])
    if exploration:
        indices = list(priors[0])
        noise = rng.dirichlet(np.full(len(indices), .3))
        priors[0] = {i:.75*priors[0][i] + .25*float(n) for i,n in zip(indices,noise)}
    root = ns["search"](model, board, n_simulations=simulations, batch=min(16, simulations), root_priors=priors[0])
    indices = list(root.children)
    counts = np.array([root.children[i].visits for i in indices], np.float64)
    if not len(indices) or counts.sum() == 0:
        raise ValueError("Search produced no visit targets")
    return indices, (counts/counts.sum()).astype(np.float32), root


class AdvancedTrainer:
    def __init__(self, ns, config):
        self.ns, self.config = ns, config
        self.base = Path(ns["RUN_DIR"])
        self.device = ns["DEVICE"]
        self.smoke = ns["MODE"] == "smoke"
        self.round_size = config["round_size"]
        self.data = ns["positions"]
        self.groups = training_groups(self.data, ns["train_idx"])
        self.validation = np.random.RandomState(101).choice(ns["validation_idx"], min(config["validation_size"], len(ns["validation_idx"])), replace=False)
        teacher_config = {"engine_sha256": ns["STOCKFISH_BINARY_SHA"], "nodes": config["teacher_nodes"],
                          "multipv": 3, "threads": 1, "temperature_cp": 80}
        self.cache_scope = hashlib.sha256(str(self.base.resolve()).encode()).hexdigest()[:12]
        self.teacher = TeacherStore(ns["ROOT"] / "teacher-cache" / self.cache_scope / "labels.sqlite",
                                    self.base / "teacher.sqlite", teacher_config)
        physical = {"version": 4, "mode": ns["MODE"], "real_graph": ns["USE_REAL_GRAPH"],
                    "dataset_sha256": ns["sha256"](ns["positions_path"]),
                    "graph_sha256": ns["sha256"](ns["GRAPH_PATH"]) if ns["USE_REAL_GRAPH"] else "synthetic",
                    "nodes_sha256": ns["sha256"](ns["NODES_PATH"]) if ns["USE_REAL_GRAPH"] else "synthetic",
                    "threshold": ns["EDGE_MIN_SYNAPSES"], "gain": ns["GAIN"], "alpha": ns["ALPHA"],
                    "perception_steps": ns["T_P"], "motor_steps": ns["T_A"], "relay_steps": ns["RELAY_STEPS"],
                    "injection_seed": 0, "current_amplitude": .5, "features": ns["N_FEATURES"], "moves": ns["N_MOVES"],
                    "graft": ns["GRAFT_KW"], "architecture": "history-sensory-curriculum-v4-v5",
                    "source_sha256": ns["MODEL_SOURCE_SHA256"], "config": config,
                    "engine_sha256": ns["STOCKFISH_BINARY_SHA"], "batch_size": ns["BATCH_SIZE"]}
        for field, name in (("sensory_ids","SENSORY_IDX"),("relay_ids","RELAY_IDX"),("premotor_ids","PREMOTOR_IDX"),("motor_ids","MOTOR_IDX")):
            physical[field] = ns["ids"][ns[name]].tolist()
        self.cache_identity = hashlib.sha256(json.dumps(physical,sort_keys=True).encode()).hexdigest()
        self.states = {}
        initial = copy.deepcopy(ns["model"].state_dict())
        for branch in config["branches"]:
            directory = self.base / branch
            directory.mkdir(exist_ok=True)
            manifest = dict(physical, version={"v4":4,"v5":5,"v55":5.5}[branch], branch=branch)
            if branch == "v55":
                manifest.update(architecture="ranked-lookahead-fly-only-v55", graft={}, current_amplitude=0,
                                assistance={"source":"material_minimax", "depth_plies":2, "ranked_k":4,
                                            "route":"sensory neurons only", "policy_choices":"all legal moves"})
            path = directory / "manifest.json"
            if path.exists() and json.loads(path.read_text()) != manifest:
                raise ValueError("Run configuration changed; use FLY_CHESS_NEW_RUN=1 once")
            ns["atomic_json"](path, manifest)
            # Save the exact implementation alongside every branch checkpoint.
            if ns.get("NOTEBOOK_SOURCE"):
                ns["atomic_json"](directory / "model-source.ipynb", ns["NOTEBOOK_SOURCE"])
            identity = hashlib.sha256(json.dumps(manifest,sort_keys=True).encode()).hexdigest()
            construction_rng = ns["rng_state"](np.random.RandomState(0))
            model = ns["make_advanced_model"]().to(self.device)
            ns["restore_rng"](construction_rng, np.random.RandomState(0))
            model.load_state_dict(initial)
            optimizer = torch.optim.AdamW(model.parameters(),lr=config["lr"],weight_decay=.01)
            scaler = torch.amp.GradScaler("cuda",enabled=self.device.type == "cuda",init_scale=128)
            state = {"model":model,"optimizer":optimizer,"scaler":scaler,"sampler":np.random.RandomState(0),
                     "step":0,"round":0,"phase":"foundation","best_loss":None,"stale":0,"status":"active",
                     "directory":directory,"identity":identity,"replay":[],"selfplay_games":0,"search_gate":False,
                     "collection_round":-1,"round_games":0,"collected_round":-1,"arena_pending":False,"champion_step":0}
            saved = ns["load_checkpoint"](directory / "main.pt",identity)
            if saved:
                model.load_state_dict(saved["model"])
                optimizer.load_state_dict(saved["optimizer"])
                scaler.load_state_dict(saved["scaler"])
                state.update({k:saved[k] for k in ("step","round","phase","best_loss","stale","status","selfplay_games","search_gate",
                                                     "collection_round","round_games","collected_round","arena_pending","champion_step")})
                state["sampler"].set_state(saved["sampler"])
                replay_path = directory / saved["replay_file"]
                if ns["sha256"](replay_path) != saved["replay_sha256"]:
                    raise ValueError("Replay snapshot integrity mismatch")
                with gzip.open(replay_path,"rt") as handle:state["replay"] = json.load(handle)
                if self.teacher.count() < saved["teacher_count"]:
                    raise ValueError("Teacher snapshot is older than the checkpoint")
                state["rng"] = saved["rng"]
            else:
                ns["initialize_signal_stats"](model, ns["train_idx"][:min(64,len(ns["train_idx"]))])
                # Every branch begins with the same stochastic training stream.
                torch.manual_seed(ns["SEED"])
                np.random.seed(ns["SEED"])
                state["rng"] = ns["rng_state"](state["sampler"])
                ns["save_checkpoint"](directory / "main.champion.pt",{
                    "identity":identity,"model":model.state_dict(),"step":0})
            self.states[branch] = state
        del initial
        # Remove the construction model to keep only the two active branches.
        ns.pop("model", None)
        self.cache = None
        self.cache_key = None

    def source_record(self, index):
        board = chess.Board(str(self.data["fen"][index]))
        key = board.fen(en_passant="fen")
        label = self.teacher.get(key)
        if not label:
            label = {"moves":self.data["policy_moves"][index].tolist(),
                     "probs":self.data["policy_probs"][index].tolist(),"value":float(self.data["value"][index]),
                     "value_source":"source_cp_proxy"}
        return {"start_fen":key,"history":[],"turn":board.turn,**label,"index":int(index)}

    def save(self, state):
        self.teacher.snapshot()
        replay_path = state["directory"] / f"replay-{state['selfplay_games']:06d}.json.gz"
        if not replay_path.exists():
            temporary = replay_path.with_suffix(".tmp")
            with gzip.open(temporary,"wt") as handle:json.dump(state["replay"],handle,allow_nan=False)
            temporary.replace(replay_path)
        saved = {k:state[k] for k in ("identity","step","round","phase","best_loss","stale","status","selfplay_games","search_gate",
                                     "collection_round","round_games","collected_round","arena_pending","champion_step")}
        saved.update(model=state["model"].state_dict(),optimizer=state["optimizer"].state_dict(),
                     scaler=state["scaler"].state_dict(),sampler=state["sampler"].get_state(),
                     rng=self.ns["rng_state"](state["sampler"]),teacher_count=self.teacher.count(),
                     replay_file=replay_path.name,replay_sha256=self.ns["sha256"](replay_path))
        self.ns["save_checkpoint"](state["directory"] / "main.pt",saved)
        self.ns["atomic_json"](state["directory"] / "progress.json",{k:saved[k] for k in ("step","round","phase","best_loss","stale","status","selfplay_games","search_gate","teacher_count")})

    def loss_batch(self, state, records, cache=None):
        boards = [replay_board(r) for r in records]
        features = None if cache else torch.tensor(np.stack([self.ns["encode_board"](b) for b in boards]),device=self.device)
        width = max(len(record_moves(r)) for r in records)
        indices = np.zeros((len(records),width),np.int64)
        probs = np.zeros((len(records),width),np.float32)
        for row,(record,board) in enumerate(zip(records,boards)):
            choices = [self.ns["move_index"](chess.Move.from_uci(m),board.turn) for m in record_moves(record)]
            indices[row] = choices[0]
            indices[row,:len(choices)] = choices
            probs[row,:len(choices)] = record["probs"]
        target = torch.tensor([r["value"] for r in records],device=self.device,dtype=torch.float32)
        state["model"].train()
        with torch.autocast(device_type=self.device.type,enabled=self.device.type == "cuda",dtype=torch.float16):
            if cache:
                logits,value = self.ns["cached_forward"](state["model"],cache,[r["index"] for r in records])
            else:
                logits,value = state["model"](features)
            policy = soft_policy_loss(self.ns["mask_logits"](logits,boards),indices,probs)
            value_loss = F.binary_cross_entropy_with_logits(value.float(),target)
            loss = policy + self.config["value_weight"]*value_loss
        if not torch.isfinite(loss):
            raise ValueError("Nonfinite loss; previous checkpoint retained")
        state["optimizer"].zero_grad(set_to_none=True)
        state["scaler"].scale(loss).backward()
        state["scaler"].unscale_(state["optimizer"])
        gradients = [p.grad for p in state["model"].parameters() if p.grad is not None]
        if any(not torch.isfinite(g).all() for g in gradients):
            if state["scaler"].is_enabled() and state["scaler"].get_scale() > 1:
                state["scaler"].step(state["optimizer"])  # scaler skips the overflowed update
                state["scaler"].update()
                state["optimizer"].zero_grad(set_to_none=True)
                print("AMP overflow skipped; scale",state["scaler"].get_scale(),flush=True)
                return None
            raise ValueError("Nonfinite gradients at minimum loss scale; previous checkpoint retained")
        torch.nn.utils.clip_grad_norm_(state["model"].parameters(),5,error_if_nonfinite=True)
        state["scaler"].step(state["optimizer"])
        state["scaler"].update()
        return float(loss.detach())

    def relabel(self, indices, deadline, limit=None):
        missing = [i for i in indices if self.teacher.get(chess.Board(str(self.data["fen"][i])).fen(en_passant="fen")) is None]
        if not missing:
            return
        with self.ns["open_stockfish"]() as engine:
            engine.configure({"Threads":1,"Hash":128,"Skill Level":20,"UCI_LimitStrength":False,"UCI_ShowWDL":True})
            for offset, i in enumerate(missing[:limit or self.config["relabel_per_round"]]):
                self.ns["check_stop"]()
                if time.monotonic() >= deadline:
                    break
                board = chess.Board(str(self.data["fen"][i]))
                self.teacher.put(board.fen(en_passant="fen"),teacher_label(engine,board,self.config["teacher_nodes"]))
                if offset % 32 == 0:
                    print("teacher labels",self.teacher.count(),flush=True)
        self.teacher.snapshot()

    def search_gate(self, state):
        # A small training-only probe. Validation/test positions never enter replay.
        indices = stratified_pool(self.groups,min(self.config["gate_positions"],len(self.ns["train_idx"])),np.random.RandomState(313),"general")
        policy_score, search_score = [], []
        with self.ns["open_stockfish"]() as engine:
            engine.configure({"Threads":1,"Hash":128,"Skill Level":20,"UCI_LimitStrength":False,"UCI_ShowWDL":True})
            for i in indices:
                board = chess.Board(str(self.data["fen"][i]))
                raw = self.ns["choose_move"](state["model"],board,False)
                choices,visits,_ = search_target(self.ns,state["model"],board,self.config["simulations"],state["sampler"])
                improved = self.ns["index_to_move"](choices[int(visits.argmax())],board)
                def evaluate(move):
                    info = engine.analyse(board,chess.engine.Limit(nodes=self.config["teacher_nodes"]),root_moves=[move],game=object())
                    return info["score"].pov(board.turn).score(mate_score=10000)
                policy_score.append(evaluate(raw)); search_score.append(evaluate(improved))
        delta = np.asarray(search_score)-np.asarray(policy_score)
        accepted = float(delta.mean()) > 0 and np.count_nonzero(delta > 0) > np.count_nonzero(delta < 0)
        report = {"n":len(indices),"mean_search_gain_cp":float(delta.mean()),"accepted":bool(accepted),
                  "note":"Small training-only engineering gate, not a strength estimate"}
        self.ns["atomic_json"](state["directory"] / "search-gate.json",report)
        print("v5 search gate",report,flush=True)
        return bool(accepted)

    def collect_selfplay(self, state, deadline):
        path = state["directory"] / "selfplay-in-progress.json"
        if state["collection_round"] != state["round"]:
            state["collection_round"],state["round_games"] = state["round"],0
        while state["round_games"] < self.config["games_per_round"]:
            saved = json.loads(path.read_text()) if path.exists() else None
            if saved:
                if saved["game_number"] <= state["selfplay_games"]:
                    path.unlink();continue
                if saved["generator_step"] != state["step"]:
                    raise ValueError("In-progress game belongs to different generator weights")
                board = replay_board(saved); samples = saved["samples"]
                self.ns["restore_rng"](pickle.loads(bytes.fromhex(saved["rng"])),state["sampler"])
                opponent_info = saved.get("opponent")
            else:
                board = chess.Board()
                opening = self.ns["make_opening_pairs"](max(1,self.config["games_per_round"]),seed=41+state["selfplay_games"])[0]
                for move in opening: board.push_uci(move)
                samples = []
                opponent_info = None
                champion = state["directory"] / "main.champion.pt"
                if state["champion_step"] > 0 and state["sampler"].rand() < .25:
                    opponent_info = {"checkpoint":champion.name,"sha256":self.ns["sha256"](champion),
                                     "agent_color":bool(state["sampler"].randint(2))}
            opponent = None
            if opponent_info:
                checkpoint_path = state["directory"] / opponent_info["checkpoint"]
                if self.ns["sha256"](checkpoint_path) != opponent_info["sha256"]:
                    raise ValueError("Self-play opponent changed during an unfinished game")
                opponent = clone_checkpoint_model(self.ns,state,checkpoint_path)
            while self.ns["terminal_value"](board) is None and len(board.move_stack) < self.config["game_cap"]:
                self.ns["check_stop"]()
                if time.monotonic() >= deadline:
                    return False
                actor_turn = opponent is None or board.turn == opponent_info["agent_color"]
                generator = state["model"] if actor_turn else opponent
                choices,probs,_ = search_target(self.ns,generator,board,self.config["simulations"],state["sampler"],True)
                sample = {"start_fen":chess.STARTING_FEN,"history":" ".join(m.uci() for m in board.move_stack),
                          "turn":board.turn,"moves":" ".join(self.ns["index_to_move"](i,board).uci() for i in choices),"probs":probs.tolist(),
                          "generator_step":state["step"],"simulations":self.config["simulations"]}
                if actor_turn:samples.append(sample)
                choice = state["sampler"].choice(len(choices),p=probs.astype(np.float64)/probs.astype(np.float64).sum()) if len(board.move_stack) < 30 else int(probs.argmax())
                board.push(self.ns["index_to_move"](choices[choice],board))
                self.ns["atomic_json"](path,{"start_fen":chess.STARTING_FEN,"history":[m.uci() for m in board.move_stack],
                    "samples":samples,"generator_step":state["step"],"opponent":opponent_info,
                    "game_number":state["selfplay_games"]+1,"rng":pickle.dumps(self.ns["rng_state"](state["sampler"])).hex()})
            outcome = board.outcome(claim_draw=True)
            records = outcome_targets(samples,outcome.winner) if outcome is not None else []
            # Split by canonical board family even for history-rich self-play.
            records = [r for r in records if self.ns["partition"](replay_board(r)) == "train"]
            if records:
                state["replay"] = (state["replay"] + records)[-self.config["replay_size"]:]
            state["selfplay_games"] += 1
            state["round_games"] += 1
            game_dir = state["directory"] / "selfplay-games"
            game_dir.mkdir(exist_ok=True)
            self.ns["atomic_json"](game_dir / f"{state['selfplay_games']:06d}.json",{
                "status":"complete" if outcome is not None else "unresolved", "result":outcome.result() if outcome else None,
                "reason":outcome.termination.name if outcome else "safety_cap", "moves":[m.uci() for m in board.move_stack],
                "generator_step":state["step"],"train_records":len(records)})
            self.save(state)
            if path.exists():path.unlink()
            print("selfplay",state["selfplay_games"],"replay",len(state["replay"]),"result",outcome.result() if outcome else "unresolved",flush=True)
        state["collected_round"] = state["round"]
        return True

    def validate(self, state):
        policy_sum,value_sum,squared,correct = 0.,0.,0.,0
        target_values = []
        with self.ns["inference_mode"](state["model"]):
            for offset in range(0,len(self.validation),self.ns["BATCH_SIZE"]):
                records = [self.source_record(i) for i in self.validation[offset:offset+self.ns["BATCH_SIZE"]]]
                boards = [replay_board(r) for r in records]
                features = torch.tensor(np.stack([self.ns["encode_board"](b) for b in boards]),device=self.device)
                logits,value = state["model"](features)
                logits = self.ns["mask_logits"](logits,boards)
                width = max(len(r["moves"]) for r in records)
                choices = np.zeros((len(records),width),np.int64)
                probabilities = np.zeros((len(records),width),np.float32)
                for row,(record,board) in enumerate(zip(records,boards)):
                    ids = [self.ns["move_index"](chess.Move.from_uci(m),board.turn) for m in record["moves"]]
                    choices[row] = ids[0];choices[row,:len(ids)] = ids
                    probabilities[row,:len(ids)] = record["probs"]
                targets = torch.tensor([r["value"] for r in records],device=self.device,dtype=torch.float32)
                policy_sum += float(soft_policy_loss(logits,choices,probabilities))*len(records)
                value_sum += float(F.binary_cross_entropy_with_logits(value,targets,reduction="sum"))
                squared += float(((value.sigmoid()-targets)**2).sum())
                correct += int((logits.argmax(1)==torch.tensor(choices[:,0],device=self.device)).sum())
                target_values.extend(targets.cpu().tolist())
        baseline = float(np.var(target_values))
        metrics = {"legal_move_match":correct/len(self.validation),"value_mse":squared/len(self.validation),
                   "validation_loss":(policy_sum+self.config["value_weight"]*value_sum)/len(self.validation),
                   "value_skill_vs_validation_mean":1-squared/len(self.validation)/baseline if baseline>1e-10 else None,
                   "value_source":"fixed_pinned_teacher_wdl","n":len(self.validation)}
        loss = metrics["validation_loss"]
        if state["best_loss"] is None or loss < state["best_loss"]-self.config["min_delta"]:
            state["best_loss"],state["stale"] = loss,0
            self.ns["save_checkpoint"](state["directory"] / "main.best.pt",{
                "identity":state["identity"],"model":state["model"].state_dict(),"step":state["step"],"metrics":metrics})
        else:
            state["stale"] += 1
        self.ns["atomic_json"](state["directory"] / "validation-latest.json",dict(metrics,step=state["step"],phase=state["phase"]))
        print(state["directory"].name,"round",state["round"],"step",state["step"],"phase",state["phase"],metrics,flush=True)
        return metrics

    def arena(self, branch, state, deadline):
        directory = state["directory"] / "arenas" / f"step-{state['step']:08d}"
        directory.mkdir(parents=True,exist_ok=True)
        records_path = directory / "records.json"
        records = json.loads(records_path.read_text()) if records_path.exists() else []
        opponent = clone_checkpoint_model(self.ns,state,state["directory"] / "main.champion.pt")
        def mover(model):
            if branch != "v5":return self.ns["model_opponent"](model,False)
            def play(board,clock,increment):
                choices,probs,_ = search_target(self.ns,model,board,self.config["simulations"],state["sampler"])
                return self.ns["index_to_move"](choices[int(probs.argmax())],board)
            return play
        current,old = mover(state["model"]),mover(opponent)
        previous_dir = self.ns["RUN_DIR"]
        self.ns["RUN_DIR"] = directory
        try:
            for pair,opening in enumerate(self.ns["make_opening_pairs"](self.config["arena_pairs"],seed=79)):
                for color in ("white","black"):
                    if time.monotonic()>=deadline:return False
                    game_id = f"{pair}-{color}"
                    if any(r["id"]==game_id for r in records):continue
                    white,black = (current,old) if color=="white" else (old,current)
                    game = self.ns["play_game"](white,black,opening,game_id,initial_clock=120,increment=1,max_plies=500)
                    score = game["score"]
                    if score is not None and color=="black":score=1-score
                    records.append({"id":game_id,"pair":pair,"color":color,"score":score,"reason":game["reason"]})
                    self.ns["atomic_json"](records_path,records)
            summary = self.ns["strength_summary"](records)
            summary.update(candidate_step=state["step"],opponent_step=state["champion_step"],
                           simulations=self.config["simulations"] if branch=="v5" else 0,
                           promotion_rule="At least 62.5% paired score; engineering gate, not statistical proof")
            promoted = summary["complete_pairs"]==self.config["arena_pairs"] and summary["score"]>=.625
            summary["promoted"] = bool(promoted)
            self.ns["atomic_json"](directory / "summary.json",summary)
            if promoted:
                self.ns["save_checkpoint"](state["directory"] / "main.champion.pt",{
                    "identity":state["identity"],"model":state["model"].state_dict(),"step":state["step"]})
                state["champion_step"],state["stale"] = state["step"],0
            state["arena_pending"] = False
            print(branch,"arena",summary,flush=True)
            return True
        finally:
            self.ns["RUN_DIR"] = previous_dir

    def run(self, minutes):
        deadline = time.monotonic()+minutes*60
        ns = self.ns
        try:
            # Freeze the whole validation label set before the first update.
            self.relabel(self.validation,deadline,limit=len(self.validation))
            if any(self.teacher.get(chess.Board(str(self.data["fen"][i])).fen(en_passant="fen")) is None for i in self.validation):
                print("Validation labelling paused; rerun to finish before training",flush=True)
                return
            while time.monotonic()<deadline and any(s["status"]=="active" for s in self.states.values()):
                for branch,state in self.states.items():
                    if state["status"]!="active" or time.monotonic()>=deadline:continue
                    if state["round"] > min(s["round"] for s in self.states.values() if s["status"]=="active"):
                        continue
                    ns["restore_rng"](state["rng"],state["sampler"])
                    ns["RUN_DIR"] = state["directory"]
                    if state["arena_pending"]:
                        if not self.arena(branch,state,deadline):return
                    old_phase = state["phase"]
                    state["phase"] = "foundation" if state["round"]<self.config["foundation_rounds"] else "general" if state["round"]<self.config["pretrain_rounds"] else "consolidation"
                    if state["phase"] != old_phase:state["stale"] = 0
                    # The deterministic shard seed is branch independent. Both branches
                    # see the same supervised curriculum; shards rotate every round.
                    pool = stratified_pool(self.groups,self.config["pool_size"],np.random.RandomState(1000+state["round"]),state["phase"])
                    shard_key = hashlib.sha256(pool.tobytes()).hexdigest()
                    if not self.teacher.stage_done(shard_key):
                        self.relabel(pool,deadline)
                        if time.monotonic()<deadline:self.teacher.mark_stage(shard_key)
                    if time.monotonic()>=deadline:break
                    if branch=="v5" and state["round"]>=self.config["pretrain_rounds"]:
                        if not state["search_gate"] and state["round"]%self.config["gate_interval"]==0:
                            state["search_gate"] = self.search_gate(state)
                        if state["search_gate"] and state["collected_round"] != state["round"]:
                            if not self.collect_selfplay(state,deadline):break
                    # Frozen cache is shared between branches with the same shard.
                    key = hashlib.sha256(pool.tobytes()).hexdigest()
                    if self.cache_key!=key:
                        cache_root = ns["ROOT"] / "perception-cache" / ns["RUN_ID"]
                        tag = "shared-"+self.cache_scope+"-"+key[:12]
                        if cache_root.exists():
                            for old in cache_root.glob("shared-"+self.cache_scope+"-*"):
                                if old.name != tag:shutil.rmtree(old)
                        self.cache = ns["cache_perception"](state["model"],tag,pool,key+self.cache_identity)
                        self.cache_key = key
                    preflight_path = state["directory"] / "learning-preflight.json"
                    if state["step"] == 0:
                        if preflight_path.exists():
                            if not json.loads(preflight_path.read_text())["passed"] and not self.smoke and branch != "v55":
                                raise RuntimeError("Saved learning preflight failed; long training remains stopped")
                        else:
                            ns["learning_preflight"](state["model"],self.cache)
                    last_save = time.monotonic()
                    while state["step"] < (state["round"]+1)*self.round_size:
                        ns["check_stop"]()
                        if time.monotonic()>=deadline:
                            self.save(state);return
                        use_replay = branch=="v5" and bool(state["replay"]) and state["sampler"].rand()<self.config["selfplay_fraction"]
                        if use_replay:
                            selected = state["sampler"].choice(len(state["replay"]),min(ns["BATCH_SIZE"],len(state["replay"])),replace=False)
                            update = self.loss_batch(state,[state["replay"][i] for i in selected])
                        else:
                            selected = state["sampler"].choice(pool,min(ns["BATCH_SIZE"],len(pool)),replace=False)
                            update = self.loss_batch(state,[self.source_record(i) for i in selected],self.cache)
                        if update is None:continue
                        state["step"]+=1
                        if time.monotonic()-last_save>=60:
                            self.save(state);last_save=time.monotonic()
                    state["round"]+=1
                    self.validate(state)
                    if not self.smoke and state["round"]%self.config["arena_every"]==0:
                        state["arena_pending"] = True
                        self.save(state)
                        if not self.arena(branch,state,deadline):return
                    # A phase gets its own patience. Self-play gets a complete patience
                    # window after the search gate succeeds, then returns control.
                    if state["round"]>=self.config["pretrain_rounds"] and state["stale"]>=self.config["patience"]:
                        state["status"]="plateau"
                    if self.smoke and state["round"]>=self.config["smoke_rounds"]:state["status"]="smoke_complete"
                    self.save(state)
                    state["rng"] = ns["rng_state"](state["sampler"])
        except (KeyboardInterrupt, ns["RunPaused"]) as exc:
            print("Paused:",str(exc),flush=True)
        finally:
            # Each branch retains its own RNG stream, including dropout/CUDA RNG.
            for state in self.states.values():
                if ns.get("RUN_DIR")==state["directory"]:
                    state["rng"] = ns["rng_state"](state["sampler"])
            for state in self.states.values():
                ns["restore_rng"](state["rng"],state["sampler"])
                self.save(state)
            ns["RUN_DIR"] = self.base


def advanced_config(branches, smoke):
    def integer(name,default):return int(os.environ.get(name,str(default)))
    config = {"branches":list(branches),"round_size":2 if smoke else integer("FLY_CHESS_ROUND_STEPS",500),
        "pool_size":64 if smoke else integer("FLY_CHESS_TRAIN_POOL",32768),
        "foundation_rounds":1 if smoke else 2,"pretrain_rounds":1 if smoke else 6,
        "validation_size":32 if smoke else 512,"validation_relabel":2 if smoke else 32,
        "teacher_nodes":100 if smoke else integer("FLY_CHESS_TEACHER_NODES",50000),
        "relabel_per_round":4 if smoke else integer("FLY_CHESS_RELABEL_PER_ROUND",256),
        "lr":float(os.environ.get("FLY_CHESS_LR","0.0003")),"value_weight":2.0,
        "patience":integer("FLY_CHESS_PATIENCE",10),"min_delta":.002,
        "simulations":4 if smoke else integer("FLY_CHESS_SIMULATIONS",64),
        "gate_positions":2 if smoke else 32,"gate_interval":1 if smoke else 2,
        "games_per_round":1 if smoke else integer("FLY_CHESS_SELFPLAY_GAMES",8),
        "game_cap":16 if smoke else 500,"replay_size":256 if smoke else 50000,
        "selfplay_fraction":.35,"smoke_rounds":2}
    config.update(arena_every=4,arena_pairs=4)
    if any(config[k]<=0 for k in ("round_size","pool_size","teacher_nodes","patience","simulations","games_per_round")):
        raise ValueError("Training configuration values must be positive")
    return config


def benchmark_advanced(trainer, minutes=15, pairs=20):
    """Paired, immutable, interleaved network/search/helper comparisons."""
    ns = trainer.ns
    deadline = time.monotonic()+minutes*60
    base = ns["RUN_DIR"]
    entries = []
    snapshots = []
    for branch,state in trainer.states.items():
        checkpoint = state["directory"] / "main.best.pt"
        if not checkpoint.exists():checkpoint=state["directory"] / "main.pt"
        snapshot = clone_checkpoint_model(ns,state,checkpoint)
        snapshots.append(snapshot)
        fingerprint = ns["sha256"](checkpoint)
        modes = ["network_search","network_raw"] if branch == "v5" else ["network_raw"]
        if branch == "v55":modes=["fly_ranked","ranker_top1","no_candidates","reverse_candidates","no_senses"]
        for mode in modes:
            def agent(board,clock,increment,mode=mode,snapshot=snapshot,state=state):
                if mode == "ranker_top1":
                    return chess.Move.from_uci(ns["ranked_lookahead"](board.fen(en_passant="fen"))[0][0])
                if mode == "network_search":
                    choices,probabilities,_ = search_target(ns,snapshot,board,trainer.config["simulations"],state["sampler"])
                    return ns["index_to_move"](choices[int(probabilities.argmax())],board)
                if mode in ("no_candidates","reverse_candidates","no_senses"):
                    features = torch.tensor(ns["encode_board"](board)[None],device=ns["DEVICE"])
                    with ns["inference_mode"](snapshot):
                        logits,_ = snapshot(features,intervention=mode)
                    return ns["index_to_move"](int(ns["mask_logits"](logits,[board]).argmax(1)[0]),board)
                return ns["choose_move"](snapshot,board,False)
            entries.append((branch,state,fingerprint,mode,agent))
    with ns["open_stockfish"]() as engine:
        minimum,maximum = engine.options["UCI_Elo"].min,engine.options["UCI_Elo"].max
        ratings = [minimum] if trainer.smoke else sorted(set([minimum,min(maximum,1600),min(maximum,2000)]))
        try:
            openings = ns["make_opening_pairs"](1 if trainer.smoke else pairs,seed=113)
            for pair,opening in enumerate(openings):
                for rating in ratings:
                    for branch,state,fingerprint,mode,agent in entries:
                        directory=state["directory"] / "strength" / fingerprint[:16] / mode / f"sf-{rating}"
                        directory.mkdir(parents=True,exist_ok=True)
                        config={"checkpoint_sha256":fingerprint,"engine_sha256":ns["STOCKFISH_BINARY_SHA"],
                                "reference_elo":rating,"clock":120,"increment":1,"opening_seed":113,
                                "simulations":trainer.config["simulations"] if mode=="network_search" else 0,
                                "assist_mode":"material_minimax_depth2_top4" if branch=="v55" else "none", "mode":mode,
                                "max_plies":12 if trainer.smoke else 600}
                        config_path=directory / "config.json"
                        if config_path.exists() and json.loads(config_path.read_text())!=config:
                            raise ValueError("Benchmark configuration changed")
                        ns["atomic_json"](config_path,config)
                        record_path=directory / "records.json"
                        records=json.loads(record_path.read_text()) if record_path.exists() else []
                        ns["RUN_DIR"]=directory
                        engine.configure({"Threads":1,"Hash":64,"Skill Level":20,"UCI_LimitStrength":True,"UCI_Elo":rating})
                        def reference(board,clock,increment):
                            return engine.play(board,chess.engine.Limit(white_clock=clock if board.turn else 120,
                                black_clock=clock if not board.turn else 120,white_inc=1,black_inc=1),game=game_token).move
                        for color in ("white","black"):
                            game_id=f"{pair}-{color}"
                            if any(r["id"]==game_id for r in records):continue
                            if time.monotonic()>=deadline:return
                            game_token = object()
                            white,black=(agent,reference) if color=="white" else (reference,agent)
                            game=ns["play_game"](white,black,opening,game_id,initial_clock=120,increment=1,max_plies=config["max_plies"])
                            score=game["score"]
                            if score is not None and color=="black":score=1-score
                            records.append({"id":game_id,"pair":pair,"color":color,"score":score,"reason":game["reason"]})
                            ns["atomic_json"](record_path,records)
                            summary=ns["strength_summary"](records,rating,"Stockfish UCI_Elo benchmark at 120+1")
                            summary.update(checkpoint_sha256=fingerprint,configuration=config,target_pairs=len(openings))
                            ns["atomic_json"](directory / "summary.json",summary)
                            print(branch,mode,"benchmark",rating,game_id,score,game["reason"],flush=True)
        finally:
            ns["RUN_DIR"]=base

In [ ]:
def stream_zst_lines(url):
    """Yields raw lines from a remote .zst file without downloading it whole."""
    dctx = zstandard.ZstdDecompressor()
    with urllib.request.urlopen(url) as response, dctx.stream_reader(response) as reader:
        buffer = b""
        while True:
            chunk = reader.read(1 << 20)
            if not chunk:
                break
            buffer += chunk
            *lines, buffer = buffer.split(b"\n")
            yield from lines
        if buffer:
            yield buffer

Dataset is pinned once. Cached single-move labels are upgraded honestly; strong teacher relabelling runs incrementally and survives disconnects.

In [ ]:
prepare_advanced_data(globals())

In [ ]:
import tarfile

STOCKFISH_DIR = CHESS_DATA / "stockfish"
STOCKFISH_BIN = STOCKFISH_DIR / "stockfish"
STOCKFISH_URL = "https://github.com/official-stockfish/Stockfish/releases/download/sf_19/stockfish-linux-x86-64-universal.tar.gz"
STOCKFISH_LOCK = STOCKFISH_DIR / "source.lock.json"

STOCKFISH_DIR.mkdir(parents=True, exist_ok=True)
if not STOCKFISH_BIN.exists():
    archive = STOCKFISH_DIR / "stockfish.tar.gz"
    print(f"Downloading {STOCKFISH_URL}", flush=True)
    urllib.request.urlretrieve(STOCKFISH_URL, archive)
    digest = sha256(archive)
    # The release also ships its own source tree, which includes docs like wiki/Stockfish-FAQ.md —
    # match the exact binary name (the asset's own filename, minus .tar.gz), not just a substring.
    expected_name = Path(STOCKFISH_URL).name.removesuffix(".tar.gz")
    with tarfile.open(archive) as tar:
        member = next(m for m in tar.getmembers() if m.isfile() and Path(m.name).name == expected_name)
        member.name = "stockfish"
        tar.extract(member, STOCKFISH_DIR, filter="data")
    archive.unlink()
    STOCKFISH_BIN.chmod(0o755)
    STOCKFISH_LOCK.write_text(json.dumps({"url": STOCKFISH_URL, "sha256": digest}, indent=2))
    print("pinned Stockfish sha256", digest)
else:
    pin = json.loads(STOCKFISH_LOCK.read_text())["sha256"]
    # Re-deriving from the extracted binary isn't meaningful (tar strips metadata); trust the pin
    # recorded at download time and only assert the binary still runs.
    print("Stockfish already present, pinned sha256", pin)

STOCKFISH_BINARY_SHA = sha256(STOCKFISH_BIN)
import shutil
# Drive stores the pin; execute from the VM, where executable permissions work.
runtime_engine = ROOT / "engine-runtime" / STOCKFISH_BINARY_SHA / "stockfish"
runtime_engine.parent.mkdir(parents=True,exist_ok=True)
if not runtime_engine.exists() or sha256(runtime_engine) != STOCKFISH_BINARY_SHA:
    shutil.copyfile(STOCKFISH_BIN,runtime_engine)
runtime_engine.chmod(0o755)
STOCKFISH_BIN = runtime_engine
assert sha256(STOCKFISH_BIN) == STOCKFISH_BINARY_SHA
engine = chess.engine.SimpleEngine.popen_uci(str(STOCKFISH_BIN))
ENGINE_ID = dict(engine.id)
ENGINE_OPTIONS = {name: {"min": opt.min, "max": opt.max, "default": opt.default}
                  for name, opt in engine.options.items() if name in ("UCI_Elo", "Skill Level")}
engine.quit()
print("Stockfish", ENGINE_ID, STOCKFISH_BINARY_SHA)

In [ ]:
USE_REAL_GRAPH = MODE == "full" or os.environ.get("FLY_CHESS_REAL_GRAPH") == "1"
if USE_REAL_GRAPH:
    GRAPH_DIR = DATA / "connectome"
    GRAPH_DIR.mkdir(parents=True, exist_ok=True)
    GRAPH_PATH = GRAPH_DIR / "full-graph.npz"
    NODES_PATH = GRAPH_DIR / "full-nodes.npz"

    # Same public MaleCNS v1.0 source fly-heaven's fruitless and fly-wirehead projects pin (CC BY 4.0).
    MALECNS_SOURCES = {
        "annotations.feather": {
            "url": "https://storage.googleapis.com/flyem-male-cns/v1.0/connectome-data/flat-connectome/body-annotations-male-cns-v1.0-minconf-0.5.feather",
            "bytes": 14483314,
            "sha256": "2177e246113e4cfbf1e7772ec37c6da1955ff22e8063d0b1f833101f99a9a3b2",
        },
        "neurotransmitters.feather": {
            "url": "https://storage.googleapis.com/flyem-male-cns/v1.0/connectome-data/flat-connectome/body-neurotransmitters-male-cns-v1.0.feather",
            "bytes": 43282834,
            "sha256": "95c9289220663abeb3409f3ad9e5a7f8a53f8093f5139d15502cd08da8879621",
        },
        "edges.feather": {
            "url": "https://storage.googleapis.com/flyem-male-cns/v1.0/connectome-data/flat-connectome/connectome-weights-male-cns-v1.0-minconf-0.5.feather",
            "bytes": 1051241946,
            "sha256": "e35da783d1c686b2b58b3b87cd6a403ae43bfcfba8bff28e08ef752c1a56afc1",
        },
    }
    for name, info in MALECNS_SOURCES.items():
        if name != "annotations.feather" and GRAPH_PATH.exists() and NODES_PATH.exists():
            continue
        path = RAW / name
        if not path.exists():
            print(f"Downloading {name} ({info['bytes'] / 1e6:,.0f} MB)...", flush=True)
            partial = path.with_suffix(".partial")
            urllib.request.urlretrieve(info["url"], partial)
            partial.replace(path)
        digest = sha256(path)
        assert digest == info["sha256"], f"Checksum mismatch: {name}"
        print(f"verified  {name:27s} {path.stat().st_size / 1e6:>8,.1f} MB")

    if not NODES_PATH.exists() or not GRAPH_PATH.exists():
        import pyarrow as pa
        import pyarrow.compute as pc
        import pyarrow.feather as feather
        from scipy.sparse import csr_matrix, save_npz

        print("Building the connectome graph (fruitless's own recipe)...", flush=True)
        started = time.perf_counter()
        rows = feather.read_table(RAW / "annotations.feather").to_pylist()
        rows = sorted((r for r in rows if r["superclass"] and "tbc" not in r["superclass"]), key=lambda r: r["bodyId"])
        ids = np.array([r["bodyId"] for r in rows], dtype=np.int64)

        nt_table = feather.read_table(RAW / "neurotransmitters.feather", columns=["body", "consensus_nt"])
        nt_table = nt_table.filter(pc.is_in(nt_table["body"], pa.array(ids)))
        nts = dict(zip(nt_table["body"].to_pylist(), nt_table["consensus_nt"].to_pylist()))
        # Assumed fast signs, not measured receptor-specific physiology. Others get zero fast weight.
        signs = np.array([{"acetylcholine": 1, "gaba": -1, "glutamate": -1}.get(nts.get(int(i)), 0) for i in ids], dtype=np.int8)

        source = pa.memory_map(str(RAW / "edges.feather"))
        reader = pa.ipc.open_file(source)
        pre, post, weight = [], [], []
        for b in range(reader.num_record_batches):
            t = pa.Table.from_batches([reader.get_batch(b)])
            t = t.filter(pc.and_(pc.is_in(t["body_pre"], pa.array(ids)), pc.is_in(t["body_post"], pa.array(ids))))
            pre.append(np.searchsorted(ids, t["body_pre"].to_numpy()).astype(np.int32))
            post.append(np.searchsorted(ids, t["body_post"].to_numpy()).astype(np.int32))
            weight.append(t["weight"].to_numpy().astype(np.float32))
            if b % 400 == 0:
                print(f"  edge batch {b + 1}/{reader.num_record_batches}", flush=True)
        pre, post, weight = map(np.concatenate, (pre, post, weight))
        graph = csr_matrix((weight, (pre, post)), shape=(len(ids), len(ids)))
        graph.sort_indices()
        save_npz(GRAPH_PATH, graph)
        np.savez(NODES_PATH, ids=ids, signs=signs)
        print(f"{len(ids):,} neurons, {graph.nnz:,} edges built in {time.perf_counter() - started:.0f} s")

    nodes = np.load(NODES_PATH)
    ids, base_signs = nodes["ids"], nodes["signs"]
    W = load_npz(GRAPH_PATH)  # CSR, rows = presynaptic, values = synapse counts
    N_NEURONS = W.shape[0]
    assert (N_NEURONS, W.nnz) == (166_606, 25_574_615), "Graph differs from the reference build"

    ann = (pd.read_feather(RAW / "annotations.feather", columns=["bodyId", "superclass", "class", "type"])
           .set_index("bodyId").reindex(ids))

    SENSORY_IDX = np.nonzero(ann["superclass"].isin(["cb_sensory", "ol_sensory"]).values)[0]
    MOTOR_IDX = np.nonzero(ann["superclass"].isin(["descending_neuron", "cb_motor", "vnc_motor"]).values)[0]
    assert len(SENSORY_IDX) == 10_966 and len(MOTOR_IDX) == 2_129
    print(f"sensory {len(SENSORY_IDX):,}  ·  DN+motor {len(MOTOR_IDX):,}  ·  total {N_NEURONS:,}")

else:
    from scipy.sparse import csr_matrix
    N_NEURONS = 384
    ids = np.arange(N_NEURONS)
    base_signs = np.where(np.arange(N_NEURONS) % 5 == 0, -1, 1).astype(np.float32)
    sensory_count = 128
    SENSORY_IDX = np.arange(sensory_count)
    MOTOR_IDX = np.arange(320, 384)
    generator = np.random.RandomState(SEED)
    pre = generator.randint(0, N_NEURONS, 4000)
    post = generator.randint(0, N_NEURONS, 4000)
    W = csr_matrix((generator.randint(3, 12, 4000).astype(np.float32), (pre, post)),
                   shape=(N_NEURONS, N_NEURONS))
    ann = pd.DataFrame({"superclass": ["cb_sensory"] * 128 + ["central"] * 192 + ["cb_motor"] * 64,
                        "class": [""] * N_NEURONS})
print("graph", W.shape, W.nnz)

In [ ]:
EDGE_MIN_SYNAPSES = 3  # picked below in the Phase-0 benchmark; kept ≥ here so later cells still run standalone


def build_signed_transpose(W, signs, min_synapses):
    """Wt[post, pre] = sign(pre) * synapse_count, thresholded and row-normalised (row = post)."""
    keep = W.data >= min_synapses
    pre = np.repeat(np.arange(W.shape[0]), np.diff(W.indptr))[keep]
    post = W.indices[keep]
    weight = (W.data[keep] * signs[pre]).astype(np.float32)
    row_norm = np.bincount(post, weights=np.abs(weight), minlength=W.shape[0])
    row_norm = np.maximum(row_norm, 1.0)
    weight = (weight / row_norm[post]).astype(np.float32)
    indices = torch.tensor(np.stack([post, pre]), dtype=torch.long)
    values = torch.tensor(weight)
    return torch.sparse_coo_tensor(indices, values, (W.shape[0], W.shape[0])).coalesce(), int(keep.sum())


Wt, n_edges_kept = build_signed_transpose(W, base_signs, EDGE_MIN_SYNAPSES)
Wt = Wt.to(DEVICE)
print(f"kept {n_edges_kept:,} / {W.nnz:,} edges at ≥{EDGE_MIN_SYNAPSES} synapses "
      f"({n_edges_kept / W.nnz:.0%}), {W.data[W.data >= EDGE_MIN_SYNAPSES].sum() / W.data.sum():.0%} of synapse mass")


class FlyBrain(nn.Module):
    """Frozen, batched, differentiable graded-rate model of the MaleCNS connectome."""

    def __init__(self, Wt, gain, alpha):
        super().__init__()
        self.register_buffer("Wt", Wt, persistent=False)
        self.n = Wt.shape[0]
        self.gain = gain
        self.alpha = alpha

    def step(self, r, current):
        total_input = torch.sparse.mm(self.Wt, r)
        target = torch.clamp(self.gain * total_input + current, 0.0, 1.0)
        return (1 - self.alpha) * r + self.alpha * target

    def run(self, r0, current, steps):
        r = r0
        for _ in range(steps):
            r = self.step(r, current)
        return r

    def run_clamped(self, r0, current, active_mask, steps):
        """Only `active_mask` neurons update; everyone else is held at r0 (§ act-phase)."""
        r = r0
        frozen = r0
        mask = active_mask.view(-1, 1)
        for _ in range(steps):
            r = torch.where(mask, self.step(r, current), frozen)
        return r

    def active_blocks(self, active_idx):
        active_idx = np.asarray(active_idx)
        coordinate = self.Wt.coalesce()
        ij = coordinate.indices().cpu().numpy()
        vals = coordinate.values().cpu().numpy()
        lookup = np.full(self.n, -1, dtype=np.int64)
        lookup[active_idx] = np.arange(len(active_idx))
        rows, cols = lookup[ij[0]], lookup[ij[1]]
        internal = (rows >= 0) & (cols >= 0)
        external = (rows >= 0) & (cols < 0)
        def matrix(mask, col, width):
            return torch.sparse_coo_tensor(torch.tensor(np.stack([rows[mask], col[mask]])),
                   torch.tensor(vals[mask]), (len(active_idx), width)).coalesce().to(self.Wt.device)
        return matrix(internal, cols, len(active_idx)), matrix(external, ij[1], self.n)

    @torch.amp.custom_fwd(device_type="cuda", cast_inputs=torch.float32)
    def run_active_prepared(self, initial, background, current_active, internal, steps):
        r = initial
        for _ in range(steps):
            target = torch.clamp(self.gain * (torch.sparse.mm(internal, r) + background)
                                 + current_active, 0, 1)
            r = (1 - self.alpha) * r + self.alpha * target
        return r

    def run_active(self, r0, current_active, active_idx, blocks, steps):
        internal, external = blocks
        r = r0[active_idx]
        background = torch.sparse.mm(external, r0)
        for _ in range(steps):
            target = torch.clamp(self.gain * (torch.sparse.mm(internal, r) + background)
                                 + current_active, 0, 1)
            r = (1 - self.alpha) * r + self.alpha * target
        return r

In [ ]:
def build_injection_map(sensory_idx, n_features, seed=0):
    """A fixed, disjoint feature -> sensory-neuron-group assignment (~14 neurons/feature)."""
    rng = np.random.RandomState(seed)
    order = rng.permutation(sensory_idx)
    groups = [order[i % len(order):i % len(order) + 1] for i in range(n_features)] if len(order) < n_features else np.array_split(order, n_features)
    rows = np.concatenate(groups)
    cols = np.concatenate([np.full(len(g), i) for i, g in enumerate(groups)])
    indices = torch.tensor(np.stack([rows, cols]), dtype=torch.long)
    values = torch.ones(len(rows))
    return torch.sparse_coo_tensor(indices, values, (N_NEURONS, n_features)).coalesce().to(DEVICE)


INJECT_MAP = build_injection_map(SENSORY_IDX, N_FEATURES)
INJECT_AMPLITUDE = 1.0 if USE_REAL_GRAPH else 0.3


def sensory_current(features_b_f):
    """(B, 780) board features -> (n, B) injected current."""
    return INJECT_AMPLITUDE * torch.sparse.mm(INJECT_MAP, features_b_f.t())


def empty_cache():
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()


def perceive_chunked(brain_, features_np, T_p, batch=128, keep_idx=None):
    """Runs perception in GPU-sized chunks and returns a CPU array — the (n, N) state for N boards
    at once does not fit in 6 GB much past a few hundred boards. `keep_idx` restricts the returned
    rows (e.g. to a relay candidate) to save host memory too."""
    chunks = []
    for start in range(0, len(features_np), batch):
        feats = torch.tensor(features_np[start:start + batch], device=DEVICE)
        with torch.no_grad():
            r = brain_.run(torch.zeros(brain_.n, feats.shape[0], device=DEVICE), sensory_current(feats), T_p)
        chunks.append((r if keep_idx is None else r[keep_idx]).cpu().numpy())
        del feats, r
    empty_cache()
    return np.concatenate(chunks, axis=1)


T_P_GRID = ([15, 25, 40] if MODE == "full" else [10, 15]) if USE_REAL_GRAPH else [3, 5]

In [ ]:
GAIN, ALPHA = 1.5, .6
T_P = 15 if USE_REAL_GRAPH else 5
brain = FlyBrain(Wt.to(DEVICE), GAIN, ALPHA)
print("frozen perception", T_P, "gain", GAIN, "alpha", ALPHA)

In [ ]:
RELAY_IDX = SENSORY_IDX.copy()
RELAY_STEPS = min(3, T_P)
print("relay",len(RELAY_IDX),"early steps",RELAY_STEPS)

In [ ]:
P_SIZE = 64 if MODE == "smoke" else 4000

motor_indicator = np.zeros(N_NEURONS, dtype=np.float32)
motor_indicator[MOTOR_IDX] = 1.0
synapses_onto_motor = W @ motor_indicator  # row i = total synapses from neuron i onto M
central = ~ann["superclass"].isin(["cb_sensory", "ol_sensory", "descending_neuron", "cb_motor", "vnc_motor"]).values
choices = np.flatnonzero(central & (synapses_onto_motor > 0))
PREMOTOR_IDX = choices[np.lexsort((ids[choices], -synapses_onto_motor[choices]))[:P_SIZE]]
PREMOTOR_IDX = PREMOTOR_IDX[synapses_onto_motor[PREMOTOR_IDX] > 0]
print(f"premotor P: {len(PREMOTOR_IDX):,} central neurons, "
      f"{synapses_onto_motor[PREMOTOR_IDX].min():.0f}-{synapses_onto_motor[PREMOTOR_IDX].max():.0f} synapses onto M each")

ACTIVE_MASK = torch.zeros(N_NEURONS, dtype=torch.bool, device=DEVICE)
ACTIVE_MASK[PREMOTOR_IDX] = True
ACTIVE_MASK[MOTOR_IDX] = True
T_A = 3

ACTIVE_IDX = np.flatnonzero(ACTIVE_MASK.cpu().numpy())
MOTOR_ACTIVE_IDX = np.searchsorted(ACTIVE_IDX, MOTOR_IDX)
PREMOTOR_ACTIVE_IDX = np.searchsorted(ACTIVE_IDX, PREMOTOR_IDX)

In [ ]:
def make_relay_pool(relay_idx, injection_map):
    mapping = injection_map.coalesce().indices().cpu().numpy()
    lookup = np.full(injection_map.shape[0], -1, dtype=np.int64)
    lookup[np.asarray(relay_idx)] = np.arange(len(relay_idx))
    selected = lookup[mapping[0]] >= 0
    rows, cols = mapping[1, selected], lookup[mapping[0, selected]]
    counts = np.bincount(rows, minlength=injection_map.shape[1])
    if (counts > 0).mean() < .95:
        return None
    values = (1/np.maximum(counts[rows],1)).astype(np.float32)
    return torch.sparse_coo_tensor(torch.tensor(np.stack([rows,cols])), torch.tensor(values),
        (injection_map.shape[1],len(relay_idx))).coalesce().to(injection_map.device)


class CortexGraft(nn.Module):
    def __init__(self, relay_idx, premotor_idx, d_model=64, n_latents=48, n_layers=2, n_heads=4, relay_pool=None):
        super().__init__()
        self.register_buffer("relay_idx", torch.tensor(relay_idx, dtype=torch.long), persistent=False)
        self.register_buffer("premotor_idx", torch.tensor(premotor_idx, dtype=torch.long), persistent=False)
        self.register_buffer("relay_pool", torch.empty(0) if relay_pool is None else relay_pool)
        n_relay = len(relay_idx) if relay_pool is None else relay_pool.shape[0]
        n_premotor = len(premotor_idx)

        self.register_buffer("relay_mean", torch.zeros(n_relay))
        self.register_buffer("relay_scale", torch.ones(n_relay))
        self.square_tokens = relay_pool is not None and n_relay >= 780
        if self.square_tokens:
            self.piece_embed = nn.Linear(12, d_model)
            self.side_embed = nn.Linear(n_relay - 768, d_model)
        else:
            self.rate_embed = nn.Linear(1, d_model)
        self.token_norm = nn.LayerNorm(d_model)
        self.relay_summary = nn.Sequential(nn.Linear(n_relay, 4*d_model), nn.LayerNorm(4*d_model), nn.GELU(), nn.Dropout(0.2))
        self.latent_from_summary = nn.Linear(4*d_model, n_latents*d_model)
        self.premotor_from_summary = nn.Linear(4*d_model, n_premotor)
        self.n_latents, self.d_model = n_latents, d_model
        nn.init.normal_(self.premotor_from_summary.weight, std=0.001)
        nn.init.zeros_(self.premotor_from_summary.bias)
        self.relay_embed = nn.Embedding(64 if self.square_tokens else n_relay, d_model)
        self.premotor_query = nn.Embedding(n_premotor, d_model)
        self.latents = nn.Parameter(torch.randn(n_latents, d_model) * 0.02)

        self.encode_in = nn.MultiheadAttention(d_model, n_heads, batch_first=True, dropout=0.1)
        self.self_layers = nn.ModuleList(
            nn.MultiheadAttention(d_model, n_heads, batch_first=True, dropout=0.1) for _ in range(n_layers)
        )
        self.norms = nn.ModuleList(nn.LayerNorm(d_model) for _ in range(n_layers + 1))
        self.decode_out = nn.MultiheadAttention(d_model, n_heads, batch_first=True, dropout=0.1)
        self.output_norm = nn.LayerNorm(d_model)
        self.ff_layers = nn.ModuleList(nn.Sequential(nn.Linear(d_model, 2*d_model), nn.GELU(),
            nn.Linear(2*d_model, d_model), nn.Dropout(0.1)) for _ in range(n_layers))
        self.ff_norms = nn.ModuleList(nn.LayerNorm(d_model) for _ in range(n_layers))
        self.current_head = nn.Linear(d_model, 1)
        nn.init.normal_(self.current_head.weight, std=0.01)
        nn.init.constant_(self.current_head.bias, 0.1)

    @torch.amp.custom_fwd(device_type="cuda", cast_inputs=torch.float32)
    def pool_relay(self, rates):
        return torch.sparse.mm(self.relay_pool, rates.t()).t() if self.relay_pool.numel() else rates

    def forward(self, relay_rate):
        """relay_rate: (B, n_relay) -> premotor current (B, n_premotor)."""
        relay_rate = self.pool_relay(relay_rate)
        B = relay_rate.shape[0]
        rates = ((relay_rate - self.relay_mean) / self.relay_scale).clamp(-10, 10)
        if self.square_tokens:
            # These channels are pooled fly rates from the fixed sensory input groups.
            squares = rates[:, :768].reshape(B, 12, 64).transpose(1, 2)
            tokens = self.token_norm(self.relay_embed.weight.unsqueeze(0) + self.piece_embed(squares))
        else:
            tokens = self.token_norm(self.relay_embed.weight.unsqueeze(0) + self.rate_embed(rates.unsqueeze(-1)))
        summary = self.relay_summary(rates)
        latents = self.latents.unsqueeze(0) + self.latent_from_summary(summary).view(B, self.n_latents, self.d_model)
        if self.square_tokens:
            latents = latents + self.side_embed(rates[:, 768:]).unsqueeze(1)
        latents = self.norms[0](latents + self.encode_in(latents, tokens, tokens, need_weights=False)[0])
        for layer, norm, ff, ff_norm in zip(self.self_layers, self.norms[1:], self.ff_layers, self.ff_norms):
            latents = norm(latents + layer(latents, latents, latents, need_weights=False)[0])
            latents = ff_norm(latents + ff(latents))
        queries = self.premotor_query.weight.unsqueeze(0).expand(B, -1, -1)
        out, _ = self.decode_out(queries, latents, latents, need_weights=False)
        out = self.output_norm(queries + out)
        return torch.tanh(self.current_head(out).squeeze(-1) + self.premotor_from_summary(summary))


class MoveDecoder(nn.Module):
    """Reads the fly's own DN+motor population; never sees the board or the relay directly."""

    def __init__(self, n_motor, n_moves=N_MOVES, hidden=256):
        super().__init__()
        self.dropout = nn.Dropout(0.1)
        self.register_buffer("motor_mean", torch.zeros(n_motor))
        self.register_buffer("motor_scale", torch.ones(n_motor))
        self.policy = nn.Linear(n_motor, n_moves)
        self.value = nn.Linear(n_motor, 1)

    def forward(self, motor_rate):
        normalized = self.dropout((motor_rate - self.motor_mean) / self.motor_scale)
        return self.policy(normalized), self.value(normalized).squeeze(-1)


class FlyChessModel(nn.Module):
    """board -> (fly sense) -> graft -> (fly act) -> policy logits, value. `brain` stays frozen."""

    def __init__(self, brain, relay_idx, premotor_idx, motor_idx, active_mask, T_p, T_a,
                 current_amplitude=0.5, relay_steps=3, **graft_kw):
        super().__init__()
        self.brain = brain
        pooling = make_relay_pool(relay_idx, INJECT_MAP) if "INJECT_MAP" in globals() else None
        self.graft = CortexGraft(relay_idx, premotor_idx, relay_pool=pooling, **graft_kw)
        self.decoder = MoveDecoder(len(motor_idx))
        self.register_buffer("relay_idx", torch.tensor(relay_idx, dtype=torch.long), persistent=False)
        self.register_buffer("premotor_idx", torch.tensor(premotor_idx, dtype=torch.long), persistent=False)
        self.register_buffer("motor_idx", torch.tensor(motor_idx, dtype=torch.long), persistent=False)
        self.register_buffer("active_mask", active_mask, persistent=False)
        self.T_p, self.T_a, self.current_amplitude = T_p, T_a, current_amplitude
        self.relay_steps = min(T_p, relay_steps)
        active_idx = np.flatnonzero(active_mask.cpu().numpy())
        self.register_buffer("active_idx", torch.tensor(active_idx), persistent=False)
        self.motor_active_idx = np.searchsorted(active_idx, motor_idx)
        self.premotor_active_idx = np.searchsorted(active_idx, premotor_idx)
        self.blocks = brain.active_blocks(active_idx)


    @torch.amp.custom_fwd(device_type="cuda", cast_inputs=torch.float32)
    def perceive(self, features):
        with torch.no_grad():
            current = sensory_current(features)
            state = self.brain.run(torch.zeros(self.brain.n, len(features), device=features.device),
                                   current, self.relay_steps)
            relay = state[self.relay_idx].t()
            state = self.brain.run(state, current, self.T_p - self.relay_steps)
            return (relay, state[self.active_idx].t(),
                    torch.sparse.mm(self.blocks[1], state).t())

    def forward_perceived(self, relay, initial, background, lesion_relay=False, intervention=None):
        if intervention == "no_senses":
            relay, initial, background = torch.zeros_like(relay), torch.zeros_like(initial), torch.zeros_like(background)
        if lesion_relay:
            relay = torch.zeros_like(relay)
        if intervention == "relay_permute":
            # Permute neuron identities; this also works on singleton inference batches.
            relay = relay.roll(1, dims=1)
        premotor = self.graft(relay) * self.current_amplitude
        if intervention == "no_graft":
            premotor = torch.zeros_like(premotor)
        current = torch.zeros(len(self.active_idx), len(relay), device=relay.device)
        current[self.premotor_active_idx] = premotor.float().t()
        final = self.brain.run_active_prepared(initial.t(), background.t(), current,
                                             self.blocks[0], self.T_a)
        return self.decoder(final[self.motor_active_idx].t())

    def forward(self, features, lesion_relay=False, intervention=None):
        return self.forward_perceived(*self.perceive(features), lesion_relay=lesion_relay, intervention=intervention)

In [ ]:
class FlyOnlyModel(nn.Module):
    """Frozen perception -> normalized linear motor readout, without a graft."""
    def __init__(self, brain, motor_idx, T_p):
        super().__init__()
        self.brain, self.T_p = brain, T_p
        self.decoder = MoveDecoder(len(motor_idx))
        self.register_buffer("motor_idx", torch.tensor(motor_idx, dtype=torch.long), persistent=False)

    @torch.amp.custom_fwd(device_type="cuda", cast_inputs=torch.float32)
    def perceive(self, features):
        with torch.no_grad():
            state = self.brain.run(torch.zeros(self.brain.n, len(features), device=features.device),
                                   sensory_current(features), self.T_p)
            return (state[self.motor_idx].t(),)

    def forward_perceived(self, motor):
        return self.decoder(motor)

    def forward(self, features, **kwargs):
        return self.forward_perceived(*self.perceive(features))

In [ ]:
def make_batch(indices):
    boards = [chess.Board(str(fens[i])) for i in indices]
    features = torch.tensor(np.stack([encode_board(b) for b in boards]), device=DEVICE)
    labels = torch.tensor([move_index(chess.Move.from_uci(str(labels_uci[i])), b.turn)
                           for i, b in zip(indices, boards)], device=DEVICE)
    targets = torch.tensor(values[indices], device=DEVICE)
    return boards, features, labels, targets


@contextlib.contextmanager
def inference_mode(model):
    was_training = model.training
    model.eval()
    try:
        with torch.no_grad():
            yield
    finally:
        model.train(was_training)


def move_match(model, indices, intervention=None, value_weight=4.0):
    correct, raw_correct, squared_error, policy_total, value_total = 0, 0, 0.0, 0.0, 0.0
    with inference_mode(model):
        for offset in range(0, len(indices), BATCH_SIZE):
            boards, features, labels, targets = make_batch(indices[offset:offset + BATCH_SIZE])
            logits, value = model(features, intervention=intervention) if intervention else model(features)
            raw_correct += int((logits.argmax(1) == labels).sum())
            correct += int((mask_logits(logits, boards).argmax(1) == labels).sum())
            squared_error += float(((value.sigmoid() - targets)**2).sum())
            policy_total += float(F.cross_entropy(mask_logits(logits, boards), labels, reduction="sum"))
            value_total += float(F.binary_cross_entropy_with_logits(value, targets, reduction="sum"))
    constant = float(np.mean(values[train_idx]))
    baseline = float(np.mean((values[indices] - constant)**2))
    mse = squared_error / len(indices)
    return {"legal_move_match": correct / len(indices), "unmasked_diagnostic": raw_correct / len(indices),
            "value_mse": mse, "constant_value_mse": baseline,
            "value_skill": 1 - mse / baseline if baseline > 1e-10 else None,
            "validation_loss": (policy_total + value_weight*value_total)/len(indices), "n": len(indices)}


def cache_perception(candidate, tag, pool, identity):
    # Full-precision CPU memmaps: exactly the same frozen states as live inference.
    directory = ROOT / "perception-cache" / RUN_ID / tag
    directory.mkdir(parents=True, exist_ok=True)
    progress_path = directory / "progress.json"
    progress = json.loads(progress_path.read_text()) if progress_path.exists() else None
    if progress and progress["identity"] != identity:
        raise ValueError("Perception cache identity changed")
    first = candidate.perceive(make_batch(pool[:1])[1])
    widths = [x.shape[1] for x in first]
    files = [directory / f"component-{i}.npy" for i in range(len(widths))]
    if progress and all(path.exists() for path in files):
        arrays = [np.lib.format.open_memmap(path, mode="r+") for path in files]
        completed = progress["completed"]
    else:
        arrays = [np.lib.format.open_memmap(path, mode="w+", dtype=np.float32, shape=(len(pool), width))
                  for path, width in zip(files, widths)]
        completed = 0
    for start in range(completed, len(pool), BATCH_SIZE):
        check_stop()
        batch = pool[start:start + BATCH_SIZE]
        with torch.no_grad():
            components = candidate.perceive(make_batch(batch)[1])
        for array, component in zip(arrays, components):
            array[start:start + len(batch)] = component.cpu().numpy()
            array.flush()
        atomic_json(progress_path, {"identity": identity, "completed": start + len(batch), "widths": widths})
        if start % (BATCH_SIZE * 20) == 0:
            print("perception cache", tag, start + len(batch), "/", len(pool), flush=True)
    return {"pool": pool, "lookup": {int(i): j for j, i in enumerate(pool)}, "arrays": arrays}


def cached_forward(candidate, cache, indices):
    rows = [cache["lookup"][int(i)] for i in indices]
    components = []
    for i, array in enumerate(cache["arrays"]):
        selected = np.asarray(array[rows])
        if cache.get("select_columns"):
            selected = selected[:, cache["select_columns"][i]]
        components.append(torch.tensor(selected, device=DEVICE))
    return candidate.forward_perceived(*components)


def initialize_signal_stats(candidate, indices):
    with torch.no_grad():
        features = make_batch(indices)[1]
        components = candidate.perceive(features) if hasattr(candidate, "perceive") else None
        if isinstance(candidate, FlyChessModel):
            relay, initial, background = components
            relay = candidate.graft.pool_relay(relay)
            candidate.graft.relay_mean.copy_(relay.mean(0))
            floor = 0.1 if candidate.graft.relay_pool.numel() else 0.0001
            candidate.graft.relay_scale.copy_(relay.std(0, unbiased=False).clamp_min(floor))
            # Motor normalization is a fixed affine transform of the zero-graft act state.
            zero = torch.zeros_like(initial.t())
            motor = candidate.brain.run_active_prepared(initial.t(), background.t(), zero,
                candidate.blocks[0], candidate.T_a)[candidate.motor_active_idx].t()
        elif isinstance(candidate, FlyOnlyModel):
            motor = components[0]
        else:
            relay, motor = candidate.interface.perceive(features)
            relay = candidate.graft.pool_relay(relay)
            candidate.graft.relay_mean.copy_(relay.mean(0))
            floor = 0.1 if candidate.graft.relay_pool.numel() else 0.0001
            candidate.graft.relay_scale.copy_(relay.std(0, unbiased=False).clamp_min(floor))
        candidate.decoder.motor_mean.copy_(motor.mean(0))
        candidate.decoder.motor_scale.copy_(motor.std(0, unbiased=False).clamp_min(0.03))


def training_update(model, optimizer, sampler, cache=None, indices=None, value_weight=4.0):
    source = cache["pool"] if cache else train_idx
    if indices is None:
        indices = sampler.choice(source, min(BATCH_SIZE, len(source)), replace=False)
    boards, features, labels, targets = make_batch(indices)
    model.train()
    logits, value = cached_forward(model, cache, indices) if cache else model(features)
    policy_loss = F.cross_entropy(mask_logits(logits, boards), labels)
    value_loss = F.binary_cross_entropy_with_logits(value, targets)
    loss = policy_loss + value_weight * value_loss
    if not torch.isfinite(loss):
        raise RuntimeError("Nonfinite training loss; checkpoint retained")
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0, error_if_nonfinite=True)
    optimizer.step()
    return {"loss": float(loss.detach()), "policy_loss": float(policy_loss.detach()),
            "value_loss": float(value_loss.detach()), "grad_norm": float(grad_norm)}


def learning_preflight(candidate, cache):
    indices = cache["pool"][:min(32, len(cache["pool"]))]
    boards, _, labels, targets = make_batch(indices)
    original = {key: value.detach().clone() for key, value in candidate.state_dict().items()}
    sampler = np.random.RandomState(SEED)
    saved_rng = rng_state(sampler)
    def metrics():
        with inference_mode(candidate):
            logits, value = cached_forward(candidate, cache, indices)
            return {"accuracy": float((mask_logits(logits, boards).argmax(1) == labels).float().mean()),
                    "value_mse": float(((value.sigmoid() - targets)**2).mean())}
    try:
        before = metrics()
        optimizer = torch.optim.AdamW(candidate.parameters(), lr=1e-3)
        for _ in range(160 if MODE == "full" else 40):
            check_stop()
            training_update(candidate, optimizer, sampler, cache, indices)
        after = metrics()
        passed = after["accuracy"] >= max(0.5, before["accuracy"] + 0.15) and after["value_mse"] < before["value_mse"] * 0.8
        result = {"before": before, "after": after, "passed": passed, "n": len(indices)}
        atomic_json(RUN_DIR / "learning-preflight.json", result)
        print("learning preflight", result, flush=True)
        if MODE == "full" and not passed:
            raise RuntimeError("Learning preflight failed; long training skipped. Inspect learning-preflight.json")
        return result
    finally:
        candidate.load_state_dict(original)
        restore_rng(saved_rng, sampler)

In [ ]:
class Node:
    def __init__(self, board, parent=None, prior=1.0):
        self.board, self.parent, self.prior = board, parent, prior
        self.children = {}
        self.visits = 0
        self.pending = 0
        self.value_sum = 0.0
        self.expanded = False
        self.neural_evaluations = 0

    def q(self):
        return self.value_sum / self.visits if self.visits else 0.5


def puct_score(child, parent_visits, c_puct=1.5):
    return 1 - child.q() + c_puct * child.prior * math.sqrt(max(1, parent_visits)) / (1 + child.visits + child.pending)


def evaluate_leaves(model, boards):
    with inference_mode(model):
        features = torch.tensor(np.stack([encode_board(b) for b in boards]), device=DEVICE)
        logits, values = model(features)
        priors = []
        for board, row in zip(boards, logits):
            indices = list(legal_move_table(board))
            probabilities = F.softmax(row[indices], dim=0).cpu().numpy()
            priors.append(dict(zip(indices, probabilities)))
        return priors, torch.sigmoid(values).cpu().numpy()


def expand(node, priors):
    if node.expanded:
        return
    for index, probability in priors.items():
        board = node.board.copy(stack=True)
        board.push(index_to_move(index, board))
        node.children[index] = Node(board, node, float(probability))
    node.expanded = True


def backup(path, value):
    for node in reversed(path):
        node.visits += 1
        node.value_sum += value
        value = 1 - value


def reserve_leaf(root):
    def descend(node, path):
        if terminal_value(node.board) is not None:
            return node, path
        if not node.expanded:
            return (node, path) if node.pending == 0 else None
        ordered = sorted(node.children.items(), key=lambda item: (-puct_score(item[1], node.visits + node.pending), item[0]))
        for _, child in ordered:
            found = descend(child, path + [child])
            if found is not None:
                return found
        return None
    found = descend(root, [root])
    if found:
        for node in found[1]: node.pending += 1
    return found


def search(model, root_board, n_simulations=64, batch=16, deadline=None, root_priors=None):
    root = Node(root_board.copy(stack=True))
    if terminal_value(root.board) is not None:
        return root
    if root_priors is None:
        priors, _ = evaluate_leaves(model, [root.board])
        root_priors = priors[0]
    expand(root, root_priors)
    root.neural_evaluations = 1
    completed = 0
    while completed < n_simulations and (deadline is None or time.monotonic() < deadline):
        check_stop()
        selected = []
        for _ in range(min(batch, n_simulations - completed)):
            found = reserve_leaf(root)
            if found is None: break
            selected.append(found)
        if not selected: break
        try:
            neural = [(node, path) for node, path in selected if terminal_value(node.board) is None]
            predictions = {}
            if neural:
                priors, values = evaluate_leaves(model, [node.board for node, _ in neural])
                root.neural_evaluations += len(neural)
                predictions = {id(node): (prior, float(value)) for (node, _), prior, value in zip(neural, priors, values)}
            for node, path in selected:
                value = terminal_value(node.board)
                if value is None:
                    prior, value = predictions[id(node)]
                    expand(node, prior)
                backup(path, value)
                completed += 1
        finally:
            for _, path in selected:
                for node in path: node.pending -= 1
    assert root.visits == completed
    return root


def best_move_by_search(model, board, n_simulations=None, deadline=None, root_priors=None):
    root = search(model, board, n_simulations or (8 if MODE == "smoke" else 64), deadline=deadline, root_priors=root_priors)
    if not root.children: return None, root
    index = max(root.children, key=lambda i: (root.children[i].visits, root.children[i].prior, -i))
    return index_to_move(index, board), root


def predict(model, boards):
    features = torch.tensor(np.stack([encode_board(b) for b in boards]), device=DEVICE)
    with inference_mode(model):
        logits, values = model(features)
        return logits, torch.sigmoid(values)


def mate_in_one(board):
    for move in list(board.legal_moves):
        board.push(move)
        mate = board.is_checkmate()
        board.pop()
        if mate: return move
    return None


def choose_move(model, board, use_search=False, assist_mode="none", return_metadata=False, deadline=None):
    started = time.monotonic()
    metadata = {"assist_mode": assist_mode, "assisted": False, "neural_evaluations": 0, "simulations": 0}
    if terminal_value(board) is not None:
        return (None, metadata) if return_metadata else None
    logits, value = predict(model, [board])
    legal_indices = list(legal_move_table(board))
    row = logits[0].detach().cpu().numpy()
    ranked = sorted(legal_indices, key=lambda i: (-float(row[i]), i))
    if use_search:
        probabilities = F.softmax(logits[0, legal_indices], dim=0).cpu().numpy()
        move, tree = best_move_by_search(model, board, deadline=deadline, root_priors=dict(zip(legal_indices, probabilities)))
        metadata.update(neural_evaluations=tree.neural_evaluations, simulations=tree.visits)
    else:
        move = index_to_move(ranked[0], board)
        metadata["neural_evaluations"] = 1
    metadata["model_move"] = move.uci()
    assert assist_mode in ("none", "mate1")
    if assist_mode == "mate1":
        mate = mate_in_one(board)
        if mate is not None:
            move = mate
        else:
            def safe(candidate):
                board.push(candidate)
                bad = mate_in_one(board) is not None if terminal_value(board) is None else False
                board.pop()
                return not bad
            if not safe(move):
                move = next((index_to_move(i, board) for i in ranked if safe(index_to_move(i, board))), move)
    metadata.update(final_move=move.uci(), assisted=move.uci() != metadata["model_move"],
                    elapsed_s=time.monotonic() - started, value=float(value[0]))
    assert move in board.legal_moves
    return (move, metadata) if return_metadata else move

In [ ]:
STOCKFISH_DEPTH = 3 if MODE == "smoke" else 12
SCORE_CACHE_PATH = RUN_DIR / "engine-scores.json"


def open_stockfish():
    engine = chess.engine.SimpleEngine.popen_uci(str(STOCKFISH_BIN))
    engine.configure({"Threads": 1, "Hash": 64, "UCI_LimitStrength": False, "Skill Level": 20})
    return engine


def score_move(engine, board, move):
    key = "|".join((board.fen(en_passant="fen"), move.uci(), STOCKFISH_BINARY_SHA, str(STOCKFISH_DEPTH)))
    if key in SCORE_CACHE:
        return SCORE_CACHE[key]
    engine.configure({"Clear Hash": None})
    info = engine.analyse(board, chess.engine.Limit(depth=STOCKFISH_DEPTH), root_moves=[move])
    score = info["score"].pov(board.turn)
    result = {"cp": score.score(), "mate": score.mate(),
              "expectation": score.wdl(model="sf", ply=board.ply()).expectation()}
    SCORE_CACHE[key] = result
    atomic_json(SCORE_CACHE_PATH, SCORE_CACHE)
    return result


def score_position(engine, board):
    return {move.uci(): score_move(engine, board, move) for move in board.legal_moves}


def position_loss(scores, move):
    best = max(scores.values(), key=lambda row: row["expectation"])
    chosen = scores[move.uci()]
    expected_loss = max(0, best["expectation"] - chosen["expectation"])
    ordinary = [row["cp"] for row in scores.values() if row["cp"] is not None]
    cp_loss = max(0, max(ordinary) - chosen["cp"]) if chosen["cp"] is not None and all(row["mate"] is None for row in scores.values()) else None
    return {"expected_score_loss": expected_loss, "cp_loss": cp_loss,
            "mate_outcome": chosen["mate"], "best_mate_outcome": best["mate"]}


def bootstrap_mean(values, seed=21):
    values = np.asarray(values, dtype=float)
    if len(values) == 0: return {"mean": None, "ci95": None, "n": 0}
    sampler = np.random.RandomState(seed)
    means = [values[sampler.randint(len(values), size=len(values))].mean() for _ in range(2000)]
    return {"mean": float(values.mean()), "ci95": np.percentile(means, [2.5, 97.5]).tolist(), "n": len(values)}


def uniform_legal_match(indices):
    return float(np.mean([1 / chess.Board(str(fens[i])).legal_moves.count() for i in indices]))

SCORE_CACHE = json.loads(SCORE_CACHE_PATH.read_text()) if SCORE_CACHE_PATH.exists() else {}


def ratio_summary(chosen, random_losses):
    chosen, random_losses = np.asarray(chosen), np.asarray(random_losses)
    if len(chosen) == 0 or random_losses.mean() <= 1e-8:
        return {"value": None, "ci95": None, "n": len(chosen)}
    sampler = np.random.RandomState(24)
    ratios = []
    for _ in range(2000):
        indices = sampler.randint(len(chosen), size=len(chosen))
        denominator = random_losses[indices].mean()
        if denominator > 1e-8:
            ratios.append(1 - chosen[indices].mean() / denominator)
    return {"value": float(1 - chosen.mean() / random_losses.mean()),
            "ci95": np.percentile(ratios, [2.5, 97.5]).tolist() if ratios else None, "n": len(chosen)}

In [ ]:
def puzzle_trial(model, row, use_search=False, assist_mode="none", full_line=False):
    board = chess.Board(row["FEN"])
    line = row["Moves"].split()
    board.push_uci(line[0])
    for ply in range(1, len(line), 2):
        predicted = choose_move(model, board, use_search, assist_mode)
        if predicted is None: return False
        board.push(predicted)
        if board.is_checkmate(): return True
        if predicted.uci() != line[ply]: return False
        if not full_line: return True
        if ply + 1 < len(line): board.push_uci(line[ply + 1])
    return True


def pair_score_summary(records):
    completed = [r for r in records if r["score"] is not None]
    pair_ids = sorted({r["pair"] for r in completed})
    pair_scores = [np.mean([r["score"] for r in completed if r["pair"] == pair]) for pair in pair_ids
                   if sum(r["pair"] == pair for r in completed) == 2]
    return {"wins": sum(r["score"] == 1 for r in completed), "draws": sum(r["score"] == 0.5 for r in completed),
            "losses": sum(r["score"] == 0 for r in completed), "unresolved": sum(r["score"] is None for r in records),
            "score": float(np.mean([r["score"] for r in completed])) if completed else None,
            "paired_score": bootstrap_mean(pair_scores), "games": len(records)}


def play_game(white, black, opening, game_id, initial_clock=60.0, increment=0.6, max_plies=600):
    checkpoint = RUN_DIR / "games" / f"{game_id}.json"
    checkpoint.parent.mkdir(exist_ok=True)
    board = chess.Board()
    clocks = {"white": initial_clock, "black": initial_clock}
    history = []
    if checkpoint.exists():
        state = json.loads(checkpoint.read_text())
        clocks, history = state["clocks"], state["history"]
        for move in state["moves"]: board.push_uci(move)
        if state["status"] == "complete": return state
    else:
        for move in opening: board.push_uci(move)
    score, reason = None, "safety_cap"
    while len(history) < max_plies:
        check_stop()
        outcome = board.outcome(claim_draw=True)
        if outcome:
            score = 0.5 if outcome.winner is None else float(outcome.winner == chess.WHITE)
            reason = outcome.termination.name
            break
        side = "white" if board.turn else "black"
        started = time.monotonic()
        move = (white if board.turn else black)(board, clocks[side], increment)
        elapsed = time.monotonic() - started
        clocks[side] -= elapsed
        if clocks[side] < 0:
            score = float(board.turn != chess.WHITE)
            reason = "timeout"
            break
        if move not in board.legal_moves:
            raise RuntimeError("Opponent returned an illegal move")
        board.push(move)
        clocks[side] += increment
        history.append({"move": move.uci(), "side": side, "elapsed_s": elapsed})
        atomic_json(checkpoint, {"status": "in_progress", "moves": [m.uci() for m in board.move_stack],
                    "clocks": clocks, "history": history})
    if score is None:
        outcome = board.outcome(claim_draw=True)
        if outcome is not None:
            score = 0.5 if outcome.winner is None else float(outcome.winner == chess.WHITE)
            reason = outcome.termination.name
    state = {"status": "complete", "score": score, "reason": reason,
             "moves": [m.uci() for m in board.move_stack], "clocks": clocks, "history": history}
    atomic_json(checkpoint, state)
    game = chess.pgn.Game.from_board(board)
    game.headers["Result"] = "*" if score is None else "1-0" if score == 1 else "0-1" if score == 0 else "1/2-1/2"
    game.headers["Termination"] = reason
    game.headers["TimeControl"] = f"{initial_clock}+{increment}"
    checkpoint.with_suffix(".pgn").write_text(str(game) + "\n")
    return state


OPENINGS = [
    "e2e4 e7e5 g1f3 b8c6 f1b5 a7a6", "e2e4 c7c5 g1f3 d7d6 d2d4 c5d4",
    "d2d4 d7d5 c2c4 e7e6 b1c3 g8f6", "d2d4 g8f6 c2c4 g7g6 b1c3 f8g7",
    "c2c4 e7e5 b1c3 g8f6 g2g3 d7d5", "g1f3 d7d5 g2g3 g8f6 f1g2 e7e6",
    "e2e4 e7e6 d2d4 d7d5 b1c3 g8f6", "e2e4 c7c6 d2d4 d7d5 b1c3 d5e4",
    "d2d4 d7d5 c2c4 c7c6 g1f3 g8f6", "e2e4 e7e5 g1f3 g8f6 f3e5 d7d6"]
for sequence in OPENINGS:
    board = chess.Board()
    for move in sequence.split(): board.push_uci(move)


def model_opponent(candidate, use_search=False, assist_mode="none"):
    def mover(board, clock, increment):
        allocation = min(max(0.01, clock / 30 + increment * 0.5), max(0.01, clock * 0.25))
        deadline = time.monotonic() + allocation
        return choose_move(candidate, board, use_search, assist_mode, deadline=deadline)
    return mover


def greedy_material(board, clock=None, increment=None):
    weights = {chess.PAWN: 1, chess.KNIGHT: 3, chess.BISHOP: 3, chess.ROOK: 5, chess.QUEEN: 9, chess.KING: 0}
    def value(move):
        board.push(move)
        result = sum(weights[p.piece_type] * (1 if p.color != board.turn else -1) for p in board.piece_map().values())
        board.pop()
        return result
    return max(board.legal_moves, key=value)


def engine_opponent(engine, skill=0):
    def mover(board, clock, increment):
        engine.configure({"UCI_LimitStrength": False, "Skill Level": skill})
        return engine.play(board, chess.engine.Limit(time=min(max(0.01, clock / 30 + increment * 0.5), max(0.01, clock * 0.25)))).move
    return mover


"""Score-based strength estimates with explicit reference scales.

Opening pairs, rather than individual games, are the independent sampling unit.
The bounded-score Hoeffding interval stays nonzero after an all-loss/all-win run.
It quantifies match sampling only, not uncertainty in an opponent's calibration.
"""
import math


def score_to_elo(score):
    if score <= 0 or score >= 1:
        return None
    return 400 * math.log10(score / (1 - score))


def strength_summary(records, reference_rating=None, reference_scale=None, alpha=0.05):
    if not 0 < alpha < 1:
        raise ValueError("alpha must lie between zero and one")
    if reference_rating is not None and not reference_scale:
        raise ValueError("A reference rating requires an explicit scale/source")
    if reference_rating is not None and (not isinstance(reference_rating,(int,float)) or not math.isfinite(reference_rating)):
        raise ValueError("Reference rating must be finite")
    groups = {}
    for record in records:
        score = record.get("score")
        if score is not None and score not in (0, 0.5, 1):
            raise ValueError("Game score must be 0, 0.5, 1, or unresolved")
        groups.setdefault(record["pair"], []).append(record)
    pairs = []
    for group in groups.values():
        if len(group) == 2 and {r["color"] for r in group} == {"white", "black"}:
            if all(r["score"] is not None for r in group):
                pairs.append(sum(r["score"] for r in group) / 2)
    report = {"complete_pairs": len(pairs), "games": len(records),
              "wins": sum(r.get("score") == 1 for r in records),
              "draws": sum(r.get("score") == .5 for r in records),
              "losses": sum(r.get("score") == 0 for r in records),
              "unresolved": sum(r.get("score") is None for r in records),
              "reference_rating": reference_rating, "reference_scale": reference_scale,
              "interval_method": "95% bounded-score Hoeffding over opening pairs" if alpha == .05
                                 else f"{1-alpha:.1%} bounded-score Hoeffding over opening pairs",
              "sampling_assumption": "Independent, representative opening pairs",
              "score": None, "score_interval": None, "elo_difference": None,
              "elo_difference_interval": None, "reference_scale_estimate": None,
              "reference_scale_interval": None, "status": "insufficient_games"}
    if not pairs:
        return report
    score = sum(pairs) / len(pairs)
    radius = math.sqrt(math.log(2 / alpha) / (2 * len(pairs)))
    interval = [max(0, score - radius), min(1, score + radius)]
    delta = score_to_elo(score)
    bounds = [score_to_elo(p) for p in interval]
    report.update(score=score, score_interval=interval, elo_difference=delta,
                  elo_difference_interval=bounds,
                  status="estimate" if delta is not None else "upper_bound" if score == 0 else "lower_bound")
    if interval == [0,1]:
        report["status"] = "unbounded_interval"
    if reference_rating is not None:
        report["reference_scale_estimate"] = reference_rating + delta if delta is not None else None
        report["reference_scale_interval"] = [reference_rating + b if b is not None else None for b in bounds]
        report["calibration_note"] = "Conditional on the reference's calibration; not a FIDE/Chess.com/Lichess rating"
    report["unbounded_endpoints"] = {"lower": interval[0] == 0, "upper": interval[1] == 1}
    return report


def make_opening_pairs(count, seed=41):
    generator = np.random.RandomState(seed)
    openings, seen = [], set()
    while len(openings) < count:
        board = chess.Board()
        moves = OPENINGS[int(generator.randint(len(OPENINGS)))].split()
        for move in moves:
            board.push_uci(move)
        for _ in range(2):
            if terminal_value(board) is not None:
                break
            move = list(board.legal_moves)[int(generator.randint(board.legal_moves.count()))]
            moves.append(move.uci())
            board.push(move)
        key = " ".join(moves)
        if terminal_value(board) is None and key not in seen:
            seen.add(key)
            openings.append(moves)
    return openings


def evaluate_strength():
    path = RUN_DIR / "strength.json"
    pairs = 1 if MODE == "smoke" else int(os.environ.get("FLY_CHESS_MATCH_PAIRS", "20"))
    assert 1 <= pairs <= 1000
    openings = make_opening_pairs(pairs)
    report = json.loads(path.read_text()) if path.exists() else {"identity": IDENTITY, "matches": {}, "summaries": {}}
    assert report["identity"] == IDENTITY
    started = time.monotonic()
    budget = float(os.environ.get("FLY_CHESS_EVAL_MINUTES", "75"))*60 if MODE == "full" else 180
    clock, increment = (120, 1) if MODE == "full" else (2, .01)
    with open_stockfish() as engine:
        minimum, maximum = engine.options["UCI_Elo"].min, engine.options["UCI_Elo"].max
        ratings = sorted(set([minimum, min(maximum, minimum + 200)]))
        opponents = [("c0", model_opponent(model_c0), None), ("greedy", greedy_material, None)]
        for rating in ratings:
            def opponent(board, remaining, inc, rating=rating):
                engine.configure({"UCI_LimitStrength": True, "UCI_Elo": rating, "Skill Level": 20})
                return engine.play(board, chess.engine.Limit(white_clock=remaining if board.turn else clock,
                    black_clock=remaining if not board.turn else clock, white_inc=inc, black_inc=inc)).move
            opponents.append((f"stockfish-uci-{rating}", opponent, rating))
        configuration = {"pairs": pairs, "clock": clock, "increment": increment,
                         "ratings": ratings, "opening_seed": 41, "assist_mode": "none"}
        if report.get("configuration") and report["configuration"] != configuration:
            raise ValueError("Strength match configuration changed; use a new run")
        report["configuration"] = configuration
        selection_path = RUN_DIR / "model-selection.json"
        selection = json.loads(selection_path.read_text()) if selection_path.exists() else {}
        fingerprints = {}
        for tag in ("main","c0"):
            checkpoint_path = RUN_DIR / selection.get(tag,{}).get("checkpoint",f"{tag}.pt")
            fingerprints[tag] = sha256(checkpoint_path) if checkpoint_path.exists() else None
        if report.get("checkpoint_fingerprints") is not None and report["checkpoint_fingerprints"] != fingerprints:
            raise ValueError("Evaluated weights changed; use a separate run for a new benchmark")
        report["checkpoint_fingerprints"] = fingerprints
        report["model_selection"] = selection
        try:
            # Interleave opponents and modes so a pause leaves evidence for each condition.
            for pair, opening in enumerate(openings):
                for mode_name, use_search in (("policy", False), ("search", True)):
                    for name, opponent, rating in opponents:
                        key = mode_name + ":" + name
                        records = report["matches"].setdefault(key, [])
                        for color in ("white", "black"):
                            check_stop()
                            if time.monotonic() - started > budget:
                                report["status"] = "paused"
                                report["pause_reason"] = "Session evaluation allocation exhausted; rerun to continue"
                                return report
                            game_id = f"strength-{mode_name}-{name}-{pair}-{color}"
                            if any(r["id"] == game_id for r in records):
                                continue
                            player = model_opponent(model, use_search)
                            white, black = (player, opponent) if color == "white" else (opponent, player)
                            game = play_game(white, black, opening, game_id, initial_clock=clock,
                                             increment=increment, max_plies=600 if MODE == "full" else 12)
                            score = game["score"]
                            if score is not None and color == "black":
                                score = 1-score
                            records.append({"id": game_id, "pair": pair, "color": color, "score": score, "reason": game["reason"]})
                            report["summaries"][key] = strength_summary(records, rating,
                                f"Stockfish UCI_Elo at {clock}+{increment}" if rating is not None else None)
                            atomic_json(path, report)
                            print(key, pair, color, score, game["reason"], flush=True)
            report["status"] = "complete"
        except (RunPaused, KeyboardInterrupt) as exc:
            report["status"] = "paused"
            report["pause_reason"] = str(exc)
        finally:
            report["time_control"] = {"initial_seconds": clock, "increment_seconds": increment}
            report["engine_sha256"] = STOCKFISH_BINARY_SHA
            atomic_json(path, report)
    return report

In [ ]:
BRANCHES = ['v5']
RANKED_EXPERIMENT = False
GRAFT_KW = {"d_model":16,"n_latents":8,"n_layers":1,"n_heads":2} if MODE == "smoke" else {"d_model":96,"n_latents":64,"n_layers":3,"n_heads":4}
BATCH_SIZE = 8 if MODE == "smoke" else int(os.environ["FLY_CHESS_BATCH"])

def make_advanced_model():
    if RANKED_EXPERIMENT:
        return FlyOnlyModel(brain,MOTOR_IDX,T_P).to(DEVICE)
    return FlyChessModel(brain,RELAY_IDX,PREMOTOR_IDX,MOTOR_IDX,ACTIVE_MASK,
                         T_P,T_A,relay_steps=RELAY_STEPS,**GRAFT_KW).to(DEVICE)

torch.manual_seed(SEED)
model = make_advanced_model()
assert not any(p.requires_grad for p in brain.parameters())
if RANKED_EXPERIMENT:
    assert not hasattr(model,"graft")
print("branches",BRANCHES,"trainable parameters",sum(p.numel() for p in model.parameters() if p.requires_grad))
CONFIG = advanced_config(BRANCHES,MODE == "smoke")

MODEL_SOURCE_SHA256 = 'f63e2fa262a16fab1abe9762cac9e8e2643ee267c5b3c92f883aef6c6d35c5ec'
NOTEBOOK_SOURCE = json.loads('{"cells": [{"cell_type": "code", "execution_count": null, "metadata": {"tags": ["encoding"]}, "outputs": [], "source": ["PIECE_INDEX = {chess.PAWN: 0, chess.KNIGHT: 1, chess.BISHOP: 2, chess.ROOK: 3, chess.QUEEN: 4, chess.KING: 5}\\n", "N_FEATURES = 788\\n", "PROMOTION_PAIRS = [(frm, to) for frm in range(48, 56) for to in range(56, 64)\\n", "                   if abs(chess.square_file(frm) - chess.square_file(to)) <= 1]\\n", "PROMOTION_KEYS = [(frm, to, piece) for frm, to in PROMOTION_PAIRS\\n", "                  for piece in (chess.QUEEN, chess.ROOK, chess.BISHOP, chess.KNIGHT)]\\n", "PROMOTION_INDEX = {key: 4096 + i for i, key in enumerate(PROMOTION_KEYS)}\\n", "N_MOVES = 4096 + len(PROMOTION_KEYS)\\n", "assert N_MOVES == 4184\\n", "\\n", "\\n", "def perspective_square(square, us):\\n", "    return square if us == chess.WHITE else chess.square_mirror(square)\\n", "\\n", "\\n", "def encode_base_board(board):\\n", "    x = np.zeros(780, dtype=np.float32)\\n", "    us, them = board.turn, not board.turn\\n", "    for square, piece in board.piece_map().items():\\n", "        sq = perspective_square(square, us)\\n", "        offset = 0 if piece.color == us else 6\\n", "        x[(offset + PIECE_INDEX[piece.piece_type]) * 64 + sq] = 1.0\\n", "    base = 768\\n", "    x[base + 0] = board.has_kingside_castling_rights(us)\\n", "    x[base + 1] = board.has_queenside_castling_rights(us)\\n", "    x[base + 2] = board.has_kingside_castling_rights(them)\\n", "    x[base + 3] = board.has_queenside_castling_rights(them)\\n", "    if board.ep_square is not None:\\n", "        ep = perspective_square(board.ep_square, us)\\n", "        x[base + 4 + chess.square_file(ep)] = 1.0\\n", "    return x\\n", "\\n", "\\n", "def encode_history_board(board):\\n", "    x = np.zeros(788, np.float32)\\n", "    x[:780] = encode_base_board(board)\\n", "    known = bool(board.move_stack)\\n", "    last = board.peek() if known else None\\n", "    x[780:] = [min(board.halfmove_clock / 150, 1), board.is_repetition(2),\\n", "               board.is_repetition(3), known, min(board.fullmove_number / 200, 1),\\n", "               perspective_square(last.from_square,board.turn)/63 if known else 0,\\n", "               perspective_square(last.to_square,board.turn)/63 if known else 0,\\n", "               bool(last.promotion) if known else 0]\\n", "    return x\\n", "\\n", "encode_board = encode_history_board\\n", "\\n", "\\n", "def move_index(move, us):\\n", "    frm = perspective_square(move.from_square, us)\\n", "    to = perspective_square(move.to_square, us)\\n", "    return PROMOTION_INDEX[(frm, to, move.promotion)] if move.promotion else frm * 64 + to\\n", "\\n", "\\n", "def legal_move_table(board):\\n", "    \\"\\"\\"{move_index: [legal chess.Move, ...]} for every legal move, mirrored to the mover\'s frame.\\"\\"\\"\\n", "    us = board.turn\\n", "    table = {}\\n", "    for move in board.legal_moves:\\n", "        table.setdefault(move_index(move, us), []).append(move)\\n", "    return table\\n", "\\n", "\\n", "def index_to_move(idx, board):\\n", "    candidates = legal_move_table(board)[int(idx)]\\n", "    assert len(candidates) == 1, \\"Move vocabulary collision\\"\\n", "    return candidates[0]\\n", "\\n", "\\n", "def terminal_value(board):\\n", "    outcome = board.outcome(claim_draw=True)\\n", "    if outcome is None:\\n", "        return None\\n", "    return 0.5 if outcome.winner is None else float(outcome.winner == board.turn)\\n", "\\n", "\\n", "def source_value(board, cp=None, mate=None):\\n", "    terminal = terminal_value(board)\\n", "    if terminal is not None:\\n", "        return terminal\\n", "    sign = 1 if board.turn == chess.WHITE else -1\\n", "    if mate is not None:\\n", "        if mate == 0:\\n", "            raise ValueError(\\"Nonterminal zero-distance mate score\\")\\n", "        return float(sign * mate > 0)\\n", "    return float(torch.sigmoid(torch.tensor(0.00368208 * sign * cp, dtype=torch.float64)))\\n", "\\n", "\\n", "def position_key(board):\\n", "    return hashlib.sha256(encode_base_board(board).astype(np.uint8).tobytes()).hexdigest()\\n", "\\n", "\\n", "def partition(board):\\n", "    bucket = int(position_key(board)[:16], 16) % 100\\n", "    return \\"train\\" if bucket < 90 else \\"validation\\" if bucket < 95 else \\"test\\"\\n", "\\n", "\\n", "def mask_logits(logits, boards):\\n", "    mask = torch.full_like(logits, -torch.inf)\\n", "    for i, board in enumerate(boards):\\n", "        indices = list(legal_move_table(board))\\n", "        mask[i, indices] = 0\\n", "    return logits + mask"]}, {"cell_type": "code", "execution_count": null, "metadata": {"tags": ["advanced_definitions"]}, "outputs": [], "source": ["\\"\\"\\"Shared V4/V5 data, curriculum, checkpoint and self-play implementation.\\n", "\\n", "The notebook builder embeds this file verbatim. The namespace argument supplies\\n", "the notebook\'s frozen fly simulator, model, encoding, and tested MCTS functions.\\n", "\\"\\"\\"\\n", "import copy\\n", "import gzip\\n", "import hashlib\\n", "import json\\n", "import math\\n", "import os\\n", "import pickle\\n", "from pathlib import Path\\n", "import shutil\\n", "import sqlite3\\n", "import time\\n", "\\n", "import chess\\n", "import chess.engine\\n", "import numpy as np\\n", "import torch\\n", "from torch.nn import functional as F\\n", "\\n", "\\n", "def policy_distribution(scores, temperature=80.0):\\n", "    scores = np.asarray(scores, dtype=np.float64)\\n", "    if len(scores) == 0 or not np.isfinite(scores).all() or temperature <= 0:\\n", "        raise ValueError(\\"Finite, nonempty move scores and positive temperature required\\")\\n", "    weights = np.exp((scores - scores.max()) / temperature)\\n", "    return (weights / weights.sum()).astype(np.float32)\\n", "\\n", "\\n", "def ranking_score(cp=None, mate=None):\\n", "    if mate is not None:\\n", "        return (10000 - min(abs(mate), 90)*100) * (1 if mate > 0 else -1)\\n", "    return float(np.clip(cp, -2000, 2000))\\n", "\\n", "\\n", "def soft_policy_loss(logits, move_indices, probabilities):\\n", "    indices = torch.as_tensor(move_indices, device=logits.device, dtype=torch.long)\\n", "    weights = torch.as_tensor(probabilities, device=logits.device, dtype=torch.float32)\\n", "    if indices.ndim != 2 or weights.shape != indices.shape or indices.shape[0] != len(logits):\\n", "        raise ValueError(\\"Policy targets must have shape (batch, candidates)\\")\\n", "    if torch.any(weights < 0) or not torch.allclose(weights.sum(1), torch.ones(len(logits), device=logits.device, dtype=torch.float32), atol=.002):\\n", "        raise ValueError(\\"Target probabilities must sum to one\\")\\n", "    gathered = F.log_softmax(logits.float(), dim=1).gather(1, indices)\\n", "    if not torch.isfinite(gathered[weights > 0]).all():\\n", "        raise ValueError(\\"A target move is illegal or its probability is nonfinite\\")\\n", "    return -(torch.where(weights > 0, gathered, torch.zeros_like(gathered)) * weights).sum(1).mean()\\n", "\\n", "\\n", "def curriculum_weights(phase):\\n", "    # Foundation, tactics, endgame, ordinary. Earlier skills remain in later phases.\\n", "    return {\\"foundation\\": (.55, .20, .20, .05),\\n", "            \\"general\\": (.15, .25, .25, .35),\\n", "            \\"consolidation\\": (.10, .25, .25, .40)}[phase]\\n", "\\n", "\\n", "def stratified_pool(groups, count, rng, phase=\\"general\\"):\\n", "    chosen = []\\n", "    weights = curriculum_weights(phase)\\n", "    available = [np.asarray(g, dtype=np.int64) for g in groups]\\n", "    if not any(len(g) for g in available):\\n", "        raise ValueError(\\"Empty training split\\")\\n", "    for group, weight in zip(available, weights):\\n", "        if len(group):\\n", "            chosen.extend(rng.choice(group, min(len(group), int(count*weight)), replace=False).tolist())\\n", "    chosen = np.unique(np.asarray(chosen, dtype=np.int64))\\n", "    total = np.unique(np.concatenate(available))\\n", "    if len(chosen) < min(count, len(total)):\\n", "        rest = np.setdiff1d(total, chosen)\\n", "        chosen = np.concatenate([chosen, rng.choice(rest, min(count-len(chosen), len(rest)), replace=False)])\\n", "    rng.shuffle(chosen)\\n", "    return chosen\\n", "\\n", "\\n", "def training_groups(data, train_indices):\\n", "    groups = [[], [], [], []]\\n", "    fens, moves = data[\\"fen\\"], data[\\"move\\"]\\n", "    for offset, i in enumerate(train_indices):\\n", "        board = chess.Board(str(fens[i]))\\n", "        move = chess.Move.from_uci(str(moves[i]))\\n", "        pieces = len(board.piece_map())\\n", "        short_mate = bool(data[\\"is_mate\\"][i]) and abs(int(data[\\"raw_mate\\"][i])) <= 2\\n", "        category = 0 if short_mate else 1 if board.is_check() or board.is_capture(move) or move.promotion else 2 if pieces <= 10 else 3\\n", "        groups[category].append(int(i))\\n", "        if offset and offset % 100000 == 0:\\n", "            print(\\"curriculum indexing\\", offset, \\"/\\", len(train_indices), flush=True)\\n", "    return [np.asarray(g, dtype=np.int64) for g in groups]\\n", "\\n", "\\n", "def ingest_advanced(lines, limit, min_depth, ns):\\n", "    rows, seen, rejected = [], set(), 0\\n", "    for raw in lines:\\n", "        ns[\\"check_stop\\"]()\\n", "        try:\\n", "            source = json.loads(raw)\\n", "            board = chess.Board(source[\\"fen\\"])\\n", "            if not board.is_valid() or ns[\\"terminal_value\\"](board) is not None:\\n", "                raise ValueError(\\"invalid_or_terminal\\")\\n", "            key = ns[\\"position_key\\"](board)\\n", "            if key in seen:\\n", "                continue\\n", "            best = max(source[\\"evals\\"], key=lambda e: (e[\\"depth\\"], e.get(\\"knodes\\", 0)))\\n", "            if best[\\"depth\\"] < min_depth:\\n", "                continue\\n", "            candidates, scores = [], []\\n", "            sign = 1 if board.turn else -1\\n", "            for pv in best[\\"pvs\\"]:\\n", "                move = board.parse_uci(pv[\\"line\\"].split()[0])\\n", "                if move.uci() in candidates:\\n", "                    continue\\n", "                candidates.append(move.uci())\\n", "                scores.append(ranking_score(sign*pv[\\"cp\\"] if \\"cp\\" in pv else None,\\n", "                                           sign*pv[\\"mate\\"] if \\"mate\\" in pv else None))\\n", "                if len(candidates) == 3:\\n", "                    break\\n", "            probabilities = policy_distribution(scores)\\n", "            pv = best[\\"pvs\\"][0]\\n", "            rows.append((board.fen(en_passant=\\"fen\\"), candidates[0],\\n", "                         ns[\\"source_value\\"](board, pv.get(\\"cp\\"), pv.get(\\"mate\\")),\\n", "                         best[\\"depth\\"], pv.get(\\"cp\\", 0), pv.get(\\"mate\\", 0), \\"mate\\" in pv,\\n", "                         key, ns[\\"partition\\"](board),\\n", "                         candidates + [candidates[0]]*(3-len(candidates)),\\n", "                         probabilities.tolist() + [0.]*(3-len(candidates))))\\n", "            seen.add(key)\\n", "        except (ValueError, KeyError, IndexError, TypeError):\\n", "            rejected += 1\\n", "        if len(rows) >= limit:\\n", "            break\\n", "    return rows, rejected\\n", "\\n", "\\n", "def prepare_advanced_data(ns):\\n", "    count = int(os.environ.get(\\"FLY_CHESS_POSITIONS\\", \\"2000\\" if ns[\\"MODE\\"] == \\"smoke\\" else \\"1000000\\"))\\n", "    path = ns[\\"CHESS_DATA\\"] / f\\"positions-v4-{ns[\'MODE\']}-{count}.npz\\"\\n", "    pin_path = path.with_suffix(\\".json\\")\\n", "    if not path.exists():\\n", "        legacy = Path(os.environ.get(\\"FLY_CHESS_SOURCE_DATASET\\", str(ns[\\"CHESS_DATA\\"] / f\\"positions-v2-{ns[\'MODE\']}-{count}.npz\\")))\\n", "        if legacy.exists():\\n", "            legacy_pin = json.loads(legacy.with_suffix(\\".json\\").read_text())\\n", "            if ns[\\"sha256\\"](legacy) != legacy_pin[\\"sha256\\"]:\\n", "                raise ValueError(\\"Source dataset integrity mismatch\\")\\n", "            with np.load(legacy, allow_pickle=False) as old:\\n", "                arrays = {k:old[k] for k in old.files}\\n", "            if \\"policy_moves\\" not in arrays:\\n", "                arrays[\\"policy_moves\\"] = np.repeat(arrays[\\"move\\"][:, None], 3, axis=1)\\n", "                arrays[\\"policy_probs\\"] = np.tile(np.array([1, 0, 0], np.float32), (len(arrays[\\"move\\"]), 1))\\n", "            # V4/V5 use the same canonical split as V3. Cached source labels are\\n", "            # explicitly identified as single-move/proxy labels until relabelled.\\n", "            origin = {\\"source\\": str(legacy), \\"source_sha256\\": legacy_pin[\\"sha256\\"], \\"policy\\": \\"copied_source_multipv\\" if legacy_pin.get(\\"schema\\") == 4 else \\"single_move_source\\"}\\n", "        else:\\n", "            import contextlib\\n", "            with contextlib.closing(ns[\\"stream_zst_lines\\"](\\"https://database.lichess.org/lichess_db_eval.jsonl.zst\\")) as lines:\\n", "                rows, rejected = ingest_advanced(lines, count, 10 if ns[\\"MODE\\"] == \\"smoke\\" else 14, ns)\\n", "            if not rows:\\n", "                raise ValueError(\\"No accepted positions\\")\\n", "            names = (\\"fen\\", \\"move\\", \\"value\\", \\"depth\\", \\"raw_cp\\", \\"raw_mate\\", \\"is_mate\\", \\"key\\", \\"split\\", \\"policy_moves\\", \\"policy_probs\\")\\n", "            arrays = {name: np.array(col) for name, col in zip(names, zip(*rows))}\\n", "            origin = {\\"source\\": \\"https://database.lichess.org/lichess_db_eval.jsonl.zst\\", \\"rejected\\": rejected, \\"policy\\": \\"available_source_multipv\\"}\\n", "        np.savez_compressed(path, **arrays)\\n", "        ns[\\"atomic_json\\"](pin_path, {\\"schema\\": 4, \\"sha256\\": ns[\\"sha256\\"](path), \\"origin\\": origin,\\n", "                                      \\"source_value\\": \\"side_to_move_centipawn_proxy; pinned teacher supplies WDL\\"})\\n", "    pin = json.loads(pin_path.read_text())\\n", "    if pin[\\"schema\\"] != 4 or ns[\\"sha256\\"](path) != pin[\\"sha256\\"]:\\n", "        raise ValueError(\\"V4 dataset integrity mismatch\\")\\n", "    with np.load(path, allow_pickle=False) as archive:\\n", "        data = {k:archive[k] for k in archive.files}\\n", "    ns.update(positions_path=path, positions=data, fens=data[\\"fen\\"], labels_uci=data[\\"move\\"], values=data[\\"value\\"],\\n", "              train_idx=np.flatnonzero(data[\\"split\\"] == \\"train\\"),\\n", "              validation_idx=np.flatnonzero(data[\\"split\\"] == \\"validation\\"), test_idx=np.flatnonzero(data[\\"split\\"] == \\"test\\"))\\n", "    if not all(len(ns[k]) for k in (\\"train_idx\\", \\"validation_idx\\", \\"test_idx\\")):\\n", "        raise ValueError(\\"All three dataset splits must be populated\\")\\n", "    print(\\"dataset\\", path, \\"positions\\", len(data[\\"fen\\"]), \\"split sizes\\", *[len(ns[k]) for k in (\\"train_idx\\", \\"validation_idx\\", \\"test_idx\\")], flush=True)\\n", "    return data\\n", "\\n", "\\n", "class TeacherStore:\\n", "    \\"\\"\\"Local SQLite with atomic persistent snapshots; one writer per run.\\"\\"\\"\\n", "    def __init__(self, local, persistent, config):\\n", "        self.local, self.persistent = Path(local), Path(persistent)\\n", "        self.local.parent.mkdir(parents=True, exist_ok=True)\\n", "        self.persistent.parent.mkdir(parents=True, exist_ok=True)\\n", "        if not self.local.exists() and self.persistent.exists():\\n", "            shutil.copyfile(self.persistent, self.local)\\n", "        self.db = sqlite3.connect(self.local)\\n", "        self.db.execute(\\"CREATE TABLE IF NOT EXISTS metadata (key TEXT PRIMARY KEY, value TEXT)\\")\\n", "        self.db.execute(\\"CREATE TABLE IF NOT EXISTS labels (key TEXT PRIMARY KEY, value TEXT)\\")\\n", "        encoded = json.dumps(config, sort_keys=True)\\n", "        old = self.db.execute(\\"SELECT value FROM metadata WHERE key=\'config\'\\").fetchone()\\n", "        if old and old[0] != encoded:\\n", "            raise ValueError(\\"Teacher configuration changed; start a new run\\")\\n", "        self.db.execute(\\"INSERT OR IGNORE INTO metadata VALUES (\'config\', ?)\\", (encoded,))\\n", "        self.db.commit()\\n", "    def get(self, key):\\n", "        row = self.db.execute(\\"SELECT value FROM labels WHERE key=?\\", (key,)).fetchone()\\n", "        return json.loads(row[0]) if row else None\\n", "    def put(self, key, value):\\n", "        self.db.execute(\\"INSERT OR IGNORE INTO labels VALUES (?, ?)\\", (key, json.dumps(value, allow_nan=False)))\\n", "        self.db.commit()\\n", "    def count(self):\\n", "        return self.db.execute(\\"SELECT COUNT(*) FROM labels\\").fetchone()[0]\\n", "    def stage_done(self, key):\\n", "        return self.db.execute(\\"SELECT 1 FROM metadata WHERE key=?\\", (\\"stage:\\"+key,)).fetchone() is not None\\n", "    def mark_stage(self, key):\\n", "        self.db.execute(\\"INSERT OR IGNORE INTO metadata VALUES (?, \'done\')\\", (\\"stage:\\"+key,))\\n", "        self.db.commit()\\n", "    def snapshot(self):\\n", "        temporary = self.persistent.with_suffix(\\".tmp.sqlite\\")\\n", "        with sqlite3.connect(temporary) as destination:\\n", "            self.db.backup(destination)\\n", "        temporary.replace(self.persistent)\\n", "    def close(self):\\n", "        self.db.close()\\n", "\\n", "\\n", "def teacher_label(engine, board, nodes=50000, candidates=3):\\n", "    information = engine.analyse(board, chess.engine.Limit(nodes=nodes), multipv=min(candidates, board.legal_moves.count()), game=object())\\n", "    if isinstance(information, dict):\\n", "        information = [information]\\n", "    moves, ranks = [], []\\n", "    for info in information:\\n", "        move = info[\\"pv\\"][0]\\n", "        score = info[\\"score\\"].pov(board.turn)\\n", "        moves.append(move.uci())\\n", "        ranks.append(ranking_score(score.score(), score.mate()))\\n", "    best = information[0]\\n", "    if \\"wdl\\" not in best:\\n", "        raise ValueError(\\"Pinned teacher did not produce WDL; enable UCI_ShowWDL\\")\\n", "    wdl = best[\\"wdl\\"].pov(board.turn)\\n", "    return {\\"moves\\": moves, \\"probs\\": policy_distribution(ranks).tolist(), \\"value\\": wdl.expectation(),\\n", "            \\"wdl\\": [wdl.wins, wdl.draws, wdl.losses], \\"depth\\": best.get(\\"depth\\"),\\n", "            \\"nodes\\": best.get(\\"nodes\\"), \\"value_source\\": \\"pinned_stockfish_wdl\\"}\\n", "\\n", "\\n", "def replay_board(record):\\n", "    board = chess.Board(record[\\"start_fen\\"])\\n", "    history = record[\\"history\\"].split() if isinstance(record[\\"history\\"], str) else record[\\"history\\"]\\n", "    for uci in history:\\n", "        board.push_uci(uci)\\n", "    return board\\n", "\\n", "\\n", "def record_moves(record):\\n", "    return record[\\"moves\\"].split() if isinstance(record[\\"moves\\"],str) else record[\\"moves\\"]\\n", "\\n", "\\n", "def clone_checkpoint_model(ns, state, path):\\n", "    rng = ns[\\"rng_state\\"](state[\\"sampler\\"])\\n", "    try:\\n", "        saved = ns[\\"load_checkpoint\\"](path,state[\\"identity\\"])\\n", "        model = ns[\\"make_advanced_model\\"]().to(ns[\\"DEVICE\\"])\\n", "        model.load_state_dict(saved[\\"model\\"])\\n", "        model.eval()\\n", "        return model\\n", "    finally:\\n", "        ns[\\"restore_rng\\"](rng,state[\\"sampler\\"])\\n", "\\n", "\\n", "def outcome_targets(samples, winner):\\n", "    return [dict(sample, value=.5 if winner is None else float(sample[\\"turn\\"] == winner),\\n", "                 value_source=\\"completed_selfplay_outcome\\") for sample in samples]\\n", "\\n", "\\n", "def search_target(ns, model, board, simulations, rng, exploration=False):\\n", "    priors, _ = ns[\\"evaluate_leaves\\"](model, [board])\\n", "    if exploration:\\n", "        indices = list(priors[0])\\n", "        noise = rng.dirichlet(np.full(len(indices), .3))\\n", "        priors[0] = {i:.75*priors[0][i] + .25*float(n) for i,n in zip(indices,noise)}\\n", "    root = ns[\\"search\\"](model, board, n_simulations=simulations, batch=min(16, simulations), root_priors=priors[0])\\n", "    indices = list(root.children)\\n", "    counts = np.array([root.children[i].visits for i in indices], np.float64)\\n", "    if not len(indices) or counts.sum() == 0:\\n", "        raise ValueError(\\"Search produced no visit targets\\")\\n", "    return indices, (counts/counts.sum()).astype(np.float32), root\\n", "\\n", "\\n", "class AdvancedTrainer:\\n", "    def __init__(self, ns, config):\\n", "        self.ns, self.config = ns, config\\n", "        self.base = Path(ns[\\"RUN_DIR\\"])\\n", "        self.device = ns[\\"DEVICE\\"]\\n", "        self.smoke = ns[\\"MODE\\"] == \\"smoke\\"\\n", "        self.round_size = config[\\"round_size\\"]\\n", "        self.data = ns[\\"positions\\"]\\n", "        self.groups = training_groups(self.data, ns[\\"train_idx\\"])\\n", "        self.validation = np.random.RandomState(101).choice(ns[\\"validation_idx\\"], min(config[\\"validation_size\\"], len(ns[\\"validation_idx\\"])), replace=False)\\n", "        teacher_config = {\\"engine_sha256\\": ns[\\"STOCKFISH_BINARY_SHA\\"], \\"nodes\\": config[\\"teacher_nodes\\"],\\n", "                          \\"multipv\\": 3, \\"threads\\": 1, \\"temperature_cp\\": 80}\\n", "        self.cache_scope = hashlib.sha256(str(self.base.resolve()).encode()).hexdigest()[:12]\\n", "        self.teacher = TeacherStore(ns[\\"ROOT\\"] / \\"teacher-cache\\" / self.cache_scope / \\"labels.sqlite\\",\\n", "                                    self.base / \\"teacher.sqlite\\", teacher_config)\\n", "        physical = {\\"version\\": 4, \\"mode\\": ns[\\"MODE\\"], \\"real_graph\\": ns[\\"USE_REAL_GRAPH\\"],\\n", "                    \\"dataset_sha256\\": ns[\\"sha256\\"](ns[\\"positions_path\\"]),\\n", "                    \\"graph_sha256\\": ns[\\"sha256\\"](ns[\\"GRAPH_PATH\\"]) if ns[\\"USE_REAL_GRAPH\\"] else \\"synthetic\\",\\n", "                    \\"nodes_sha256\\": ns[\\"sha256\\"](ns[\\"NODES_PATH\\"]) if ns[\\"USE_REAL_GRAPH\\"] else \\"synthetic\\",\\n", "                    \\"threshold\\": ns[\\"EDGE_MIN_SYNAPSES\\"], \\"gain\\": ns[\\"GAIN\\"], \\"alpha\\": ns[\\"ALPHA\\"],\\n", "                    \\"perception_steps\\": ns[\\"T_P\\"], \\"motor_steps\\": ns[\\"T_A\\"], \\"relay_steps\\": ns[\\"RELAY_STEPS\\"],\\n", "                    \\"injection_seed\\": 0, \\"current_amplitude\\": .5, \\"features\\": ns[\\"N_FEATURES\\"], \\"moves\\": ns[\\"N_MOVES\\"],\\n", "                    \\"graft\\": ns[\\"GRAFT_KW\\"], \\"architecture\\": \\"history-sensory-curriculum-v4-v5\\",\\n", "                    \\"source_sha256\\": ns[\\"MODEL_SOURCE_SHA256\\"], \\"config\\": config,\\n", "                    \\"engine_sha256\\": ns[\\"STOCKFISH_BINARY_SHA\\"], \\"batch_size\\": ns[\\"BATCH_SIZE\\"]}\\n", "        for field, name in ((\\"sensory_ids\\",\\"SENSORY_IDX\\"),(\\"relay_ids\\",\\"RELAY_IDX\\"),(\\"premotor_ids\\",\\"PREMOTOR_IDX\\"),(\\"motor_ids\\",\\"MOTOR_IDX\\")):\\n", "            physical[field] = ns[\\"ids\\"][ns[name]].tolist()\\n", "        self.cache_identity = hashlib.sha256(json.dumps(physical,sort_keys=True).encode()).hexdigest()\\n", "        self.states = {}\\n", "        initial = copy.deepcopy(ns[\\"model\\"].state_dict())\\n", "        for branch in config[\\"branches\\"]:\\n", "            directory = self.base / branch\\n", "            directory.mkdir(exist_ok=True)\\n", "            manifest = dict(physical, version={\\"v4\\":4,\\"v5\\":5,\\"v55\\":5.5}[branch], branch=branch)\\n", "            if branch == \\"v55\\":\\n", "                manifest.update(architecture=\\"ranked-lookahead-fly-only-v55\\", graft={}, current_amplitude=0,\\n", "                                assistance={\\"source\\":\\"material_minimax\\", \\"depth_plies\\":2, \\"ranked_k\\":4,\\n", "                                            \\"route\\":\\"sensory neurons only\\", \\"policy_choices\\":\\"all legal moves\\"})\\n", "            path = directory / \\"manifest.json\\"\\n", "            if path.exists() and json.loads(path.read_text()) != manifest:\\n", "                raise ValueError(\\"Run configuration changed; use FLY_CHESS_NEW_RUN=1 once\\")\\n", "            ns[\\"atomic_json\\"](path, manifest)\\n", "            # Save the exact implementation alongside every branch checkpoint.\\n", "            if ns.get(\\"NOTEBOOK_SOURCE\\"):\\n", "                ns[\\"atomic_json\\"](directory / \\"model-source.ipynb\\", ns[\\"NOTEBOOK_SOURCE\\"])\\n", "            identity = hashlib.sha256(json.dumps(manifest,sort_keys=True).encode()).hexdigest()\\n", "            construction_rng = ns[\\"rng_state\\"](np.random.RandomState(0))\\n", "            model = ns[\\"make_advanced_model\\"]().to(self.device)\\n", "            ns[\\"restore_rng\\"](construction_rng, np.random.RandomState(0))\\n", "            model.load_state_dict(initial)\\n", "            optimizer = torch.optim.AdamW(model.parameters(),lr=config[\\"lr\\"],weight_decay=.01)\\n", "            scaler = torch.amp.GradScaler(\\"cuda\\",enabled=self.device.type == \\"cuda\\",init_scale=128)\\n", "            state = {\\"model\\":model,\\"optimizer\\":optimizer,\\"scaler\\":scaler,\\"sampler\\":np.random.RandomState(0),\\n", "                     \\"step\\":0,\\"round\\":0,\\"phase\\":\\"foundation\\",\\"best_loss\\":None,\\"stale\\":0,\\"status\\":\\"active\\",\\n", "                     \\"directory\\":directory,\\"identity\\":identity,\\"replay\\":[],\\"selfplay_games\\":0,\\"search_gate\\":False,\\n", "                     \\"collection_round\\":-1,\\"round_games\\":0,\\"collected_round\\":-1,\\"arena_pending\\":False,\\"champion_step\\":0}\\n", "            saved = ns[\\"load_checkpoint\\"](directory / \\"main.pt\\",identity)\\n", "            if saved:\\n", "                model.load_state_dict(saved[\\"model\\"])\\n", "                optimizer.load_state_dict(saved[\\"optimizer\\"])\\n", "                scaler.load_state_dict(saved[\\"scaler\\"])\\n", "                state.update({k:saved[k] for k in (\\"step\\",\\"round\\",\\"phase\\",\\"best_loss\\",\\"stale\\",\\"status\\",\\"selfplay_games\\",\\"search_gate\\",\\n", "                                                     \\"collection_round\\",\\"round_games\\",\\"collected_round\\",\\"arena_pending\\",\\"champion_step\\")})\\n", "                state[\\"sampler\\"].set_state(saved[\\"sampler\\"])\\n", "                replay_path = directory / saved[\\"replay_file\\"]\\n", "                if ns[\\"sha256\\"](replay_path) != saved[\\"replay_sha256\\"]:\\n", "                    raise ValueError(\\"Replay snapshot integrity mismatch\\")\\n", "                with gzip.open(replay_path,\\"rt\\") as handle:state[\\"replay\\"] = json.load(handle)\\n", "                if self.teacher.count() < saved[\\"teacher_count\\"]:\\n", "                    raise ValueError(\\"Teacher snapshot is older than the checkpoint\\")\\n", "                state[\\"rng\\"] = saved[\\"rng\\"]\\n", "            else:\\n", "                ns[\\"initialize_signal_stats\\"](model, ns[\\"train_idx\\"][:min(64,len(ns[\\"train_idx\\"]))])\\n", "                # Every branch begins with the same stochastic training stream.\\n", "                torch.manual_seed(ns[\\"SEED\\"])\\n", "                np.random.seed(ns[\\"SEED\\"])\\n", "                state[\\"rng\\"] = ns[\\"rng_state\\"](state[\\"sampler\\"])\\n", "                ns[\\"save_checkpoint\\"](directory / \\"main.champion.pt\\",{\\n", "                    \\"identity\\":identity,\\"model\\":model.state_dict(),\\"step\\":0})\\n", "            self.states[branch] = state\\n", "        del initial\\n", "        # Remove the construction model to keep only the two active branches.\\n", "        ns.pop(\\"model\\", None)\\n", "        self.cache = None\\n", "        self.cache_key = None\\n", "\\n", "    def source_record(self, index):\\n", "        board = chess.Board(str(self.data[\\"fen\\"][index]))\\n", "        key = board.fen(en_passant=\\"fen\\")\\n", "        label = self.teacher.get(key)\\n", "        if not label:\\n", "            label = {\\"moves\\":self.data[\\"policy_moves\\"][index].tolist(),\\n", "                     \\"probs\\":self.data[\\"policy_probs\\"][index].tolist(),\\"value\\":float(self.data[\\"value\\"][index]),\\n", "                     \\"value_source\\":\\"source_cp_proxy\\"}\\n", "        return {\\"start_fen\\":key,\\"history\\":[],\\"turn\\":board.turn,**label,\\"index\\":int(index)}\\n", "\\n", "    def save(self, state):\\n", "        self.teacher.snapshot()\\n", "        replay_path = state[\\"directory\\"] / f\\"replay-{state[\'selfplay_games\']:06d}.json.gz\\"\\n", "        if not replay_path.exists():\\n", "            temporary = replay_path.with_suffix(\\".tmp\\")\\n", "            with gzip.open(temporary,\\"wt\\") as handle:json.dump(state[\\"replay\\"],handle,allow_nan=False)\\n", "            temporary.replace(replay_path)\\n", "        saved = {k:state[k] for k in (\\"identity\\",\\"step\\",\\"round\\",\\"phase\\",\\"best_loss\\",\\"stale\\",\\"status\\",\\"selfplay_games\\",\\"search_gate\\",\\n", "                                     \\"collection_round\\",\\"round_games\\",\\"collected_round\\",\\"arena_pending\\",\\"champion_step\\")}\\n", "        saved.update(model=state[\\"model\\"].state_dict(),optimizer=state[\\"optimizer\\"].state_dict(),\\n", "                     scaler=state[\\"scaler\\"].state_dict(),sampler=state[\\"sampler\\"].get_state(),\\n", "                     rng=self.ns[\\"rng_state\\"](state[\\"sampler\\"]),teacher_count=self.teacher.count(),\\n", "                     replay_file=replay_path.name,replay_sha256=self.ns[\\"sha256\\"](replay_path))\\n", "        self.ns[\\"save_checkpoint\\"](state[\\"directory\\"] / \\"main.pt\\",saved)\\n", "        self.ns[\\"atomic_json\\"](state[\\"directory\\"] / \\"progress.json\\",{k:saved[k] for k in (\\"step\\",\\"round\\",\\"phase\\",\\"best_loss\\",\\"stale\\",\\"status\\",\\"selfplay_games\\",\\"search_gate\\",\\"teacher_count\\")})\\n", "\\n", "    def loss_batch(self, state, records, cache=None):\\n", "        boards = [replay_board(r) for r in records]\\n", "        features = None if cache else torch.tensor(np.stack([self.ns[\\"encode_board\\"](b) for b in boards]),device=self.device)\\n", "        width = max(len(record_moves(r)) for r in records)\\n", "        indices = np.zeros((len(records),width),np.int64)\\n", "        probs = np.zeros((len(records),width),np.float32)\\n", "        for row,(record,board) in enumerate(zip(records,boards)):\\n", "            choices = [self.ns[\\"move_index\\"](chess.Move.from_uci(m),board.turn) for m in record_moves(record)]\\n", "            indices[row] = choices[0]\\n", "            indices[row,:len(choices)] = choices\\n", "            probs[row,:len(choices)] = record[\\"probs\\"]\\n", "        target = torch.tensor([r[\\"value\\"] for r in records],device=self.device,dtype=torch.float32)\\n", "        state[\\"model\\"].train()\\n", "        with torch.autocast(device_type=self.device.type,enabled=self.device.type == \\"cuda\\",dtype=torch.float16):\\n", "            if cache:\\n", "                logits,value = self.ns[\\"cached_forward\\"](state[\\"model\\"],cache,[r[\\"index\\"] for r in records])\\n", "            else:\\n", "                logits,value = state[\\"model\\"](features)\\n", "            policy = soft_policy_loss(self.ns[\\"mask_logits\\"](logits,boards),indices,probs)\\n", "            value_loss = F.binary_cross_entropy_with_logits(value.float(),target)\\n", "            loss = policy + self.config[\\"value_weight\\"]*value_loss\\n", "        if not torch.isfinite(loss):\\n", "            raise ValueError(\\"Nonfinite loss; previous checkpoint retained\\")\\n", "        state[\\"optimizer\\"].zero_grad(set_to_none=True)\\n", "        state[\\"scaler\\"].scale(loss).backward()\\n", "        state[\\"scaler\\"].unscale_(state[\\"optimizer\\"])\\n", "        gradients = [p.grad for p in state[\\"model\\"].parameters() if p.grad is not None]\\n", "        if any(not torch.isfinite(g).all() for g in gradients):\\n", "            if state[\\"scaler\\"].is_enabled() and state[\\"scaler\\"].get_scale() > 1:\\n", "                state[\\"scaler\\"].step(state[\\"optimizer\\"])  # scaler skips the overflowed update\\n", "                state[\\"scaler\\"].update()\\n", "                state[\\"optimizer\\"].zero_grad(set_to_none=True)\\n", "                print(\\"AMP overflow skipped; scale\\",state[\\"scaler\\"].get_scale(),flush=True)\\n", "                return None\\n", "            raise ValueError(\\"Nonfinite gradients at minimum loss scale; previous checkpoint retained\\")\\n", "        torch.nn.utils.clip_grad_norm_(state[\\"model\\"].parameters(),5,error_if_nonfinite=True)\\n", "        state[\\"scaler\\"].step(state[\\"optimizer\\"])\\n", "        state[\\"scaler\\"].update()\\n", "        return float(loss.detach())\\n", "\\n", "    def relabel(self, indices, deadline, limit=None):\\n", "        missing = [i for i in indices if self.teacher.get(chess.Board(str(self.data[\\"fen\\"][i])).fen(en_passant=\\"fen\\")) is None]\\n", "        if not missing:\\n", "            return\\n", "        with self.ns[\\"open_stockfish\\"]() as engine:\\n", "            engine.configure({\\"Threads\\":1,\\"Hash\\":128,\\"Skill Level\\":20,\\"UCI_LimitStrength\\":False,\\"UCI_ShowWDL\\":True})\\n", "            for offset, i in enumerate(missing[:limit or self.config[\\"relabel_per_round\\"]]):\\n", "                self.ns[\\"check_stop\\"]()\\n", "                if time.monotonic() >= deadline:\\n", "                    break\\n", "                board = chess.Board(str(self.data[\\"fen\\"][i]))\\n", "                self.teacher.put(board.fen(en_passant=\\"fen\\"),teacher_label(engine,board,self.config[\\"teacher_nodes\\"]))\\n", "                if offset % 32 == 0:\\n", "                    print(\\"teacher labels\\",self.teacher.count(),flush=True)\\n", "        self.teacher.snapshot()\\n", "\\n", "    def search_gate(self, state):\\n", "        # A small training-only probe. Validation/test positions never enter replay.\\n", "        indices = stratified_pool(self.groups,min(self.config[\\"gate_positions\\"],len(self.ns[\\"train_idx\\"])),np.random.RandomState(313),\\"general\\")\\n", "        policy_score, search_score = [], []\\n", "        with self.ns[\\"open_stockfish\\"]() as engine:\\n", "            engine.configure({\\"Threads\\":1,\\"Hash\\":128,\\"Skill Level\\":20,\\"UCI_LimitStrength\\":False,\\"UCI_ShowWDL\\":True})\\n", "            for i in indices:\\n", "                board = chess.Board(str(self.data[\\"fen\\"][i]))\\n", "                raw = self.ns[\\"choose_move\\"](state[\\"model\\"],board,False)\\n", "                choices,visits,_ = search_target(self.ns,state[\\"model\\"],board,self.config[\\"simulations\\"],state[\\"sampler\\"])\\n", "                improved = self.ns[\\"index_to_move\\"](choices[int(visits.argmax())],board)\\n", "                def evaluate(move):\\n", "                    info = engine.analyse(board,chess.engine.Limit(nodes=self.config[\\"teacher_nodes\\"]),root_moves=[move],game=object())\\n", "                    return info[\\"score\\"].pov(board.turn).score(mate_score=10000)\\n", "                policy_score.append(evaluate(raw)); search_score.append(evaluate(improved))\\n", "        delta = np.asarray(search_score)-np.asarray(policy_score)\\n", "        accepted = float(delta.mean()) > 0 and np.count_nonzero(delta > 0) > np.count_nonzero(delta < 0)\\n", "        report = {\\"n\\":len(indices),\\"mean_search_gain_cp\\":float(delta.mean()),\\"accepted\\":bool(accepted),\\n", "                  \\"note\\":\\"Small training-only engineering gate, not a strength estimate\\"}\\n", "        self.ns[\\"atomic_json\\"](state[\\"directory\\"] / \\"search-gate.json\\",report)\\n", "        print(\\"v5 search gate\\",report,flush=True)\\n", "        return bool(accepted)\\n", "\\n", "    def collect_selfplay(self, state, deadline):\\n", "        path = state[\\"directory\\"] / \\"selfplay-in-progress.json\\"\\n", "        if state[\\"collection_round\\"] != state[\\"round\\"]:\\n", "            state[\\"collection_round\\"],state[\\"round_games\\"] = state[\\"round\\"],0\\n", "        while state[\\"round_games\\"] < self.config[\\"games_per_round\\"]:\\n", "            saved = json.loads(path.read_text()) if path.exists() else None\\n", "            if saved:\\n", "                if saved[\\"game_number\\"] <= state[\\"selfplay_games\\"]:\\n", "                    path.unlink();continue\\n", "                if saved[\\"generator_step\\"] != state[\\"step\\"]:\\n", "                    raise ValueError(\\"In-progress game belongs to different generator weights\\")\\n", "                board = replay_board(saved); samples = saved[\\"samples\\"]\\n", "                self.ns[\\"restore_rng\\"](pickle.loads(bytes.fromhex(saved[\\"rng\\"])),state[\\"sampler\\"])\\n", "                opponent_info = saved.get(\\"opponent\\")\\n", "            else:\\n", "                board = chess.Board()\\n", "                opening = self.ns[\\"make_opening_pairs\\"](max(1,self.config[\\"games_per_round\\"]),seed=41+state[\\"selfplay_games\\"])[0]\\n", "                for move in opening: board.push_uci(move)\\n", "                samples = []\\n", "                opponent_info = None\\n", "                champion = state[\\"directory\\"] / \\"main.champion.pt\\"\\n", "                if state[\\"champion_step\\"] > 0 and state[\\"sampler\\"].rand() < .25:\\n", "                    opponent_info = {\\"checkpoint\\":champion.name,\\"sha256\\":self.ns[\\"sha256\\"](champion),\\n", "                                     \\"agent_color\\":bool(state[\\"sampler\\"].randint(2))}\\n", "            opponent = None\\n", "            if opponent_info:\\n", "                checkpoint_path = state[\\"directory\\"] / opponent_info[\\"checkpoint\\"]\\n", "                if self.ns[\\"sha256\\"](checkpoint_path) != opponent_info[\\"sha256\\"]:\\n", "                    raise ValueError(\\"Self-play opponent changed during an unfinished game\\")\\n", "                opponent = clone_checkpoint_model(self.ns,state,checkpoint_path)\\n", "            while self.ns[\\"terminal_value\\"](board) is None and len(board.move_stack) < self.config[\\"game_cap\\"]:\\n", "                self.ns[\\"check_stop\\"]()\\n", "                if time.monotonic() >= deadline:\\n", "                    return False\\n", "                actor_turn = opponent is None or board.turn == opponent_info[\\"agent_color\\"]\\n", "                generator = state[\\"model\\"] if actor_turn else opponent\\n", "                choices,probs,_ = search_target(self.ns,generator,board,self.config[\\"simulations\\"],state[\\"sampler\\"],True)\\n", "                sample = {\\"start_fen\\":chess.STARTING_FEN,\\"history\\":\\" \\".join(m.uci() for m in board.move_stack),\\n", "                          \\"turn\\":board.turn,\\"moves\\":\\" \\".join(self.ns[\\"index_to_move\\"](i,board).uci() for i in choices),\\"probs\\":probs.tolist(),\\n", "                          \\"generator_step\\":state[\\"step\\"],\\"simulations\\":self.config[\\"simulations\\"]}\\n", "                if actor_turn:samples.append(sample)\\n", "                choice = state[\\"sampler\\"].choice(len(choices),p=probs.astype(np.float64)/probs.astype(np.float64).sum()) if len(board.move_stack) < 30 else int(probs.argmax())\\n", "                board.push(self.ns[\\"index_to_move\\"](choices[choice],board))\\n", "                self.ns[\\"atomic_json\\"](path,{\\"start_fen\\":chess.STARTING_FEN,\\"history\\":[m.uci() for m in board.move_stack],\\n", "                    \\"samples\\":samples,\\"generator_step\\":state[\\"step\\"],\\"opponent\\":opponent_info,\\n", "                    \\"game_number\\":state[\\"selfplay_games\\"]+1,\\"rng\\":pickle.dumps(self.ns[\\"rng_state\\"](state[\\"sampler\\"])).hex()})\\n", "            outcome = board.outcome(claim_draw=True)\\n", "            records = outcome_targets(samples,outcome.winner) if outcome is not None else []\\n", "            # Split by canonical board family even for history-rich self-play.\\n", "            records = [r for r in records if self.ns[\\"partition\\"](replay_board(r)) == \\"train\\"]\\n", "            if records:\\n", "                state[\\"replay\\"] = (state[\\"replay\\"] + records)[-self.config[\\"replay_size\\"]:]\\n", "            state[\\"selfplay_games\\"] += 1\\n", "            state[\\"round_games\\"] += 1\\n", "            game_dir = state[\\"directory\\"] / \\"selfplay-games\\"\\n", "            game_dir.mkdir(exist_ok=True)\\n", "            self.ns[\\"atomic_json\\"](game_dir / f\\"{state[\'selfplay_games\']:06d}.json\\",{\\n", "                \\"status\\":\\"complete\\" if outcome is not None else \\"unresolved\\", \\"result\\":outcome.result() if outcome else None,\\n", "                \\"reason\\":outcome.termination.name if outcome else \\"safety_cap\\", \\"moves\\":[m.uci() for m in board.move_stack],\\n", "                \\"generator_step\\":state[\\"step\\"],\\"train_records\\":len(records)})\\n", "            self.save(state)\\n", "            if path.exists():path.unlink()\\n", "            print(\\"selfplay\\",state[\\"selfplay_games\\"],\\"replay\\",len(state[\\"replay\\"]),\\"result\\",outcome.result() if outcome else \\"unresolved\\",flush=True)\\n", "        state[\\"collected_round\\"] = state[\\"round\\"]\\n", "        return True\\n", "\\n", "    def validate(self, state):\\n", "        policy_sum,value_sum,squared,correct = 0.,0.,0.,0\\n", "        target_values = []\\n", "        with self.ns[\\"inference_mode\\"](state[\\"model\\"]):\\n", "            for offset in range(0,len(self.validation),self.ns[\\"BATCH_SIZE\\"]):\\n", "                records = [self.source_record(i) for i in self.validation[offset:offset+self.ns[\\"BATCH_SIZE\\"]]]\\n", "                boards = [replay_board(r) for r in records]\\n", "                features = torch.tensor(np.stack([self.ns[\\"encode_board\\"](b) for b in boards]),device=self.device)\\n", "                logits,value = state[\\"model\\"](features)\\n", "                logits = self.ns[\\"mask_logits\\"](logits,boards)\\n", "                width = max(len(r[\\"moves\\"]) for r in records)\\n", "                choices = np.zeros((len(records),width),np.int64)\\n", "                probabilities = np.zeros((len(records),width),np.float32)\\n", "                for row,(record,board) in enumerate(zip(records,boards)):\\n", "                    ids = [self.ns[\\"move_index\\"](chess.Move.from_uci(m),board.turn) for m in record[\\"moves\\"]]\\n", "                    choices[row] = ids[0];choices[row,:len(ids)] = ids\\n", "                    probabilities[row,:len(ids)] = record[\\"probs\\"]\\n", "                targets = torch.tensor([r[\\"value\\"] for r in records],device=self.device,dtype=torch.float32)\\n", "                policy_sum += float(soft_policy_loss(logits,choices,probabilities))*len(records)\\n", "                value_sum += float(F.binary_cross_entropy_with_logits(value,targets,reduction=\\"sum\\"))\\n", "                squared += float(((value.sigmoid()-targets)**2).sum())\\n", "                correct += int((logits.argmax(1)==torch.tensor(choices[:,0],device=self.device)).sum())\\n", "                target_values.extend(targets.cpu().tolist())\\n", "        baseline = float(np.var(target_values))\\n", "        metrics = {\\"legal_move_match\\":correct/len(self.validation),\\"value_mse\\":squared/len(self.validation),\\n", "                   \\"validation_loss\\":(policy_sum+self.config[\\"value_weight\\"]*value_sum)/len(self.validation),\\n", "                   \\"value_skill_vs_validation_mean\\":1-squared/len(self.validation)/baseline if baseline>1e-10 else None,\\n", "                   \\"value_source\\":\\"fixed_pinned_teacher_wdl\\",\\"n\\":len(self.validation)}\\n", "        loss = metrics[\\"validation_loss\\"]\\n", "        if state[\\"best_loss\\"] is None or loss < state[\\"best_loss\\"]-self.config[\\"min_delta\\"]:\\n", "            state[\\"best_loss\\"],state[\\"stale\\"] = loss,0\\n", "            self.ns[\\"save_checkpoint\\"](state[\\"directory\\"] / \\"main.best.pt\\",{\\n", "                \\"identity\\":state[\\"identity\\"],\\"model\\":state[\\"model\\"].state_dict(),\\"step\\":state[\\"step\\"],\\"metrics\\":metrics})\\n", "        else:\\n", "            state[\\"stale\\"] += 1\\n", "        self.ns[\\"atomic_json\\"](state[\\"directory\\"] / \\"validation-latest.json\\",dict(metrics,step=state[\\"step\\"],phase=state[\\"phase\\"]))\\n", "        print(state[\\"directory\\"].name,\\"round\\",state[\\"round\\"],\\"step\\",state[\\"step\\"],\\"phase\\",state[\\"phase\\"],metrics,flush=True)\\n", "        return metrics\\n", "\\n", "    def arena(self, branch, state, deadline):\\n", "        directory = state[\\"directory\\"] / \\"arenas\\" / f\\"step-{state[\'step\']:08d}\\"\\n", "        directory.mkdir(parents=True,exist_ok=True)\\n", "        records_path = directory / \\"records.json\\"\\n", "        records = json.loads(records_path.read_text()) if records_path.exists() else []\\n", "        opponent = clone_checkpoint_model(self.ns,state,state[\\"directory\\"] / \\"main.champion.pt\\")\\n", "        def mover(model):\\n", "            if branch != \\"v5\\":return self.ns[\\"model_opponent\\"](model,False)\\n", "            def play(board,clock,increment):\\n", "                choices,probs,_ = search_target(self.ns,model,board,self.config[\\"simulations\\"],state[\\"sampler\\"])\\n", "                return self.ns[\\"index_to_move\\"](choices[int(probs.argmax())],board)\\n", "            return play\\n", "        current,old = mover(state[\\"model\\"]),mover(opponent)\\n", "        previous_dir = self.ns[\\"RUN_DIR\\"]\\n", "        self.ns[\\"RUN_DIR\\"] = directory\\n", "        try:\\n", "            for pair,opening in enumerate(self.ns[\\"make_opening_pairs\\"](self.config[\\"arena_pairs\\"],seed=79)):\\n", "                for color in (\\"white\\",\\"black\\"):\\n", "                    if time.monotonic()>=deadline:return False\\n", "                    game_id = f\\"{pair}-{color}\\"\\n", "                    if any(r[\\"id\\"]==game_id for r in records):continue\\n", "                    white,black = (current,old) if color==\\"white\\" else (old,current)\\n", "                    game = self.ns[\\"play_game\\"](white,black,opening,game_id,initial_clock=120,increment=1,max_plies=500)\\n", "                    score = game[\\"score\\"]\\n", "                    if score is not None and color==\\"black\\":score=1-score\\n", "                    records.append({\\"id\\":game_id,\\"pair\\":pair,\\"color\\":color,\\"score\\":score,\\"reason\\":game[\\"reason\\"]})\\n", "                    self.ns[\\"atomic_json\\"](records_path,records)\\n", "            summary = self.ns[\\"strength_summary\\"](records)\\n", "            summary.update(candidate_step=state[\\"step\\"],opponent_step=state[\\"champion_step\\"],\\n", "                           simulations=self.config[\\"simulations\\"] if branch==\\"v5\\" else 0,\\n", "                           promotion_rule=\\"At least 62.5% paired score; engineering gate, not statistical proof\\")\\n", "            promoted = summary[\\"complete_pairs\\"]==self.config[\\"arena_pairs\\"] and summary[\\"score\\"]>=.625\\n", "            summary[\\"promoted\\"] = bool(promoted)\\n", "            self.ns[\\"atomic_json\\"](directory / \\"summary.json\\",summary)\\n", "            if promoted:\\n", "                self.ns[\\"save_checkpoint\\"](state[\\"directory\\"] / \\"main.champion.pt\\",{\\n", "                    \\"identity\\":state[\\"identity\\"],\\"model\\":state[\\"model\\"].state_dict(),\\"step\\":state[\\"step\\"]})\\n", "                state[\\"champion_step\\"],state[\\"stale\\"] = state[\\"step\\"],0\\n", "            state[\\"arena_pending\\"] = False\\n", "            print(branch,\\"arena\\",summary,flush=True)\\n", "            return True\\n", "        finally:\\n", "            self.ns[\\"RUN_DIR\\"] = previous_dir\\n", "\\n", "    def run(self, minutes):\\n", "        deadline = time.monotonic()+minutes*60\\n", "        ns = self.ns\\n", "        try:\\n", "            # Freeze the whole validation label set before the first update.\\n", "            self.relabel(self.validation,deadline,limit=len(self.validation))\\n", "            if any(self.teacher.get(chess.Board(str(self.data[\\"fen\\"][i])).fen(en_passant=\\"fen\\")) is None for i in self.validation):\\n", "                print(\\"Validation labelling paused; rerun to finish before training\\",flush=True)\\n", "                return\\n", "            while time.monotonic()<deadline and any(s[\\"status\\"]==\\"active\\" for s in self.states.values()):\\n", "                for branch,state in self.states.items():\\n", "                    if state[\\"status\\"]!=\\"active\\" or time.monotonic()>=deadline:continue\\n", "                    if state[\\"round\\"] > min(s[\\"round\\"] for s in self.states.values() if s[\\"status\\"]==\\"active\\"):\\n", "                        continue\\n", "                    ns[\\"restore_rng\\"](state[\\"rng\\"],state[\\"sampler\\"])\\n", "                    ns[\\"RUN_DIR\\"] = state[\\"directory\\"]\\n", "                    if state[\\"arena_pending\\"]:\\n", "                        if not self.arena(branch,state,deadline):return\\n", "                    old_phase = state[\\"phase\\"]\\n", "                    state[\\"phase\\"] = \\"foundation\\" if state[\\"round\\"]<self.config[\\"foundation_rounds\\"] else \\"general\\" if state[\\"round\\"]<self.config[\\"pretrain_rounds\\"] else \\"consolidation\\"\\n", "                    if state[\\"phase\\"] != old_phase:state[\\"stale\\"] = 0\\n", "                    # The deterministic shard seed is branch independent. Both branches\\n", "                    # see the same supervised curriculum; shards rotate every round.\\n", "                    pool = stratified_pool(self.groups,self.config[\\"pool_size\\"],np.random.RandomState(1000+state[\\"round\\"]),state[\\"phase\\"])\\n", "                    shard_key = hashlib.sha256(pool.tobytes()).hexdigest()\\n", "                    if not self.teacher.stage_done(shard_key):\\n", "                        self.relabel(pool,deadline)\\n", "                        if time.monotonic()<deadline:self.teacher.mark_stage(shard_key)\\n", "                    if time.monotonic()>=deadline:break\\n", "                    if branch==\\"v5\\" and state[\\"round\\"]>=self.config[\\"pretrain_rounds\\"]:\\n", "                        if not state[\\"search_gate\\"] and state[\\"round\\"]%self.config[\\"gate_interval\\"]==0:\\n", "                            state[\\"search_gate\\"] = self.search_gate(state)\\n", "                        if state[\\"search_gate\\"] and state[\\"collected_round\\"] != state[\\"round\\"]:\\n", "                            if not self.collect_selfplay(state,deadline):break\\n", "                    # Frozen cache is shared between branches with the same shard.\\n", "                    key = hashlib.sha256(pool.tobytes()).hexdigest()\\n", "                    if self.cache_key!=key:\\n", "                        cache_root = ns[\\"ROOT\\"] / \\"perception-cache\\" / ns[\\"RUN_ID\\"]\\n", "                        tag = \\"shared-\\"+self.cache_scope+\\"-\\"+key[:12]\\n", "                        if cache_root.exists():\\n", "                            for old in cache_root.glob(\\"shared-\\"+self.cache_scope+\\"-*\\"):\\n", "                                if old.name != tag:shutil.rmtree(old)\\n", "                        self.cache = ns[\\"cache_perception\\"](state[\\"model\\"],tag,pool,key+self.cache_identity)\\n", "                        self.cache_key = key\\n", "                    preflight_path = state[\\"directory\\"] / \\"learning-preflight.json\\"\\n", "                    if state[\\"step\\"] == 0:\\n", "                        if preflight_path.exists():\\n", "                            if not json.loads(preflight_path.read_text())[\\"passed\\"] and not self.smoke and branch != \\"v55\\":\\n", "                                raise RuntimeError(\\"Saved learning preflight failed; long training remains stopped\\")\\n", "                        else:\\n", "                            ns[\\"learning_preflight\\"](state[\\"model\\"],self.cache)\\n", "                    last_save = time.monotonic()\\n", "                    while state[\\"step\\"] < (state[\\"round\\"]+1)*self.round_size:\\n", "                        ns[\\"check_stop\\"]()\\n", "                        if time.monotonic()>=deadline:\\n", "                            self.save(state);return\\n", "                        use_replay = branch==\\"v5\\" and bool(state[\\"replay\\"]) and state[\\"sampler\\"].rand()<self.config[\\"selfplay_fraction\\"]\\n", "                        if use_replay:\\n", "                            selected = state[\\"sampler\\"].choice(len(state[\\"replay\\"]),min(ns[\\"BATCH_SIZE\\"],len(state[\\"replay\\"])),replace=False)\\n", "                            update = self.loss_batch(state,[state[\\"replay\\"][i] for i in selected])\\n", "                        else:\\n", "                            selected = state[\\"sampler\\"].choice(pool,min(ns[\\"BATCH_SIZE\\"],len(pool)),replace=False)\\n", "                            update = self.loss_batch(state,[self.source_record(i) for i in selected],self.cache)\\n", "                        if update is None:continue\\n", "                        state[\\"step\\"]+=1\\n", "                        if time.monotonic()-last_save>=60:\\n", "                            self.save(state);last_save=time.monotonic()\\n", "                    state[\\"round\\"]+=1\\n", "                    self.validate(state)\\n", "                    if not self.smoke and state[\\"round\\"]%self.config[\\"arena_every\\"]==0:\\n", "                        state[\\"arena_pending\\"] = True\\n", "                        self.save(state)\\n", "                        if not self.arena(branch,state,deadline):return\\n", "                    # A phase gets its own patience. Self-play gets a complete patience\\n", "                    # window after the search gate succeeds, then returns control.\\n", "                    if state[\\"round\\"]>=self.config[\\"pretrain_rounds\\"] and state[\\"stale\\"]>=self.config[\\"patience\\"]:\\n", "                        state[\\"status\\"]=\\"plateau\\"\\n", "                    if self.smoke and state[\\"round\\"]>=self.config[\\"smoke_rounds\\"]:state[\\"status\\"]=\\"smoke_complete\\"\\n", "                    self.save(state)\\n", "                    state[\\"rng\\"] = ns[\\"rng_state\\"](state[\\"sampler\\"])\\n", "        except (KeyboardInterrupt, ns[\\"RunPaused\\"]) as exc:\\n", "            print(\\"Paused:\\",str(exc),flush=True)\\n", "        finally:\\n", "            # Each branch retains its own RNG stream, including dropout/CUDA RNG.\\n", "            for state in self.states.values():\\n", "                if ns.get(\\"RUN_DIR\\")==state[\\"directory\\"]:\\n", "                    state[\\"rng\\"] = ns[\\"rng_state\\"](state[\\"sampler\\"])\\n", "            for state in self.states.values():\\n", "                ns[\\"restore_rng\\"](state[\\"rng\\"],state[\\"sampler\\"])\\n", "                self.save(state)\\n", "            ns[\\"RUN_DIR\\"] = self.base\\n", "\\n", "\\n", "def advanced_config(branches, smoke):\\n", "    def integer(name,default):return int(os.environ.get(name,str(default)))\\n", "    config = {\\"branches\\":list(branches),\\"round_size\\":2 if smoke else integer(\\"FLY_CHESS_ROUND_STEPS\\",500),\\n", "        \\"pool_size\\":64 if smoke else integer(\\"FLY_CHESS_TRAIN_POOL\\",32768),\\n", "        \\"foundation_rounds\\":1 if smoke else 2,\\"pretrain_rounds\\":1 if smoke else 6,\\n", "        \\"validation_size\\":32 if smoke else 512,\\"validation_relabel\\":2 if smoke else 32,\\n", "        \\"teacher_nodes\\":100 if smoke else integer(\\"FLY_CHESS_TEACHER_NODES\\",50000),\\n", "        \\"relabel_per_round\\":4 if smoke else integer(\\"FLY_CHESS_RELABEL_PER_ROUND\\",256),\\n", "        \\"lr\\":float(os.environ.get(\\"FLY_CHESS_LR\\",\\"0.0003\\")),\\"value_weight\\":2.0,\\n", "        \\"patience\\":integer(\\"FLY_CHESS_PATIENCE\\",10),\\"min_delta\\":.002,\\n", "        \\"simulations\\":4 if smoke else integer(\\"FLY_CHESS_SIMULATIONS\\",64),\\n", "        \\"gate_positions\\":2 if smoke else 32,\\"gate_interval\\":1 if smoke else 2,\\n", "        \\"games_per_round\\":1 if smoke else integer(\\"FLY_CHESS_SELFPLAY_GAMES\\",8),\\n", "        \\"game_cap\\":16 if smoke else 500,\\"replay_size\\":256 if smoke else 50000,\\n", "        \\"selfplay_fraction\\":.35,\\"smoke_rounds\\":2}\\n", "    config.update(arena_every=4,arena_pairs=4)\\n", "    if any(config[k]<=0 for k in (\\"round_size\\",\\"pool_size\\",\\"teacher_nodes\\",\\"patience\\",\\"simulations\\",\\"games_per_round\\")):\\n", "        raise ValueError(\\"Training configuration values must be positive\\")\\n", "    return config\\n", "\\n", "\\n", "def benchmark_advanced(trainer, minutes=15, pairs=20):\\n", "    \\"\\"\\"Paired, immutable, interleaved network/search/helper comparisons.\\"\\"\\"\\n", "    ns = trainer.ns\\n", "    deadline = time.monotonic()+minutes*60\\n", "    base = ns[\\"RUN_DIR\\"]\\n", "    entries = []\\n", "    snapshots = []\\n", "    for branch,state in trainer.states.items():\\n", "        checkpoint = state[\\"directory\\"] / \\"main.best.pt\\"\\n", "        if not checkpoint.exists():checkpoint=state[\\"directory\\"] / \\"main.pt\\"\\n", "        snapshot = clone_checkpoint_model(ns,state,checkpoint)\\n", "        snapshots.append(snapshot)\\n", "        fingerprint = ns[\\"sha256\\"](checkpoint)\\n", "        modes = [\\"network_search\\",\\"network_raw\\"] if branch == \\"v5\\" else [\\"network_raw\\"]\\n", "        if branch == \\"v55\\":modes=[\\"fly_ranked\\",\\"ranker_top1\\",\\"no_candidates\\",\\"reverse_candidates\\",\\"no_senses\\"]\\n", "        for mode in modes:\\n", "            def agent(board,clock,increment,mode=mode,snapshot=snapshot,state=state):\\n", "                if mode == \\"ranker_top1\\":\\n", "                    return chess.Move.from_uci(ns[\\"ranked_lookahead\\"](board.fen(en_passant=\\"fen\\"))[0][0])\\n", "                if mode == \\"network_search\\":\\n", "                    choices,probabilities,_ = search_target(ns,snapshot,board,trainer.config[\\"simulations\\"],state[\\"sampler\\"])\\n", "                    return ns[\\"index_to_move\\"](choices[int(probabilities.argmax())],board)\\n", "                if mode in (\\"no_candidates\\",\\"reverse_candidates\\",\\"no_senses\\"):\\n", "                    features = torch.tensor(ns[\\"encode_board\\"](board)[None],device=ns[\\"DEVICE\\"])\\n", "                    with ns[\\"inference_mode\\"](snapshot):\\n", "                        logits,_ = snapshot(features,intervention=mode)\\n", "                    return ns[\\"index_to_move\\"](int(ns[\\"mask_logits\\"](logits,[board]).argmax(1)[0]),board)\\n", "                return ns[\\"choose_move\\"](snapshot,board,False)\\n", "            entries.append((branch,state,fingerprint,mode,agent))\\n", "    with ns[\\"open_stockfish\\"]() as engine:\\n", "        minimum,maximum = engine.options[\\"UCI_Elo\\"].min,engine.options[\\"UCI_Elo\\"].max\\n", "        ratings = [minimum] if trainer.smoke else sorted(set([minimum,min(maximum,1600),min(maximum,2000)]))\\n", "        try:\\n", "            openings = ns[\\"make_opening_pairs\\"](1 if trainer.smoke else pairs,seed=113)\\n", "            for pair,opening in enumerate(openings):\\n", "                for rating in ratings:\\n", "                    for branch,state,fingerprint,mode,agent in entries:\\n", "                        directory=state[\\"directory\\"] / \\"strength\\" / fingerprint[:16] / mode / f\\"sf-{rating}\\"\\n", "                        directory.mkdir(parents=True,exist_ok=True)\\n", "                        config={\\"checkpoint_sha256\\":fingerprint,\\"engine_sha256\\":ns[\\"STOCKFISH_BINARY_SHA\\"],\\n", "                                \\"reference_elo\\":rating,\\"clock\\":120,\\"increment\\":1,\\"opening_seed\\":113,\\n", "                                \\"simulations\\":trainer.config[\\"simulations\\"] if mode==\\"network_search\\" else 0,\\n", "                                \\"assist_mode\\":\\"material_minimax_depth2_top4\\" if branch==\\"v55\\" else \\"none\\", \\"mode\\":mode,\\n", "                                \\"max_plies\\":12 if trainer.smoke else 600}\\n", "                        config_path=directory / \\"config.json\\"\\n", "                        if config_path.exists() and json.loads(config_path.read_text())!=config:\\n", "                            raise ValueError(\\"Benchmark configuration changed\\")\\n", "                        ns[\\"atomic_json\\"](config_path,config)\\n", "                        record_path=directory / \\"records.json\\"\\n", "                        records=json.loads(record_path.read_text()) if record_path.exists() else []\\n", "                        ns[\\"RUN_DIR\\"]=directory\\n", "                        engine.configure({\\"Threads\\":1,\\"Hash\\":64,\\"Skill Level\\":20,\\"UCI_LimitStrength\\":True,\\"UCI_Elo\\":rating})\\n", "                        def reference(board,clock,increment):\\n", "                            return engine.play(board,chess.engine.Limit(white_clock=clock if board.turn else 120,\\n", "                                black_clock=clock if not board.turn else 120,white_inc=1,black_inc=1),game=game_token).move\\n", "                        for color in (\\"white\\",\\"black\\"):\\n", "                            game_id=f\\"{pair}-{color}\\"\\n", "                            if any(r[\\"id\\"]==game_id for r in records):continue\\n", "                            if time.monotonic()>=deadline:return\\n", "                            game_token = object()\\n", "                            white,black=(agent,reference) if color==\\"white\\" else (reference,agent)\\n", "                            game=ns[\\"play_game\\"](white,black,opening,game_id,initial_clock=120,increment=1,max_plies=config[\\"max_plies\\"])\\n", "                            score=game[\\"score\\"]\\n", "                            if score is not None and color==\\"black\\":score=1-score\\n", "                            records.append({\\"id\\":game_id,\\"pair\\":pair,\\"color\\":color,\\"score\\":score,\\"reason\\":game[\\"reason\\"]})\\n", "                            ns[\\"atomic_json\\"](record_path,records)\\n", "                            summary=ns[\\"strength_summary\\"](records,rating,\\"Stockfish UCI_Elo benchmark at 120+1\\")\\n", "                            summary.update(checkpoint_sha256=fingerprint,configuration=config,target_pairs=len(openings))\\n", "                            ns[\\"atomic_json\\"](directory / \\"summary.json\\",summary)\\n", "                            print(branch,mode,\\"benchmark\\",rating,game_id,score,game[\\"reason\\"],flush=True)\\n", "        finally:\\n", "            ns[\\"RUN_DIR\\"]=base"]}, {"cell_type": "code", "execution_count": null, "metadata": {"tags": ["brain"]}, "outputs": [], "source": ["EDGE_MIN_SYNAPSES = 3  # picked below in the Phase-0 benchmark; kept \\u2265 here so later cells still run standalone\\n", "\\n", "\\n", "def build_signed_transpose(W, signs, min_synapses):\\n", "    \\"\\"\\"Wt[post, pre] = sign(pre) * synapse_count, thresholded and row-normalised (row = post).\\"\\"\\"\\n", "    keep = W.data >= min_synapses\\n", "    pre = np.repeat(np.arange(W.shape[0]), np.diff(W.indptr))[keep]\\n", "    post = W.indices[keep]\\n", "    weight = (W.data[keep] * signs[pre]).astype(np.float32)\\n", "    row_norm = np.bincount(post, weights=np.abs(weight), minlength=W.shape[0])\\n", "    row_norm = np.maximum(row_norm, 1.0)\\n", "    weight = (weight / row_norm[post]).astype(np.float32)\\n", "    indices = torch.tensor(np.stack([post, pre]), dtype=torch.long)\\n", "    values = torch.tensor(weight)\\n", "    return torch.sparse_coo_tensor(indices, values, (W.shape[0], W.shape[0])).coalesce(), int(keep.sum())\\n", "\\n", "\\n", "Wt, n_edges_kept = build_signed_transpose(W, base_signs, EDGE_MIN_SYNAPSES)\\n", "Wt = Wt.to(DEVICE)\\n", "print(f\\"kept {n_edges_kept:,} / {W.nnz:,} edges at \\u2265{EDGE_MIN_SYNAPSES} synapses \\"\\n", "      f\\"({n_edges_kept / W.nnz:.0%}), {W.data[W.data >= EDGE_MIN_SYNAPSES].sum() / W.data.sum():.0%} of synapse mass\\")\\n", "\\n", "\\n", "class FlyBrain(nn.Module):\\n", "    \\"\\"\\"Frozen, batched, differentiable graded-rate model of the MaleCNS connectome.\\"\\"\\"\\n", "\\n", "    def __init__(self, Wt, gain, alpha):\\n", "        super().__init__()\\n", "        self.register_buffer(\\"Wt\\", Wt, persistent=False)\\n", "        self.n = Wt.shape[0]\\n", "        self.gain = gain\\n", "        self.alpha = alpha\\n", "\\n", "    def step(self, r, current):\\n", "        total_input = torch.sparse.mm(self.Wt, r)\\n", "        target = torch.clamp(self.gain * total_input + current, 0.0, 1.0)\\n", "        return (1 - self.alpha) * r + self.alpha * target\\n", "\\n", "    def run(self, r0, current, steps):\\n", "        r = r0\\n", "        for _ in range(steps):\\n", "            r = self.step(r, current)\\n", "        return r\\n", "\\n", "    def run_clamped(self, r0, current, active_mask, steps):\\n", "        \\"\\"\\"Only `active_mask` neurons update; everyone else is held at r0 (\\u00a7 act-phase).\\"\\"\\"\\n", "        r = r0\\n", "        frozen = r0\\n", "        mask = active_mask.view(-1, 1)\\n", "        for _ in range(steps):\\n", "            r = torch.where(mask, self.step(r, current), frozen)\\n", "        return r\\n", "\\n", "    def active_blocks(self, active_idx):\\n", "        active_idx = np.asarray(active_idx)\\n", "        coordinate = self.Wt.coalesce()\\n", "        ij = coordinate.indices().cpu().numpy()\\n", "        vals = coordinate.values().cpu().numpy()\\n", "        lookup = np.full(self.n, -1, dtype=np.int64)\\n", "        lookup[active_idx] = np.arange(len(active_idx))\\n", "        rows, cols = lookup[ij[0]], lookup[ij[1]]\\n", "        internal = (rows >= 0) & (cols >= 0)\\n", "        external = (rows >= 0) & (cols < 0)\\n", "        def matrix(mask, col, width):\\n", "            return torch.sparse_coo_tensor(torch.tensor(np.stack([rows[mask], col[mask]])),\\n", "                   torch.tensor(vals[mask]), (len(active_idx), width)).coalesce().to(self.Wt.device)\\n", "        return matrix(internal, cols, len(active_idx)), matrix(external, ij[1], self.n)\\n", "\\n", "    @torch.amp.custom_fwd(device_type=\\"cuda\\", cast_inputs=torch.float32)\\n", "    def run_active_prepared(self, initial, background, current_active, internal, steps):\\n", "        r = initial\\n", "        for _ in range(steps):\\n", "            target = torch.clamp(self.gain * (torch.sparse.mm(internal, r) + background)\\n", "                                 + current_active, 0, 1)\\n", "            r = (1 - self.alpha) * r + self.alpha * target\\n", "        return r\\n", "\\n", "    def run_active(self, r0, current_active, active_idx, blocks, steps):\\n", "        internal, external = blocks\\n", "        r = r0[active_idx]\\n", "        background = torch.sparse.mm(external, r0)\\n", "        for _ in range(steps):\\n", "            target = torch.clamp(self.gain * (torch.sparse.mm(internal, r) + background)\\n", "                                 + current_active, 0, 1)\\n", "            r = (1 - self.alpha) * r + self.alpha * target\\n", "        return r"]}, {"cell_type": "code", "execution_count": null, "metadata": {"tags": ["injection"]}, "outputs": [], "source": ["def build_injection_map(sensory_idx, n_features, seed=0):\\n", "    \\"\\"\\"A fixed, disjoint feature -> sensory-neuron-group assignment (~14 neurons/feature).\\"\\"\\"\\n", "    rng = np.random.RandomState(seed)\\n", "    order = rng.permutation(sensory_idx)\\n", "    groups = [order[i % len(order):i % len(order) + 1] for i in range(n_features)] if len(order) < n_features else np.array_split(order, n_features)\\n", "    rows = np.concatenate(groups)\\n", "    cols = np.concatenate([np.full(len(g), i) for i, g in enumerate(groups)])\\n", "    indices = torch.tensor(np.stack([rows, cols]), dtype=torch.long)\\n", "    values = torch.ones(len(rows))\\n", "    return torch.sparse_coo_tensor(indices, values, (N_NEURONS, n_features)).coalesce().to(DEVICE)\\n", "\\n", "\\n", "INJECT_MAP = build_injection_map(SENSORY_IDX, N_FEATURES)\\n", "INJECT_AMPLITUDE = 1.0 if USE_REAL_GRAPH else 0.3\\n", "\\n", "\\n", "def sensory_current(features_b_f):\\n", "    \\"\\"\\"(B, 780) board features -> (n, B) injected current.\\"\\"\\"\\n", "    return INJECT_AMPLITUDE * torch.sparse.mm(INJECT_MAP, features_b_f.t())\\n", "\\n", "\\n", "def empty_cache():\\n", "    if DEVICE.type == \\"cuda\\":\\n", "        torch.cuda.empty_cache()\\n", "\\n", "\\n", "def perceive_chunked(brain_, features_np, T_p, batch=128, keep_idx=None):\\n", "    \\"\\"\\"Runs perception in GPU-sized chunks and returns a CPU array \\u2014 the (n, N) state for N boards\\n", "    at once does not fit in 6 GB much past a few hundred boards. `keep_idx` restricts the returned\\n", "    rows (e.g. to a relay candidate) to save host memory too.\\"\\"\\"\\n", "    chunks = []\\n", "    for start in range(0, len(features_np), batch):\\n", "        feats = torch.tensor(features_np[start:start + batch], device=DEVICE)\\n", "        with torch.no_grad():\\n", "            r = brain_.run(torch.zeros(brain_.n, feats.shape[0], device=DEVICE), sensory_current(feats), T_p)\\n", "        chunks.append((r if keep_idx is None else r[keep_idx]).cpu().numpy())\\n", "        del feats, r\\n", "    empty_cache()\\n", "    return np.concatenate(chunks, axis=1)\\n", "\\n", "\\n", "T_P_GRID = ([15, 25, 40] if MODE == \\"full\\" else [10, 15]) if USE_REAL_GRAPH else [3, 5]"]}, {"cell_type": "code", "execution_count": null, "metadata": {"tags": ["models"]}, "outputs": [], "source": ["def make_relay_pool(relay_idx, injection_map):\\n", "    mapping = injection_map.coalesce().indices().cpu().numpy()\\n", "    lookup = np.full(injection_map.shape[0], -1, dtype=np.int64)\\n", "    lookup[np.asarray(relay_idx)] = np.arange(len(relay_idx))\\n", "    selected = lookup[mapping[0]] >= 0\\n", "    rows, cols = mapping[1, selected], lookup[mapping[0, selected]]\\n", "    counts = np.bincount(rows, minlength=injection_map.shape[1])\\n", "    if (counts > 0).mean() < .95:\\n", "        return None\\n", "    values = (1/np.maximum(counts[rows],1)).astype(np.float32)\\n", "    return torch.sparse_coo_tensor(torch.tensor(np.stack([rows,cols])), torch.tensor(values),\\n", "        (injection_map.shape[1],len(relay_idx))).coalesce().to(injection_map.device)\\n", "\\n", "\\n", "class CortexGraft(nn.Module):\\n", "    def __init__(self, relay_idx, premotor_idx, d_model=64, n_latents=48, n_layers=2, n_heads=4, relay_pool=None):\\n", "        super().__init__()\\n", "        self.register_buffer(\\"relay_idx\\", torch.tensor(relay_idx, dtype=torch.long), persistent=False)\\n", "        self.register_buffer(\\"premotor_idx\\", torch.tensor(premotor_idx, dtype=torch.long), persistent=False)\\n", "        self.register_buffer(\\"relay_pool\\", torch.empty(0) if relay_pool is None else relay_pool)\\n", "        n_relay = len(relay_idx) if relay_pool is None else relay_pool.shape[0]\\n", "        n_premotor = len(premotor_idx)\\n", "\\n", "        self.register_buffer(\\"relay_mean\\", torch.zeros(n_relay))\\n", "        self.register_buffer(\\"relay_scale\\", torch.ones(n_relay))\\n", "        self.square_tokens = relay_pool is not None and n_relay >= 780\\n", "        if self.square_tokens:\\n", "            self.piece_embed = nn.Linear(12, d_model)\\n", "            self.side_embed = nn.Linear(n_relay - 768, d_model)\\n", "        else:\\n", "            self.rate_embed = nn.Linear(1, d_model)\\n", "        self.token_norm = nn.LayerNorm(d_model)\\n", "        self.relay_summary = nn.Sequential(nn.Linear(n_relay, 4*d_model), nn.LayerNorm(4*d_model), nn.GELU(), nn.Dropout(0.2))\\n", "        self.latent_from_summary = nn.Linear(4*d_model, n_latents*d_model)\\n", "        self.premotor_from_summary = nn.Linear(4*d_model, n_premotor)\\n", "        self.n_latents, self.d_model = n_latents, d_model\\n", "        nn.init.normal_(self.premotor_from_summary.weight, std=0.001)\\n", "        nn.init.zeros_(self.premotor_from_summary.bias)\\n", "        self.relay_embed = nn.Embedding(64 if self.square_tokens else n_relay, d_model)\\n", "        self.premotor_query = nn.Embedding(n_premotor, d_model)\\n", "        self.latents = nn.Parameter(torch.randn(n_latents, d_model) * 0.02)\\n", "\\n", "        self.encode_in = nn.MultiheadAttention(d_model, n_heads, batch_first=True, dropout=0.1)\\n", "        self.self_layers = nn.ModuleList(\\n", "            nn.MultiheadAttention(d_model, n_heads, batch_first=True, dropout=0.1) for _ in range(n_layers)\\n", "        )\\n", "        self.norms = nn.ModuleList(nn.LayerNorm(d_model) for _ in range(n_layers + 1))\\n", "        self.decode_out = nn.MultiheadAttention(d_model, n_heads, batch_first=True, dropout=0.1)\\n", "        self.output_norm = nn.LayerNorm(d_model)\\n", "        self.ff_layers = nn.ModuleList(nn.Sequential(nn.Linear(d_model, 2*d_model), nn.GELU(),\\n", "            nn.Linear(2*d_model, d_model), nn.Dropout(0.1)) for _ in range(n_layers))\\n", "        self.ff_norms = nn.ModuleList(nn.LayerNorm(d_model) for _ in range(n_layers))\\n", "        self.current_head = nn.Linear(d_model, 1)\\n", "        nn.init.normal_(self.current_head.weight, std=0.01)\\n", "        nn.init.constant_(self.current_head.bias, 0.1)\\n", "\\n", "    @torch.amp.custom_fwd(device_type=\\"cuda\\", cast_inputs=torch.float32)\\n", "    def pool_relay(self, rates):\\n", "        return torch.sparse.mm(self.relay_pool, rates.t()).t() if self.relay_pool.numel() else rates\\n", "\\n", "    def forward(self, relay_rate):\\n", "        \\"\\"\\"relay_rate: (B, n_relay) -> premotor current (B, n_premotor).\\"\\"\\"\\n", "        relay_rate = self.pool_relay(relay_rate)\\n", "        B = relay_rate.shape[0]\\n", "        rates = ((relay_rate - self.relay_mean) / self.relay_scale).clamp(-10, 10)\\n", "        if self.square_tokens:\\n", "            # These channels are pooled fly rates from the fixed sensory input groups.\\n", "            squares = rates[:, :768].reshape(B, 12, 64).transpose(1, 2)\\n", "            tokens = self.token_norm(self.relay_embed.weight.unsqueeze(0) + self.piece_embed(squares))\\n", "        else:\\n", "            tokens = self.token_norm(self.relay_embed.weight.unsqueeze(0) + self.rate_embed(rates.unsqueeze(-1)))\\n", "        summary = self.relay_summary(rates)\\n", "        latents = self.latents.unsqueeze(0) + self.latent_from_summary(summary).view(B, self.n_latents, self.d_model)\\n", "        if self.square_tokens:\\n", "            latents = latents + self.side_embed(rates[:, 768:]).unsqueeze(1)\\n", "        latents = self.norms[0](latents + self.encode_in(latents, tokens, tokens, need_weights=False)[0])\\n", "        for layer, norm, ff, ff_norm in zip(self.self_layers, self.norms[1:], self.ff_layers, self.ff_norms):\\n", "            latents = norm(latents + layer(latents, latents, latents, need_weights=False)[0])\\n", "            latents = ff_norm(latents + ff(latents))\\n", "        queries = self.premotor_query.weight.unsqueeze(0).expand(B, -1, -1)\\n", "        out, _ = self.decode_out(queries, latents, latents, need_weights=False)\\n", "        out = self.output_norm(queries + out)\\n", "        return torch.tanh(self.current_head(out).squeeze(-1) + self.premotor_from_summary(summary))\\n", "\\n", "\\n", "class MoveDecoder(nn.Module):\\n", "    \\"\\"\\"Reads the fly\'s own DN+motor population; never sees the board or the relay directly.\\"\\"\\"\\n", "\\n", "    def __init__(self, n_motor, n_moves=N_MOVES, hidden=256):\\n", "        super().__init__()\\n", "        self.dropout = nn.Dropout(0.1)\\n", "        self.register_buffer(\\"motor_mean\\", torch.zeros(n_motor))\\n", "        self.register_buffer(\\"motor_scale\\", torch.ones(n_motor))\\n", "        self.policy = nn.Linear(n_motor, n_moves)\\n", "        self.value = nn.Linear(n_motor, 1)\\n", "\\n", "    def forward(self, motor_rate):\\n", "        normalized = self.dropout((motor_rate - self.motor_mean) / self.motor_scale)\\n", "        return self.policy(normalized), self.value(normalized).squeeze(-1)\\n", "\\n", "\\n", "class FlyChessModel(nn.Module):\\n", "    \\"\\"\\"board -> (fly sense) -> graft -> (fly act) -> policy logits, value. `brain` stays frozen.\\"\\"\\"\\n", "\\n", "    def __init__(self, brain, relay_idx, premotor_idx, motor_idx, active_mask, T_p, T_a,\\n", "                 current_amplitude=0.5, relay_steps=3, **graft_kw):\\n", "        super().__init__()\\n", "        self.brain = brain\\n", "        pooling = make_relay_pool(relay_idx, INJECT_MAP) if \\"INJECT_MAP\\" in globals() else None\\n", "        self.graft = CortexGraft(relay_idx, premotor_idx, relay_pool=pooling, **graft_kw)\\n", "        self.decoder = MoveDecoder(len(motor_idx))\\n", "        self.register_buffer(\\"relay_idx\\", torch.tensor(relay_idx, dtype=torch.long), persistent=False)\\n", "        self.register_buffer(\\"premotor_idx\\", torch.tensor(premotor_idx, dtype=torch.long), persistent=False)\\n", "        self.register_buffer(\\"motor_idx\\", torch.tensor(motor_idx, dtype=torch.long), persistent=False)\\n", "        self.register_buffer(\\"active_mask\\", active_mask, persistent=False)\\n", "        self.T_p, self.T_a, self.current_amplitude = T_p, T_a, current_amplitude\\n", "        self.relay_steps = min(T_p, relay_steps)\\n", "        active_idx = np.flatnonzero(active_mask.cpu().numpy())\\n", "        self.register_buffer(\\"active_idx\\", torch.tensor(active_idx), persistent=False)\\n", "        self.motor_active_idx = np.searchsorted(active_idx, motor_idx)\\n", "        self.premotor_active_idx = np.searchsorted(active_idx, premotor_idx)\\n", "        self.blocks = brain.active_blocks(active_idx)\\n", "\\n", "\\n", "    @torch.amp.custom_fwd(device_type=\\"cuda\\", cast_inputs=torch.float32)\\n", "    def perceive(self, features):\\n", "        with torch.no_grad():\\n", "            current = sensory_current(features)\\n", "            state = self.brain.run(torch.zeros(self.brain.n, len(features), device=features.device),\\n", "                                   current, self.relay_steps)\\n", "            relay = state[self.relay_idx].t()\\n", "            state = self.brain.run(state, current, self.T_p - self.relay_steps)\\n", "            return (relay, state[self.active_idx].t(),\\n", "                    torch.sparse.mm(self.blocks[1], state).t())\\n", "\\n", "    def forward_perceived(self, relay, initial, background, lesion_relay=False, intervention=None):\\n", "        if intervention == \\"no_senses\\":\\n", "            relay, initial, background = torch.zeros_like(relay), torch.zeros_like(initial), torch.zeros_like(background)\\n", "        if lesion_relay:\\n", "            relay = torch.zeros_like(relay)\\n", "        if intervention == \\"relay_permute\\":\\n", "            # Permute neuron identities; this also works on singleton inference batches.\\n", "            relay = relay.roll(1, dims=1)\\n", "        premotor = self.graft(relay) * self.current_amplitude\\n", "        if intervention == \\"no_graft\\":\\n", "            premotor = torch.zeros_like(premotor)\\n", "        current = torch.zeros(len(self.active_idx), len(relay), device=relay.device)\\n", "        current[self.premotor_active_idx] = premotor.float().t()\\n", "        final = self.brain.run_active_prepared(initial.t(), background.t(), current,\\n", "                                             self.blocks[0], self.T_a)\\n", "        return self.decoder(final[self.motor_active_idx].t())\\n", "\\n", "    def forward(self, features, lesion_relay=False, intervention=None):\\n", "        return self.forward_perceived(*self.perceive(features), lesion_relay=lesion_relay, intervention=intervention)"]}, {"cell_type": "code", "execution_count": null, "metadata": {"tags": ["control_flyonly"]}, "outputs": [], "source": ["class FlyOnlyModel(nn.Module):\\n", "    \\"\\"\\"Frozen perception -> normalized linear motor readout, without a graft.\\"\\"\\"\\n", "    def __init__(self, brain, motor_idx, T_p):\\n", "        super().__init__()\\n", "        self.brain, self.T_p = brain, T_p\\n", "        self.decoder = MoveDecoder(len(motor_idx))\\n", "        self.register_buffer(\\"motor_idx\\", torch.tensor(motor_idx, dtype=torch.long), persistent=False)\\n", "\\n", "    @torch.amp.custom_fwd(device_type=\\"cuda\\", cast_inputs=torch.float32)\\n", "    def perceive(self, features):\\n", "        with torch.no_grad():\\n", "            state = self.brain.run(torch.zeros(self.brain.n, len(features), device=features.device),\\n", "                                   sensory_current(features), self.T_p)\\n", "            return (state[self.motor_idx].t(),)\\n", "\\n", "    def forward_perceived(self, motor):\\n", "        return self.decoder(motor)\\n", "\\n", "    def forward(self, features, **kwargs):\\n", "        return self.forward_perceived(*self.perceive(features))"]}, {"cell_type": "code", "execution_count": null, "metadata": {"tags": ["training_definitions"]}, "outputs": [], "source": ["def make_batch(indices):\\n", "    boards = [chess.Board(str(fens[i])) for i in indices]\\n", "    features = torch.tensor(np.stack([encode_board(b) for b in boards]), device=DEVICE)\\n", "    labels = torch.tensor([move_index(chess.Move.from_uci(str(labels_uci[i])), b.turn)\\n", "                           for i, b in zip(indices, boards)], device=DEVICE)\\n", "    targets = torch.tensor(values[indices], device=DEVICE)\\n", "    return boards, features, labels, targets\\n", "\\n", "\\n", "@contextlib.contextmanager\\n", "def inference_mode(model):\\n", "    was_training = model.training\\n", "    model.eval()\\n", "    try:\\n", "        with torch.no_grad():\\n", "            yield\\n", "    finally:\\n", "        model.train(was_training)\\n", "\\n", "\\n", "def move_match(model, indices, intervention=None, value_weight=4.0):\\n", "    correct, raw_correct, squared_error, policy_total, value_total = 0, 0, 0.0, 0.0, 0.0\\n", "    with inference_mode(model):\\n", "        for offset in range(0, len(indices), BATCH_SIZE):\\n", "            boards, features, labels, targets = make_batch(indices[offset:offset + BATCH_SIZE])\\n", "            logits, value = model(features, intervention=intervention) if intervention else model(features)\\n", "            raw_correct += int((logits.argmax(1) == labels).sum())\\n", "            correct += int((mask_logits(logits, boards).argmax(1) == labels).sum())\\n", "            squared_error += float(((value.sigmoid() - targets)**2).sum())\\n", "            policy_total += float(F.cross_entropy(mask_logits(logits, boards), labels, reduction=\\"sum\\"))\\n", "            value_total += float(F.binary_cross_entropy_with_logits(value, targets, reduction=\\"sum\\"))\\n", "    constant = float(np.mean(values[train_idx]))\\n", "    baseline = float(np.mean((values[indices] - constant)**2))\\n", "    mse = squared_error / len(indices)\\n", "    return {\\"legal_move_match\\": correct / len(indices), \\"unmasked_diagnostic\\": raw_correct / len(indices),\\n", "            \\"value_mse\\": mse, \\"constant_value_mse\\": baseline,\\n", "            \\"value_skill\\": 1 - mse / baseline if baseline > 1e-10 else None,\\n", "            \\"validation_loss\\": (policy_total + value_weight*value_total)/len(indices), \\"n\\": len(indices)}\\n", "\\n", "\\n", "def cache_perception(candidate, tag, pool, identity):\\n", "    # Full-precision CPU memmaps: exactly the same frozen states as live inference.\\n", "    directory = ROOT / \\"perception-cache\\" / RUN_ID / tag\\n", "    directory.mkdir(parents=True, exist_ok=True)\\n", "    progress_path = directory / \\"progress.json\\"\\n", "    progress = json.loads(progress_path.read_text()) if progress_path.exists() else None\\n", "    if progress and progress[\\"identity\\"] != identity:\\n", "        raise ValueError(\\"Perception cache identity changed\\")\\n", "    first = candidate.perceive(make_batch(pool[:1])[1])\\n", "    widths = [x.shape[1] for x in first]\\n", "    files = [directory / f\\"component-{i}.npy\\" for i in range(len(widths))]\\n", "    if progress and all(path.exists() for path in files):\\n", "        arrays = [np.lib.format.open_memmap(path, mode=\\"r+\\") for path in files]\\n", "        completed = progress[\\"completed\\"]\\n", "    else:\\n", "        arrays = [np.lib.format.open_memmap(path, mode=\\"w+\\", dtype=np.float32, shape=(len(pool), width))\\n", "                  for path, width in zip(files, widths)]\\n", "        completed = 0\\n", "    for start in range(completed, len(pool), BATCH_SIZE):\\n", "        check_stop()\\n", "        batch = pool[start:start + BATCH_SIZE]\\n", "        with torch.no_grad():\\n", "            components = candidate.perceive(make_batch(batch)[1])\\n", "        for array, component in zip(arrays, components):\\n", "            array[start:start + len(batch)] = component.cpu().numpy()\\n", "            array.flush()\\n", "        atomic_json(progress_path, {\\"identity\\": identity, \\"completed\\": start + len(batch), \\"widths\\": widths})\\n", "        if start % (BATCH_SIZE * 20) == 0:\\n", "            print(\\"perception cache\\", tag, start + len(batch), \\"/\\", len(pool), flush=True)\\n", "    return {\\"pool\\": pool, \\"lookup\\": {int(i): j for j, i in enumerate(pool)}, \\"arrays\\": arrays}\\n", "\\n", "\\n", "def cached_forward(candidate, cache, indices):\\n", "    rows = [cache[\\"lookup\\"][int(i)] for i in indices]\\n", "    components = []\\n", "    for i, array in enumerate(cache[\\"arrays\\"]):\\n", "        selected = np.asarray(array[rows])\\n", "        if cache.get(\\"select_columns\\"):\\n", "            selected = selected[:, cache[\\"select_columns\\"][i]]\\n", "        components.append(torch.tensor(selected, device=DEVICE))\\n", "    return candidate.forward_perceived(*components)\\n", "\\n", "\\n", "def initialize_signal_stats(candidate, indices):\\n", "    with torch.no_grad():\\n", "        features = make_batch(indices)[1]\\n", "        components = candidate.perceive(features) if hasattr(candidate, \\"perceive\\") else None\\n", "        if isinstance(candidate, FlyChessModel):\\n", "            relay, initial, background = components\\n", "            relay = candidate.graft.pool_relay(relay)\\n", "            candidate.graft.relay_mean.copy_(relay.mean(0))\\n", "            floor = 0.1 if candidate.graft.relay_pool.numel() else 0.0001\\n", "            candidate.graft.relay_scale.copy_(relay.std(0, unbiased=False).clamp_min(floor))\\n", "            # Motor normalization is a fixed affine transform of the zero-graft act state.\\n", "            zero = torch.zeros_like(initial.t())\\n", "            motor = candidate.brain.run_active_prepared(initial.t(), background.t(), zero,\\n", "                candidate.blocks[0], candidate.T_a)[candidate.motor_active_idx].t()\\n", "        elif isinstance(candidate, FlyOnlyModel):\\n", "            motor = components[0]\\n", "        else:\\n", "            relay, motor = candidate.interface.perceive(features)\\n", "            relay = candidate.graft.pool_relay(relay)\\n", "            candidate.graft.relay_mean.copy_(relay.mean(0))\\n", "            floor = 0.1 if candidate.graft.relay_pool.numel() else 0.0001\\n", "            candidate.graft.relay_scale.copy_(relay.std(0, unbiased=False).clamp_min(floor))\\n", "        candidate.decoder.motor_mean.copy_(motor.mean(0))\\n", "        candidate.decoder.motor_scale.copy_(motor.std(0, unbiased=False).clamp_min(0.03))\\n", "\\n", "\\n", "def training_update(model, optimizer, sampler, cache=None, indices=None, value_weight=4.0):\\n", "    source = cache[\\"pool\\"] if cache else train_idx\\n", "    if indices is None:\\n", "        indices = sampler.choice(source, min(BATCH_SIZE, len(source)), replace=False)\\n", "    boards, features, labels, targets = make_batch(indices)\\n", "    model.train()\\n", "    logits, value = cached_forward(model, cache, indices) if cache else model(features)\\n", "    policy_loss = F.cross_entropy(mask_logits(logits, boards), labels)\\n", "    value_loss = F.binary_cross_entropy_with_logits(value, targets)\\n", "    loss = policy_loss + value_weight * value_loss\\n", "    if not torch.isfinite(loss):\\n", "        raise RuntimeError(\\"Nonfinite training loss; checkpoint retained\\")\\n", "    optimizer.zero_grad(set_to_none=True)\\n", "    loss.backward()\\n", "    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0, error_if_nonfinite=True)\\n", "    optimizer.step()\\n", "    return {\\"loss\\": float(loss.detach()), \\"policy_loss\\": float(policy_loss.detach()),\\n", "            \\"value_loss\\": float(value_loss.detach()), \\"grad_norm\\": float(grad_norm)}\\n", "\\n", "\\n", "def learning_preflight(candidate, cache):\\n", "    indices = cache[\\"pool\\"][:min(32, len(cache[\\"pool\\"]))]\\n", "    boards, _, labels, targets = make_batch(indices)\\n", "    original = {key: value.detach().clone() for key, value in candidate.state_dict().items()}\\n", "    sampler = np.random.RandomState(SEED)\\n", "    saved_rng = rng_state(sampler)\\n", "    def metrics():\\n", "        with inference_mode(candidate):\\n", "            logits, value = cached_forward(candidate, cache, indices)\\n", "            return {\\"accuracy\\": float((mask_logits(logits, boards).argmax(1) == labels).float().mean()),\\n", "                    \\"value_mse\\": float(((value.sigmoid() - targets)**2).mean())}\\n", "    try:\\n", "        before = metrics()\\n", "        optimizer = torch.optim.AdamW(candidate.parameters(), lr=1e-3)\\n", "        for _ in range(160 if MODE == \\"full\\" else 40):\\n", "            check_stop()\\n", "            training_update(candidate, optimizer, sampler, cache, indices)\\n", "        after = metrics()\\n", "        passed = after[\\"accuracy\\"] >= max(0.5, before[\\"accuracy\\"] + 0.15) and after[\\"value_mse\\"] < before[\\"value_mse\\"] * 0.8\\n", "        result = {\\"before\\": before, \\"after\\": after, \\"passed\\": passed, \\"n\\": len(indices)}\\n", "        atomic_json(RUN_DIR / \\"learning-preflight.json\\", result)\\n", "        print(\\"learning preflight\\", result, flush=True)\\n", "        if MODE == \\"full\\" and not passed:\\n", "            raise RuntimeError(\\"Learning preflight failed; long training skipped. Inspect learning-preflight.json\\")\\n", "        return result\\n", "    finally:\\n", "        candidate.load_state_dict(original)\\n", "        restore_rng(saved_rng, sampler)"]}, {"cell_type": "code", "execution_count": null, "metadata": {"tags": ["search"]}, "outputs": [], "source": ["class Node:\\n", "    def __init__(self, board, parent=None, prior=1.0):\\n", "        self.board, self.parent, self.prior = board, parent, prior\\n", "        self.children = {}\\n", "        self.visits = 0\\n", "        self.pending = 0\\n", "        self.value_sum = 0.0\\n", "        self.expanded = False\\n", "        self.neural_evaluations = 0\\n", "\\n", "    def q(self):\\n", "        return self.value_sum / self.visits if self.visits else 0.5\\n", "\\n", "\\n", "def puct_score(child, parent_visits, c_puct=1.5):\\n", "    return 1 - child.q() + c_puct * child.prior * math.sqrt(max(1, parent_visits)) / (1 + child.visits + child.pending)\\n", "\\n", "\\n", "def evaluate_leaves(model, boards):\\n", "    with inference_mode(model):\\n", "        features = torch.tensor(np.stack([encode_board(b) for b in boards]), device=DEVICE)\\n", "        logits, values = model(features)\\n", "        priors = []\\n", "        for board, row in zip(boards, logits):\\n", "            indices = list(legal_move_table(board))\\n", "            probabilities = F.softmax(row[indices], dim=0).cpu().numpy()\\n", "            priors.append(dict(zip(indices, probabilities)))\\n", "        return priors, torch.sigmoid(values).cpu().numpy()\\n", "\\n", "\\n", "def expand(node, priors):\\n", "    if node.expanded:\\n", "        return\\n", "    for index, probability in priors.items():\\n", "        board = node.board.copy(stack=True)\\n", "        board.push(index_to_move(index, board))\\n", "        node.children[index] = Node(board, node, float(probability))\\n", "    node.expanded = True\\n", "\\n", "\\n", "def backup(path, value):\\n", "    for node in reversed(path):\\n", "        node.visits += 1\\n", "        node.value_sum += value\\n", "        value = 1 - value\\n", "\\n", "\\n", "def reserve_leaf(root):\\n", "    def descend(node, path):\\n", "        if terminal_value(node.board) is not None:\\n", "            return node, path\\n", "        if not node.expanded:\\n", "            return (node, path) if node.pending == 0 else None\\n", "        ordered = sorted(node.children.items(), key=lambda item: (-puct_score(item[1], node.visits + node.pending), item[0]))\\n", "        for _, child in ordered:\\n", "            found = descend(child, path + [child])\\n", "            if found is not None:\\n", "                return found\\n", "        return None\\n", "    found = descend(root, [root])\\n", "    if found:\\n", "        for node in found[1]: node.pending += 1\\n", "    return found\\n", "\\n", "\\n", "def search(model, root_board, n_simulations=64, batch=16, deadline=None, root_priors=None):\\n", "    root = Node(root_board.copy(stack=True))\\n", "    if terminal_value(root.board) is not None:\\n", "        return root\\n", "    if root_priors is None:\\n", "        priors, _ = evaluate_leaves(model, [root.board])\\n", "        root_priors = priors[0]\\n", "    expand(root, root_priors)\\n", "    root.neural_evaluations = 1\\n", "    completed = 0\\n", "    while completed < n_simulations and (deadline is None or time.monotonic() < deadline):\\n", "        check_stop()\\n", "        selected = []\\n", "        for _ in range(min(batch, n_simulations - completed)):\\n", "            found = reserve_leaf(root)\\n", "            if found is None: break\\n", "            selected.append(found)\\n", "        if not selected: break\\n", "        try:\\n", "            neural = [(node, path) for node, path in selected if terminal_value(node.board) is None]\\n", "            predictions = {}\\n", "            if neural:\\n", "                priors, values = evaluate_leaves(model, [node.board for node, _ in neural])\\n", "                root.neural_evaluations += len(neural)\\n", "                predictions = {id(node): (prior, float(value)) for (node, _), prior, value in zip(neural, priors, values)}\\n", "            for node, path in selected:\\n", "                value = terminal_value(node.board)\\n", "                if value is None:\\n", "                    prior, value = predictions[id(node)]\\n", "                    expand(node, prior)\\n", "                backup(path, value)\\n", "                completed += 1\\n", "        finally:\\n", "            for _, path in selected:\\n", "                for node in path: node.pending -= 1\\n", "    assert root.visits == completed\\n", "    return root\\n", "\\n", "\\n", "def best_move_by_search(model, board, n_simulations=None, deadline=None, root_priors=None):\\n", "    root = search(model, board, n_simulations or (8 if MODE == \\"smoke\\" else 64), deadline=deadline, root_priors=root_priors)\\n", "    if not root.children: return None, root\\n", "    index = max(root.children, key=lambda i: (root.children[i].visits, root.children[i].prior, -i))\\n", "    return index_to_move(index, board), root\\n", "\\n", "\\n", "def predict(model, boards):\\n", "    features = torch.tensor(np.stack([encode_board(b) for b in boards]), device=DEVICE)\\n", "    with inference_mode(model):\\n", "        logits, values = model(features)\\n", "        return logits, torch.sigmoid(values)\\n", "\\n", "\\n", "def mate_in_one(board):\\n", "    for move in list(board.legal_moves):\\n", "        board.push(move)\\n", "        mate = board.is_checkmate()\\n", "        board.pop()\\n", "        if mate: return move\\n", "    return None\\n", "\\n", "\\n", "def choose_move(model, board, use_search=False, assist_mode=\\"none\\", return_metadata=False, deadline=None):\\n", "    started = time.monotonic()\\n", "    metadata = {\\"assist_mode\\": assist_mode, \\"assisted\\": False, \\"neural_evaluations\\": 0, \\"simulations\\": 0}\\n", "    if terminal_value(board) is not None:\\n", "        return (None, metadata) if return_metadata else None\\n", "    logits, value = predict(model, [board])\\n", "    legal_indices = list(legal_move_table(board))\\n", "    row = logits[0].detach().cpu().numpy()\\n", "    ranked = sorted(legal_indices, key=lambda i: (-float(row[i]), i))\\n", "    if use_search:\\n", "        probabilities = F.softmax(logits[0, legal_indices], dim=0).cpu().numpy()\\n", "        move, tree = best_move_by_search(model, board, deadline=deadline, root_priors=dict(zip(legal_indices, probabilities)))\\n", "        metadata.update(neural_evaluations=tree.neural_evaluations, simulations=tree.visits)\\n", "    else:\\n", "        move = index_to_move(ranked[0], board)\\n", "        metadata[\\"neural_evaluations\\"] = 1\\n", "    metadata[\\"model_move\\"] = move.uci()\\n", "    assert assist_mode in (\\"none\\", \\"mate1\\")\\n", "    if assist_mode == \\"mate1\\":\\n", "        mate = mate_in_one(board)\\n", "        if mate is not None:\\n", "            move = mate\\n", "        else:\\n", "            def safe(candidate):\\n", "                board.push(candidate)\\n", "                bad = mate_in_one(board) is not None if terminal_value(board) is None else False\\n", "                board.pop()\\n", "                return not bad\\n", "            if not safe(move):\\n", "                move = next((index_to_move(i, board) for i in ranked if safe(index_to_move(i, board))), move)\\n", "    metadata.update(final_move=move.uci(), assisted=move.uci() != metadata[\\"model_move\\"],\\n", "                    elapsed_s=time.monotonic() - started, value=float(value[0]))\\n", "    assert move in board.legal_moves\\n", "    return (move, metadata) if return_metadata else move"]}, {"cell_type": "code", "execution_count": null, "metadata": {"tags": ["scoring"]}, "outputs": [], "source": ["STOCKFISH_DEPTH = 3 if MODE == \\"smoke\\" else 12\\n", "SCORE_CACHE_PATH = RUN_DIR / \\"engine-scores.json\\"\\n", "\\n", "\\n", "def open_stockfish():\\n", "    engine = chess.engine.SimpleEngine.popen_uci(str(STOCKFISH_BIN))\\n", "    engine.configure({\\"Threads\\": 1, \\"Hash\\": 64, \\"UCI_LimitStrength\\": False, \\"Skill Level\\": 20})\\n", "    return engine\\n", "\\n", "\\n", "def score_move(engine, board, move):\\n", "    key = \\"|\\".join((board.fen(en_passant=\\"fen\\"), move.uci(), STOCKFISH_BINARY_SHA, str(STOCKFISH_DEPTH)))\\n", "    if key in SCORE_CACHE:\\n", "        return SCORE_CACHE[key]\\n", "    engine.configure({\\"Clear Hash\\": None})\\n", "    info = engine.analyse(board, chess.engine.Limit(depth=STOCKFISH_DEPTH), root_moves=[move])\\n", "    score = info[\\"score\\"].pov(board.turn)\\n", "    result = {\\"cp\\": score.score(), \\"mate\\": score.mate(),\\n", "              \\"expectation\\": score.wdl(model=\\"sf\\", ply=board.ply()).expectation()}\\n", "    SCORE_CACHE[key] = result\\n", "    atomic_json(SCORE_CACHE_PATH, SCORE_CACHE)\\n", "    return result\\n", "\\n", "\\n", "def score_position(engine, board):\\n", "    return {move.uci(): score_move(engine, board, move) for move in board.legal_moves}\\n", "\\n", "\\n", "def position_loss(scores, move):\\n", "    best = max(scores.values(), key=lambda row: row[\\"expectation\\"])\\n", "    chosen = scores[move.uci()]\\n", "    expected_loss = max(0, best[\\"expectation\\"] - chosen[\\"expectation\\"])\\n", "    ordinary = [row[\\"cp\\"] for row in scores.values() if row[\\"cp\\"] is not None]\\n", "    cp_loss = max(0, max(ordinary) - chosen[\\"cp\\"]) if chosen[\\"cp\\"] is not None and all(row[\\"mate\\"] is None for row in scores.values()) else None\\n", "    return {\\"expected_score_loss\\": expected_loss, \\"cp_loss\\": cp_loss,\\n", "            \\"mate_outcome\\": chosen[\\"mate\\"], \\"best_mate_outcome\\": best[\\"mate\\"]}\\n", "\\n", "\\n", "def bootstrap_mean(values, seed=21):\\n", "    values = np.asarray(values, dtype=float)\\n", "    if len(values) == 0: return {\\"mean\\": None, \\"ci95\\": None, \\"n\\": 0}\\n", "    sampler = np.random.RandomState(seed)\\n", "    means = [values[sampler.randint(len(values), size=len(values))].mean() for _ in range(2000)]\\n", "    return {\\"mean\\": float(values.mean()), \\"ci95\\": np.percentile(means, [2.5, 97.5]).tolist(), \\"n\\": len(values)}\\n", "\\n", "\\n", "def uniform_legal_match(indices):\\n", "    return float(np.mean([1 / chess.Board(str(fens[i])).legal_moves.count() for i in indices]))\\n", "\\n", "SCORE_CACHE = json.loads(SCORE_CACHE_PATH.read_text()) if SCORE_CACHE_PATH.exists() else {}\\n", "\\n", "\\n", "def ratio_summary(chosen, random_losses):\\n", "    chosen, random_losses = np.asarray(chosen), np.asarray(random_losses)\\n", "    if len(chosen) == 0 or random_losses.mean() <= 1e-8:\\n", "        return {\\"value\\": None, \\"ci95\\": None, \\"n\\": len(chosen)}\\n", "    sampler = np.random.RandomState(24)\\n", "    ratios = []\\n", "    for _ in range(2000):\\n", "        indices = sampler.randint(len(chosen), size=len(chosen))\\n", "        denominator = random_losses[indices].mean()\\n", "        if denominator > 1e-8:\\n", "            ratios.append(1 - chosen[indices].mean() / denominator)\\n", "    return {\\"value\\": float(1 - chosen.mean() / random_losses.mean()),\\n", "            \\"ci95\\": np.percentile(ratios, [2.5, 97.5]).tolist() if ratios else None, \\"n\\": len(chosen)}"]}, {"cell_type": "code", "execution_count": null, "metadata": {"tags": ["matches"]}, "outputs": [], "source": ["def puzzle_trial(model, row, use_search=False, assist_mode=\\"none\\", full_line=False):\\n", "    board = chess.Board(row[\\"FEN\\"])\\n", "    line = row[\\"Moves\\"].split()\\n", "    board.push_uci(line[0])\\n", "    for ply in range(1, len(line), 2):\\n", "        predicted = choose_move(model, board, use_search, assist_mode)\\n", "        if predicted is None: return False\\n", "        board.push(predicted)\\n", "        if board.is_checkmate(): return True\\n", "        if predicted.uci() != line[ply]: return False\\n", "        if not full_line: return True\\n", "        if ply + 1 < len(line): board.push_uci(line[ply + 1])\\n", "    return True\\n", "\\n", "\\n", "def pair_score_summary(records):\\n", "    completed = [r for r in records if r[\\"score\\"] is not None]\\n", "    pair_ids = sorted({r[\\"pair\\"] for r in completed})\\n", "    pair_scores = [np.mean([r[\\"score\\"] for r in completed if r[\\"pair\\"] == pair]) for pair in pair_ids\\n", "                   if sum(r[\\"pair\\"] == pair for r in completed) == 2]\\n", "    return {\\"wins\\": sum(r[\\"score\\"] == 1 for r in completed), \\"draws\\": sum(r[\\"score\\"] == 0.5 for r in completed),\\n", "            \\"losses\\": sum(r[\\"score\\"] == 0 for r in completed), \\"unresolved\\": sum(r[\\"score\\"] is None for r in records),\\n", "            \\"score\\": float(np.mean([r[\\"score\\"] for r in completed])) if completed else None,\\n", "            \\"paired_score\\": bootstrap_mean(pair_scores), \\"games\\": len(records)}\\n", "\\n", "\\n", "def play_game(white, black, opening, game_id, initial_clock=60.0, increment=0.6, max_plies=600):\\n", "    checkpoint = RUN_DIR / \\"games\\" / f\\"{game_id}.json\\"\\n", "    checkpoint.parent.mkdir(exist_ok=True)\\n", "    board = chess.Board()\\n", "    clocks = {\\"white\\": initial_clock, \\"black\\": initial_clock}\\n", "    history = []\\n", "    if checkpoint.exists():\\n", "        state = json.loads(checkpoint.read_text())\\n", "        clocks, history = state[\\"clocks\\"], state[\\"history\\"]\\n", "        for move in state[\\"moves\\"]: board.push_uci(move)\\n", "        if state[\\"status\\"] == \\"complete\\": return state\\n", "    else:\\n", "        for move in opening: board.push_uci(move)\\n", "    score, reason = None, \\"safety_cap\\"\\n", "    while len(history) < max_plies:\\n", "        check_stop()\\n", "        outcome = board.outcome(claim_draw=True)\\n", "        if outcome:\\n", "            score = 0.5 if outcome.winner is None else float(outcome.winner == chess.WHITE)\\n", "            reason = outcome.termination.name\\n", "            break\\n", "        side = \\"white\\" if board.turn else \\"black\\"\\n", "        started = time.monotonic()\\n", "        move = (white if board.turn else black)(board, clocks[side], increment)\\n", "        elapsed = time.monotonic() - started\\n", "        clocks[side] -= elapsed\\n", "        if clocks[side] < 0:\\n", "            score = float(board.turn != chess.WHITE)\\n", "            reason = \\"timeout\\"\\n", "            break\\n", "        if move not in board.legal_moves:\\n", "            raise RuntimeError(\\"Opponent returned an illegal move\\")\\n", "        board.push(move)\\n", "        clocks[side] += increment\\n", "        history.append({\\"move\\": move.uci(), \\"side\\": side, \\"elapsed_s\\": elapsed})\\n", "        atomic_json(checkpoint, {\\"status\\": \\"in_progress\\", \\"moves\\": [m.uci() for m in board.move_stack],\\n", "                    \\"clocks\\": clocks, \\"history\\": history})\\n", "    if score is None:\\n", "        outcome = board.outcome(claim_draw=True)\\n", "        if outcome is not None:\\n", "            score = 0.5 if outcome.winner is None else float(outcome.winner == chess.WHITE)\\n", "            reason = outcome.termination.name\\n", "    state = {\\"status\\": \\"complete\\", \\"score\\": score, \\"reason\\": reason,\\n", "             \\"moves\\": [m.uci() for m in board.move_stack], \\"clocks\\": clocks, \\"history\\": history}\\n", "    atomic_json(checkpoint, state)\\n", "    game = chess.pgn.Game.from_board(board)\\n", "    game.headers[\\"Result\\"] = \\"*\\" if score is None else \\"1-0\\" if score == 1 else \\"0-1\\" if score == 0 else \\"1/2-1/2\\"\\n", "    game.headers[\\"Termination\\"] = reason\\n", "    game.headers[\\"TimeControl\\"] = f\\"{initial_clock}+{increment}\\"\\n", "    checkpoint.with_suffix(\\".pgn\\").write_text(str(game) + \\"\\\\n\\")\\n", "    return state\\n", "\\n", "\\n", "OPENINGS = [\\n", "    \\"e2e4 e7e5 g1f3 b8c6 f1b5 a7a6\\", \\"e2e4 c7c5 g1f3 d7d6 d2d4 c5d4\\",\\n", "    \\"d2d4 d7d5 c2c4 e7e6 b1c3 g8f6\\", \\"d2d4 g8f6 c2c4 g7g6 b1c3 f8g7\\",\\n", "    \\"c2c4 e7e5 b1c3 g8f6 g2g3 d7d5\\", \\"g1f3 d7d5 g2g3 g8f6 f1g2 e7e6\\",\\n", "    \\"e2e4 e7e6 d2d4 d7d5 b1c3 g8f6\\", \\"e2e4 c7c6 d2d4 d7d5 b1c3 d5e4\\",\\n", "    \\"d2d4 d7d5 c2c4 c7c6 g1f3 g8f6\\", \\"e2e4 e7e5 g1f3 g8f6 f3e5 d7d6\\"]\\n", "for sequence in OPENINGS:\\n", "    board = chess.Board()\\n", "    for move in sequence.split(): board.push_uci(move)\\n", "\\n", "\\n", "def model_opponent(candidate, use_search=False, assist_mode=\\"none\\"):\\n", "    def mover(board, clock, increment):\\n", "        allocation = min(max(0.01, clock / 30 + increment * 0.5), max(0.01, clock * 0.25))\\n", "        deadline = time.monotonic() + allocation\\n", "        return choose_move(candidate, board, use_search, assist_mode, deadline=deadline)\\n", "    return mover\\n", "\\n", "\\n", "def greedy_material(board, clock=None, increment=None):\\n", "    weights = {chess.PAWN: 1, chess.KNIGHT: 3, chess.BISHOP: 3, chess.ROOK: 5, chess.QUEEN: 9, chess.KING: 0}\\n", "    def value(move):\\n", "        board.push(move)\\n", "        result = sum(weights[p.piece_type] * (1 if p.color != board.turn else -1) for p in board.piece_map().values())\\n", "        board.pop()\\n", "        return result\\n", "    return max(board.legal_moves, key=value)\\n", "\\n", "\\n", "def engine_opponent(engine, skill=0):\\n", "    def mover(board, clock, increment):\\n", "        engine.configure({\\"UCI_LimitStrength\\": False, \\"Skill Level\\": skill})\\n", "        return engine.play(board, chess.engine.Limit(time=min(max(0.01, clock / 30 + increment * 0.5), max(0.01, clock * 0.25)))).move\\n", "    return mover\\n", "\\n", "\\n", "\\"\\"\\"Score-based strength estimates with explicit reference scales.\\n", "\\n", "Opening pairs, rather than individual games, are the independent sampling unit.\\n", "The bounded-score Hoeffding interval stays nonzero after an all-loss/all-win run.\\n", "It quantifies match sampling only, not uncertainty in an opponent\'s calibration.\\n", "\\"\\"\\"\\n", "import math\\n", "\\n", "\\n", "def score_to_elo(score):\\n", "    if score <= 0 or score >= 1:\\n", "        return None\\n", "    return 400 * math.log10(score / (1 - score))\\n", "\\n", "\\n", "def strength_summary(records, reference_rating=None, reference_scale=None, alpha=0.05):\\n", "    if not 0 < alpha < 1:\\n", "        raise ValueError(\\"alpha must lie between zero and one\\")\\n", "    if reference_rating is not None and not reference_scale:\\n", "        raise ValueError(\\"A reference rating requires an explicit scale/source\\")\\n", "    if reference_rating is not None and (not isinstance(reference_rating,(int,float)) or not math.isfinite(reference_rating)):\\n", "        raise ValueError(\\"Reference rating must be finite\\")\\n", "    groups = {}\\n", "    for record in records:\\n", "        score = record.get(\\"score\\")\\n", "        if score is not None and score not in (0, 0.5, 1):\\n", "            raise ValueError(\\"Game score must be 0, 0.5, 1, or unresolved\\")\\n", "        groups.setdefault(record[\\"pair\\"], []).append(record)\\n", "    pairs = []\\n", "    for group in groups.values():\\n", "        if len(group) == 2 and {r[\\"color\\"] for r in group} == {\\"white\\", \\"black\\"}:\\n", "            if all(r[\\"score\\"] is not None for r in group):\\n", "                pairs.append(sum(r[\\"score\\"] for r in group) / 2)\\n", "    report = {\\"complete_pairs\\": len(pairs), \\"games\\": len(records),\\n", "              \\"wins\\": sum(r.get(\\"score\\") == 1 for r in records),\\n", "              \\"draws\\": sum(r.get(\\"score\\") == .5 for r in records),\\n", "              \\"losses\\": sum(r.get(\\"score\\") == 0 for r in records),\\n", "              \\"unresolved\\": sum(r.get(\\"score\\") is None for r in records),\\n", "              \\"reference_rating\\": reference_rating, \\"reference_scale\\": reference_scale,\\n", "              \\"interval_method\\": \\"95% bounded-score Hoeffding over opening pairs\\" if alpha == .05\\n", "                                 else f\\"{1-alpha:.1%} bounded-score Hoeffding over opening pairs\\",\\n", "              \\"sampling_assumption\\": \\"Independent, representative opening pairs\\",\\n", "              \\"score\\": None, \\"score_interval\\": None, \\"elo_difference\\": None,\\n", "              \\"elo_difference_interval\\": None, \\"reference_scale_estimate\\": None,\\n", "              \\"reference_scale_interval\\": None, \\"status\\": \\"insufficient_games\\"}\\n", "    if not pairs:\\n", "        return report\\n", "    score = sum(pairs) / len(pairs)\\n", "    radius = math.sqrt(math.log(2 / alpha) / (2 * len(pairs)))\\n", "    interval = [max(0, score - radius), min(1, score + radius)]\\n", "    delta = score_to_elo(score)\\n", "    bounds = [score_to_elo(p) for p in interval]\\n", "    report.update(score=score, score_interval=interval, elo_difference=delta,\\n", "                  elo_difference_interval=bounds,\\n", "                  status=\\"estimate\\" if delta is not None else \\"upper_bound\\" if score == 0 else \\"lower_bound\\")\\n", "    if interval == [0,1]:\\n", "        report[\\"status\\"] = \\"unbounded_interval\\"\\n", "    if reference_rating is not None:\\n", "        report[\\"reference_scale_estimate\\"] = reference_rating + delta if delta is not None else None\\n", "        report[\\"reference_scale_interval\\"] = [reference_rating + b if b is not None else None for b in bounds]\\n", "        report[\\"calibration_note\\"] = \\"Conditional on the reference\'s calibration; not a FIDE/Chess.com/Lichess rating\\"\\n", "    report[\\"unbounded_endpoints\\"] = {\\"lower\\": interval[0] == 0, \\"upper\\": interval[1] == 1}\\n", "    return report\\n", "\\n", "\\n", "def make_opening_pairs(count, seed=41):\\n", "    generator = np.random.RandomState(seed)\\n", "    openings, seen = [], set()\\n", "    while len(openings) < count:\\n", "        board = chess.Board()\\n", "        moves = OPENINGS[int(generator.randint(len(OPENINGS)))].split()\\n", "        for move in moves:\\n", "            board.push_uci(move)\\n", "        for _ in range(2):\\n", "            if terminal_value(board) is not None:\\n", "                break\\n", "            move = list(board.legal_moves)[int(generator.randint(board.legal_moves.count()))]\\n", "            moves.append(move.uci())\\n", "            board.push(move)\\n", "        key = \\" \\".join(moves)\\n", "        if terminal_value(board) is None and key not in seen:\\n", "            seen.add(key)\\n", "            openings.append(moves)\\n", "    return openings\\n", "\\n", "\\n", "def evaluate_strength():\\n", "    path = RUN_DIR / \\"strength.json\\"\\n", "    pairs = 1 if MODE == \\"smoke\\" else int(os.environ.get(\\"FLY_CHESS_MATCH_PAIRS\\", \\"20\\"))\\n", "    assert 1 <= pairs <= 1000\\n", "    openings = make_opening_pairs(pairs)\\n", "    report = json.loads(path.read_text()) if path.exists() else {\\"identity\\": IDENTITY, \\"matches\\": {}, \\"summaries\\": {}}\\n", "    assert report[\\"identity\\"] == IDENTITY\\n", "    started = time.monotonic()\\n", "    budget = float(os.environ.get(\\"FLY_CHESS_EVAL_MINUTES\\", \\"75\\"))*60 if MODE == \\"full\\" else 180\\n", "    clock, increment = (120, 1) if MODE == \\"full\\" else (2, .01)\\n", "    with open_stockfish() as engine:\\n", "        minimum, maximum = engine.options[\\"UCI_Elo\\"].min, engine.options[\\"UCI_Elo\\"].max\\n", "        ratings = sorted(set([minimum, min(maximum, minimum + 200)]))\\n", "        opponents = [(\\"c0\\", model_opponent(model_c0), None), (\\"greedy\\", greedy_material, None)]\\n", "        for rating in ratings:\\n", "            def opponent(board, remaining, inc, rating=rating):\\n", "                engine.configure({\\"UCI_LimitStrength\\": True, \\"UCI_Elo\\": rating, \\"Skill Level\\": 20})\\n", "                return engine.play(board, chess.engine.Limit(white_clock=remaining if board.turn else clock,\\n", "                    black_clock=remaining if not board.turn else clock, white_inc=inc, black_inc=inc)).move\\n", "            opponents.append((f\\"stockfish-uci-{rating}\\", opponent, rating))\\n", "        configuration = {\\"pairs\\": pairs, \\"clock\\": clock, \\"increment\\": increment,\\n", "                         \\"ratings\\": ratings, \\"opening_seed\\": 41, \\"assist_mode\\": \\"none\\"}\\n", "        if report.get(\\"configuration\\") and report[\\"configuration\\"] != configuration:\\n", "            raise ValueError(\\"Strength match configuration changed; use a new run\\")\\n", "        report[\\"configuration\\"] = configuration\\n", "        selection_path = RUN_DIR / \\"model-selection.json\\"\\n", "        selection = json.loads(selection_path.read_text()) if selection_path.exists() else {}\\n", "        fingerprints = {}\\n", "        for tag in (\\"main\\",\\"c0\\"):\\n", "            checkpoint_path = RUN_DIR / selection.get(tag,{}).get(\\"checkpoint\\",f\\"{tag}.pt\\")\\n", "            fingerprints[tag] = sha256(checkpoint_path) if checkpoint_path.exists() else None\\n", "        if report.get(\\"checkpoint_fingerprints\\") is not None and report[\\"checkpoint_fingerprints\\"] != fingerprints:\\n", "            raise ValueError(\\"Evaluated weights changed; use a separate run for a new benchmark\\")\\n", "        report[\\"checkpoint_fingerprints\\"] = fingerprints\\n", "        report[\\"model_selection\\"] = selection\\n", "        try:\\n", "            # Interleave opponents and modes so a pause leaves evidence for each condition.\\n", "            for pair, opening in enumerate(openings):\\n", "                for mode_name, use_search in ((\\"policy\\", False), (\\"search\\", True)):\\n", "                    for name, opponent, rating in opponents:\\n", "                        key = mode_name + \\":\\" + name\\n", "                        records = report[\\"matches\\"].setdefault(key, [])\\n", "                        for color in (\\"white\\", \\"black\\"):\\n", "                            check_stop()\\n", "                            if time.monotonic() - started > budget:\\n", "                                report[\\"status\\"] = \\"paused\\"\\n", "                                report[\\"pause_reason\\"] = \\"Session evaluation allocation exhausted; rerun to continue\\"\\n", "                                return report\\n", "                            game_id = f\\"strength-{mode_name}-{name}-{pair}-{color}\\"\\n", "                            if any(r[\\"id\\"] == game_id for r in records):\\n", "                                continue\\n", "                            player = model_opponent(model, use_search)\\n", "                            white, black = (player, opponent) if color == \\"white\\" else (opponent, player)\\n", "                            game = play_game(white, black, opening, game_id, initial_clock=clock,\\n", "                                             increment=increment, max_plies=600 if MODE == \\"full\\" else 12)\\n", "                            score = game[\\"score\\"]\\n", "                            if score is not None and color == \\"black\\":\\n", "                                score = 1-score\\n", "                            records.append({\\"id\\": game_id, \\"pair\\": pair, \\"color\\": color, \\"score\\": score, \\"reason\\": game[\\"reason\\"]})\\n", "                            report[\\"summaries\\"][key] = strength_summary(records, rating,\\n", "                                f\\"Stockfish UCI_Elo at {clock}+{increment}\\" if rating is not None else None)\\n", "                            atomic_json(path, report)\\n", "                            print(key, pair, color, score, game[\\"reason\\"], flush=True)\\n", "            report[\\"status\\"] = \\"complete\\"\\n", "        except (RunPaused, KeyboardInterrupt) as exc:\\n", "            report[\\"status\\"] = \\"paused\\"\\n", "            report[\\"pause_reason\\"] = str(exc)\\n", "        finally:\\n", "            report[\\"time_control\\"] = {\\"initial_seconds\\": clock, \\"increment_seconds\\": increment}\\n", "            report[\\"engine_sha256\\"] = STOCKFISH_BINARY_SHA\\n", "            atomic_json(path, report)\\n", "    return report"]}], "metadata": {}, "nbformat": 4, "nbformat_minor": 5}')

Training saves on pause and every minute. Each re-run gets a fresh allowance. Paired branches stay at the same curriculum round, sharing labels and frozen caches.

In [ ]:
trainer = AdvancedTrainer(globals(),CONFIG)
with writer_lock():
    trainer.run(RUN_HOURS * 60 if MODE == "full" else 3)

Games capped before a result are unresolved and excluded from rating. More completed opening pairs narrow uncertainty. Resume evaluation of the same immutable snapshot; a new checkpoint gets a separate result folder.

In [ ]:
if EVALUATE:
    with writer_lock():
        benchmark_advanced(trainer,minutes=float(os.environ.get("FLY_CHESS_EVAL_MINUTES","1" if MODE == "smoke" else "15")),
                           pairs=int(os.environ["FLY_CHESS_MATCH_PAIRS"]))
else:
    print("Evaluation skipped. Set EVALUATE=True and re-run this cell for paired matches.")

In [ ]:
# Optional: set FLY_CHESS_ARCHIVE=1 to package this run after training/evaluation.
if os.environ.get("FLY_CHESS_ARCHIVE") == "1":
    import tarfile
    archive_path = ROOT / f"{RUN_ID}-checkpoints.tar.gz"
    with tarfile.open(archive_path,"w:gz") as archive:
        archive.add(STOP_ROOT,arcname=RUN_ID)
    print("archive",archive_path)